# Lab Day10 — Qwen3-VL road damage on Kaggle
Student code is in student/. Instructor scripts/tests are unchanged.


## 0. Môi trường và nguồn local


In [ ]:
import base64, hashlib, json, os, subprocess, sys
from pathlib import Path
root = Path('/kaggle/working/day10-source')
root.mkdir(exist_ok=False)
payload = json.loads('{"README.md": "IyBMYWIgRGF5IDEwOiBGaW5ldHVuZSBRd2VuMy1WTC00QiAoTG9SQSkgxJHhu4MgxJHhu4tuaCB24buLIGjGsCBo4bqhaSBt4bq3dCDEkcaw4budbmcKCj4gVHJhY2sgNCDCtyBOZ8OgeSAxMCDCtyAqTmjDo24sIGF1dG8tbGFiZWxpbmcgdsOgIFZpc2lvbuKAk0xhbmd1YWdlIE1vZGVsKiDCtyAqKmLDoGkgY8OhIG5ow6JuKiogwrcg4omIIDQgZ2nhu50gdHLDqm4gKipHb29nbGUgQ29sYWIgVDQgKG1p4buFbiBwaMOtKSoqCj4gQuG6oW4gKip04buxIHZp4bq/dCoqIHBo4bqnbiBodeG6pW4gbHV54buHbiB2w6Agc3V5IGx14bqtbiB0aGVvIHBzZXVkbyBjb2RlIHRyb25nIFtgR1VJREUubWRgXShHVUlERS5tZCkuIEPDtG5nIGPhu6UgY2hpYSBk4buvIGxp4buHdSB2w6AgY2jhuqVtIMSRaeG7g20gZG8gZ2nhuqNuZyB2acOqbiB2aeG6v3Qgc+G6tW4gdsOgICoqa2jDtG5nIMSRxrDhu6NjIHPhu61hKiouCgp8IEZpbGUgfCBEw7luZyDEkeG7gyBsw6BtIGfDrCB8CnwtLS18LS0tfAp8IGBSRUFETUUubWRgIChmaWxlIG7DoHkpIHwgQuG7qWMgdHJhbmggdG/DoG4gbGFiLCBk4buvIGxp4buHdSwgbcO0aSB0csaw4budbmcsIGPDoWNoIG7hu5lwIHwKfCBbYEdVSURFLm1kYF0oR1VJREUubWQpIHwgVOG7q25nIGLGsOG7m2MgKyBwc2V1ZG8gY29kZSArIGPDonUgaOG7j2kgfAp8IFtgUlVCUklDLm1kYF0oUlVCUklDLm1kKSB8IFRoYW5nIMSRaeG7g20gMTAwICgrMTApIHwKfCBbYFJFUE9SVC5tZGBdKFJFUE9SVC5tZCkgfCBN4bqrdSBiw6FvIGPDoW8gMTAgY8OidSB8CnwgYGxhYl9kYXkxMC5pcHluYmAgfCBLaHVuZyBub3RlYm9vayDEkeG7gyBi4bqhbiDEkWnhu4FuIG3DoyB8CnwgYGNvbmZpZ3MvZTDigKZlNS55YW1sYCB8IFNpw6p1IHRoYW0gc+G7kSB04burbmcgdGjDrSBuZ2hp4buHbSB8CnwgYHNjcmlwdHMvcHJlcGFyZV9kYXRhLnB5YCB8IFThuqNpIGThu68gbGnhu4d1LCBjaGlhIHNwbGl0IHwKfCBgc2NyaXB0cy9zY29yZS5weWAgfCAqKlRy4buNbmcgdMOgaSoqOiBjaOG6pW0gYHByZWRpY3Rpb25zLmpzb25sYCB8CnwgYHNjcmlwdHMvY2hlY2tfc3VibWlzc2lvbi5weWAgfCBLaeG7g20gdHJhIGLDoGkgdHLGsOG7m2Mga2hpIG7hu5lwIHwKCi0tLQoKIyMgMS4gTeG7pWMgdGnDqnUgaOG7jWMgdOG6rXAKClNhdSBiw6BpIGxhYiwgYuG6oW4gY8OzIHRo4buDOgoKMS4gTcO0IHThuqMgxJHGsOG7o2MgKipWaVQg4oaSIHByb2plY3RvciDihpIgTExNKiogY+G7p2EgUXdlbjMtVkwtNEIsIHbDoCBuw7NpIHLDtSBraOG7kWkgbsOgbyBi4bqhbiDEkcOjIGPhuq1wIG5o4bqtdCB0cm9uZyB04burbmcgdGjDrSBuZ2hp4buHbS4KMi4gRmluZXR1bmUgVkxNIGLhurFuZyAqKlFMb1JBKiogdsOgICoqc28gc8OhbmggYuG6sW5nIHPhu5EqKiBjw6FjIGPDoWNoIMSRw7NuZyBixINuZy9t4bufIGto4buRaSAoTG9SQSBMTE07IHRow6ptIHByb2plY3RvcjsgdGjDqm0gVmlUKS4KMy4gR2nhuqNpIHRow61jaCB2w6wgc2FvIHBo4bqjaSBjaGlhIGThu68gbGnhu4d1ICoqdGhlbyBwaGnDqm4gcXVheSoqIChraMO0bmcgY2hpYSBuZ+G6q3Ugbmhpw6puKSB2w6AgdsOsIHNhbyAqKmzhu4djaCB0aGFuZyB0b+G6oSDEkeG7mSAw4oCTMTAwMCoqIGzDoG0gaOG7j25nIGvhur90IHF14bqjLgo0LiDEkOG7jWMgRjEgKiprw6htIGtob+G6o25nIHRpbiBj4bqteSoqIHRyw6puIG3hu5l0IHThuq1wIHRlc3Qgbmjhu48gdsOgIGJp4bq/dCBr4bq/dCBsdeG6rW4gbsOgbyBjw7JuIMSR4bupbmcgduG7r25nLgo1LiAoVHXhu7MgY2jhu41uKSBWaeG6v3QgaMOgbSAqKnJld2FyZCBJb1UqKiBjaG8gUkwgduG7m2kgcGjhuqduIHRoxrDhu59uZyBraeG7g20gY2jhu6luZyDEkcaw4bujYy4KCiMjIDIuIELDoGkgdG/DoW4KCuG6om5oIGRhc2hjYW0gNjQww5czNjAg4oaSIG1vZGVsIHRy4bqjICoqbeG7mXQgSlNPTioqOgoKYGBganNvbgpbeyJsYWJlbCI6ICJwb3Rob2xlIiwgImJib3hfMmQiOiBbeDEsIHkxLCB4MiwgeTJdfV0KYGBgCgpUb+G6oSDEkeG7mSBsw6AgKipz4buRIG5ndXnDqm4gdHLDqm4gdGhhbmcgMOKAkzEwMDAqKiAodMawxqFuZyDEkeG7kWkgc28gduG7m2kgY2hp4buBdSBy4buZbmcvY2FvIOG6o25oKSwgxJHDum5nIHF1eSDGsOG7m2MgZ3JvdW5kaW5nIGPhu6dhIFF3ZW4zLVZMIHRyb25nIHNsaWRlIERheSAxMC4gQmEgbOG7m3A6IGBwb3Rob2xlYCAo4buVIGfDoCksIGBjcmFja2AgKG7hu6l0KSwgYG1hbmhvbGVgIChu4bqvcCBj4buRbmcpLiBN4buXaSDhuqNuaCA2NDDDlzM2MCBjaGnhur9tICoqMjIwIHRva2VuKiogdmlzaW9uLgoKIyMgMy4gROG7ryBsaeG7h3UKCkthZ2dsZSBbYGxvcmVuem9hcmNpb25pL3JvYWQtZGFtYWdlLWRhdGFzZXQtcG90aG9sZXMtY3JhY2tzLWFuZC1tYW5ob2xlc2BdKGh0dHBzOi8vd3d3LmthZ2dsZS5jb20vZGF0YXNldHMvbG9yZW56b2FyY2lvbmkvcm9hZC1kYW1hZ2UtZGF0YXNldC1wb3Rob2xlcy1jcmFja3MtYW5kLW1hbmhvbGVzKSwgZ2nhuqV5IHBow6lwICoqTUlUKiouCgp8IHwgfAp8LS0tfC0tLXwKfCDhuqJuaCB8IDIuMDA5IOG6o25oIEpQRyA2NDDDlzM2MCAoZGFzaGNhbSwgcXVheSB0aMOhbmcgMuKAkzMvMjAyNSkgfAp8IEJveCB8IDQuNzM3IChjcmFjayAyLjUxOSDCtyBwb3Rob2xlIDEuMjYxIMK3IG1hbmhvbGUgOTU3KSwgdHJ1bmcgYsOsbmggMiw0IGJveC/huqNuaCB8CnwgxJDhu5kga2jDsyB8IEJveCBuaOG7jzogZGnhu4duIHTDrWNoIHRydW5nIHbhu4sgY2jhu4kg4omIIDAsNyUga2h1bmcgaMOsbmggfAp8IE5ow6NuIGTDuW5nIHwgYGxhYmVscy1ZT0xPL2AgKGBjbGFzcyB4X2NlbnRlciB5X2NlbnRlciB3IGhgLCBjaHXhuqluIGhvw6EgWzAsMV0pIHwKCioqQ8OhYyDhuqNuaCBsw6AgZnJhbWUgY+G6r3QgdOG7qyB2aWRlbyoqLCBuw6puIGhhaSBmcmFtZSBsaeG7gW4ga+G7gSBn4bqnbiBuaMawIHRyw7luZyBuaGF1LiBO4bq/dSBjaGlhIG5n4bqrdSBuaGnDqm4sIOG6o25oIGfhuqduIGdp4buRbmcgbmhhdSBz4bq9IG7hurFtIOG7nyBj4bqjIHRyYWluIHbDoCB0ZXN0ICgqKnLDsiBy4buJKiopLiBgcHJlcGFyZV9kYXRhLnB5YCBnb20g4bqjbmggdGhlbyAqKnBoacOqbiBxdWF5KiogKG5nw6B5ICsgZ2nhu50gdHJvbmcgdMOqbiBmaWxlKSB2w6AgZ8OhbiAqKmPhuqMgcGhpw6puKiogdsOgbyBt4buZdCBzcGxpdC4gU3BsaXQgbsOgeSAqKmPhu5EgxJHhu4tuaCoqOiBhaSBjaOG6oXkgY8WpbmcgcmEgY8O5bmcga+G6v3QgcXXhuqMuCgp8IFNwbGl0IHwg4bqibmggfCBwb3Rob2xlIHwgY3JhY2sgfCBtYW5ob2xlIHwKfC0tLXwtLS18LS0tfC0tLXwtLS18CnwgdHJhaW4gfCAxLjQyMiB8IDEuMDc0IHwgMS43NTAgfCA2OTEgfAp8IHZhbCB8IDI3NiB8IDY2IHwgNDcwIHwgMTAzIHwKfCB0ZXN0IHwgMzExIHwgMTIxIHwgMjk5IHwgMTYzIHwKCkzGsHUgw706IHNwbGl0IHRoZW8gcGhpw6puIG7Dqm4gcGjDom4gYuG7kSBs4bubcCAqKmtow7RuZyDEkeG7gXUqKiAodmFsIGPDsyBy4bqldCDDrXQgcG90aG9sZSkuIMSQw7MgbMOgIHTDrG5oIGh14buRbmcgdGjhuq10LCB2w6AgbMOgIGzDvSBkbyBi4bqhbiBwaOG6o2kgxJHhu41jICoqRjEgdOG7q25nIGzhu5twKiogdsOgICoqa2hv4bqjbmcgdGluIGPhuq15KiouIETDuW5nICoqdmFsIMSR4buDIGfhu6EgbOG7l2kqKiwgY2jhu4kgY2jhuqVtICoqdGVzdCoqIGtoaSBu4buZcCBr4bq/dCBxdeG6oy4KCiMjIDQuIEPDoWMgdGjDrSBuZ2hp4buHbQoKfCBNw6MgfCBTZXR1cCB8IELhuq90IGJ14buZYyB8CnwtLS18LS0tfC0tLXwKfCBFMCB8IFplcm8tc2hvdCwga2jDtG5nIGh14bqlbiBsdXnhu4duIHwg4pyUIHwKfCBFMSB8IExvUkEgY2jhu4kgTExNOyBWaVQgdsOgIHByb2plY3RvciDEkcOzbmcgYsSDbmcgfCDinJQgfAp8IEUyIHwgRTEgKyBt4bufIMSR4bqneSDEkeG7pyBwcm9qZWN0b3IgKGB2aXN1YWwubWVyZ2VyYCwgYHZpc3VhbC5kZWVwc3RhY2tfbWVyZ2VyX2xpc3RgLCDiiYggMTA5TSB0aGFtIHPhu5EpIHwg4pyUIHwKfCBFMyB8IEUyICsgTG9SQSB0csOqbiBjw6FjIGto4buRaSBhdHRlbnRpb24vTUxQIGPhu6dhIFZpVCB8IOKclCB8CnwgRTQgfCBBYmxhdGlvbjogY2jhu41uICoqbeG7mXQqKiBiaeG6v24gKHJhbmsgLyBrw61jaCB0aMaw4bubYyBk4buvIGxp4buHdSAvIHRoYW5nIHRv4bqhIMSR4buZKSB8IOKclCAoY2jhu41uIDEpIHwKfCBFNSB8IEdSUE8gduG7m2kgcmV3YXJkIElvVSB8IHR14buzIGNo4buNbiwgKzEwIMSRaeG7g20gfAoKTeG7jWkgc2V0dXAgZMO5bmcgKipRTG9SQSA0LWJpdCoqIChORjQpLiBSacOqbmcgVmlUIHbDoCBgbG1faGVhZGAgKipraMO0bmcgbMaw4bujbmcgdOG7rSBob8OhKiogKHhlbSBHVUlERSBixrDhu5tjIDMgxJHhu4MgYmnhur90IHbDrCBzYW8pLgoKIyMgNS4gQ2h14bqpbiBi4buLICjiiYggMzAgcGjDunQpCgoxLiBN4bufIGBsYWJfZGF5MTAuaXB5bmJgIHRyw6puIENvbGFiOiBSdW50aW1lIOKGkiAqQ2hhbmdlIHJ1bnRpbWUgdHlwZSog4oaSICoqVDQgR1BVKiouIE1vdW50IEdvb2dsZSBEcml2ZSDEkeG7gyBsxrB1IGFkYXB0ZXIgKHNlc3Npb24gbWnhu4VuIHBow60gY8OzIHRo4buDIGLhu4sgbmfhuq90KS4KMi4gVOG6oW8gKipDb2xhYiBTZWNyZXRzKiogdMOqbiBgS0FHR0xFX1VTRVJOQU1FYCB2w6AgYEtBR0dMRV9LRVlgIGLhurFuZyB0w6BpIGtob+G6o24gS2FnZ2xlIGPhu6dhIGLhuqFuLiAqKktow7RuZyBnaGkga2hvw6EgdsOgbyBub3RlYm9vayBoYXkgY29tbWl0LioqCjMuIGBwaXAgaW5zdGFsbCAtciByZXF1aXJlbWVudHMtY29sYWIudHh0YCwgcuG7k2kgYHB5dGhvbiBzY3JpcHRzL3ByZXBhcmVfZGF0YS5weSAtLWthZ2dsZWAuCiAgIEtow7RuZyBjw7MgdMOgaSBraG/huqNuIEthZ2dsZT8gVOG6o2kgemlwIGRhdGFzZXQgduG7gSB0YXkgdsOgIGTDuW5nIGAtLXppcCA8xJHGsOG7nW5nIGThuqtuPmAgaG/hurdjIGAtLXNvdXJjZSA8dGjGsCBt4bulYyDEkcOjIGdp4bqjaSBuw6luPmAuCjQuIGBweXRob24gLW0gcHl0ZXN0IC1xYCDEkeG7gyBjaOG6r2MgY8O0bmcgY+G7pSBj4bunYSDEkeG7gSBjaOG6oXkgxJHDum5nIHRyw6puIG3DoXkgYuG6oW4uCgojIyA2LiBZw6p1IGPhuqd1IHbDoCBjw6FjaCBu4buZcAoKLSBDb2xhYiAqKlQ0IDE2IEdCKiouIE3hu5dpIGzhuqduIGh14bqlbiBsdXnhu4duIMaw4bubYyB0w61uaCAqKjIw4oCTMzAgcGjDunQqKiAoMzAwIGLGsOG7m2M7IGNvbiBz4buRIMaw4bubYyBsxrDhu6NuZywgaMOjeSBnaGkgdGjhu51pIGdpYW4gdGjhuq10IGPhu6dhIGLhuqFuIHbDoG8gYsOhbyBjw6FvKS4gTuG6v3UgdGjhuqV5IGjhur90IFZSQU0gaG/hurdjIHF1w6EgY2jhuq1tLCB4ZW0gYuG6o25nICJLaGkgZ+G6t3Agc+G7sSBj4buRIiB0cm9uZyBHVUlERS4KLSBO4buZcCAqKnJlcG8gR2l0SHViIGPDtG5nIGtoYWkqKiAo4omlIDUgY29tbWl0KSBn4buTbTogYHJlcG9ydC9ldmlkZW5jZS9lMOKAk2UzLmpzb25gLCDDrXQgbmjhuqV0IG3hu5l0IGBlNCouanNvbmAsIGBSRVBPUlQubWRgIMSRw6MgxJFp4buBbiwg4bqjbmggbWluaCBob+G6oSwgbm90ZWJvb2svbcOjIGPhu6dhIGLhuqFuLgotIFRyxrDhu5tjIGtoaSBu4buZcDogYHB5dGhvbiBzY3JpcHRzL2NoZWNrX3N1Ym1pc3Npb24ucHlgIGtow7RuZyBjw7JuIEZBSUwuCi0gKipLaMO0bmcgY29tbWl0Kio6IGBkYXRhL2AsIGB3b3JrL2AsIGAuZW52YCwgdHLhu41uZyBz4buRIChgKi5zYWZldGVuc29yc2AsIGAqLmJpbmAsIGAqLnB0YCkuCi0gKipLaMO0bmcgc+G7rWEqKiBgc2NyaXB0cy9gIHbDoCBgdGVzdHMvYCBj4bunYSDEkeG7gS4KLSBMw6BtIGPDoSBuaMOibjogYuG6oW4gY8OzIHRo4buDIHRo4bqjbyBsdeG6rW4gw70gdMaw4bufbmcsIG5oxrBuZyBtw6MsIHPhu5EgbGnhu4d1IHbDoCBiw6FvIGPDoW8gcGjhuqNpIGzDoCBj4bunYSBi4bqhbi4gR2nhuqNuZyB2acOqbiDEkeG7kWkgY2hp4bq/dSBjw6FjIHJlcG8gduG7m2kgbmhhdSB2w6AgY8OzIHRo4buDIGfhu41pICoqduG6pW4gxJHDoXAgNSBwaMO6dCoqLgo=", "GUIDE.md": "IyBIxrDhu5tuZyBk4bqrbiB0aOG7sWMgaGnhu4duIExhYiBEYXkgMTAKCj4gxJDhu41jIFtgUkVBRE1FLm1kYF0oUkVBRE1FLm1kKSB0csaw4bubYy4gTeG7l2kgYsaw4bubYyBn4buTbSAqKk3hu6VjIHRpw6p1IMK3IFBzZXVkbyBjb2RlIMK3IEtp4buDbSB0cmEgwrcgQ8OidSBo4buPaSoqICjEkWnhu4FuIHbDoG8gW2BSRVBPUlQubWRgXShSRVBPUlQubWQpKS4gUHNldWRvIGNvZGUgKipraMO0bmcgY2jhuqF5IMSRxrDhu6NjIG5nYXkqKjogYuG6oW4gdOG7sSB2aeG6v3QgbcOjIHRo4bqtdCB0cm9uZyBub3RlYm9vayAoaG/hurdjIGZpbGUgYC5weWAgY+G7p2EgYuG6oW4pLgo+IFF1eSDGsOG7m2M6IHTDqm4gbW9kdWxlIGzhuqV5IHThu6sgY8OieSBtb2RlbCBRd2VuMy1WTC00QiB0aOG6rXQuIEjDo3kgdOG7sSBpbiBgW24gZm9yIG4sIF8gaW4gbW9kZWwubmFtZWRfbW9kdWxlcygpXWAgxJHhu4MgeMOhYyBuaOG6rW4uCgp8IFBow7p0IHwgTuG7mWkgZHVuZyB8CnwtLS18LS0tfAp8IDDigJMzMCB8IELGsOG7m2MgMOKAkzE6IG3DtGkgdHLGsOG7nW5nLCBk4buvIGxp4buHdSB8CnwgMzDigJM2MCB8IELGsOG7m2MgMuKAkzM6IG5ow6NuIEpTT04sIEUwIHplcm8tc2hvdCB8CnwgNjDigJM5MCB8IELGsOG7m2MgNOKAkzU6IGThu68gbGnhu4d1IGh14bqlbiBsdXnhu4duLCBzZXR1cCBMb1JBIHwKfCA5MOKAkzIwMCB8IELGsOG7m2MgNuKAkzc6IGh14bqlbiBsdXnhu4duIHbDoCBjaOG6pW0gRTEsIEUyLCBFMyB8CnwgMjAw4oCTMjMwIHwgQsaw4bubYyA4OiBFNCB8CnwgMjMwKyB8IELGsOG7m2MgOSAodHXhu7MgY2jhu41uKSB2w6AgYsOhbyBjw6FvIHwKCi0tLQoKIyMgQsaw4bubYyAwIMK3IE3DtGkgdHLGsOG7nW5nCgpgYGBiYXNoCnBpcCBpbnN0YWxsIC1yIHJlcXVpcmVtZW50cy1jb2xhYi50eHQKcHl0aG9uIC1tIHB5dGVzdCAtcQpgYGAKCk7hur91IGfhurdwIGBJbXBvcnRFcnJvciAuLi4gcmVxdWlyZXMgdGhlIFRvcmNodmlzaW9uIGxpYnJhcnlgOiBjw6BpIGB0b3JjaHZpc2lvbmAgY8O5bmcgYuG6o24gdG9yY2ggY+G7p2EgQ29sYWIuCgojIyBCxrDhu5tjIDEgwrcgROG7ryBsaeG7h3UKCmBgYGJhc2gKcHl0aG9uIHNjcmlwdHMvcHJlcGFyZV9kYXRhLnB5IC0ta2FnZ2xlCmBgYAoKU2luaCBgZGF0YS9pbWFnZXMvYCwgYGRhdGEvc3BsaXRzL3t0cmFpbix2YWwsdGVzdH0uanNvbmxgIHbDoCBgZGF0YS9zcGxpdHMvbWV0YS5qc29uYC4gTeG7l2kgZMOybmcgYC5qc29ubGA6CgpgYGBqc29uCnsiaW1hZ2VfaWQiOiAiLi4uIiwgImltYWdlX3BhdGgiOiAiaW1hZ2VzLy4uLi5qcGciLCAid2lkdGgiOiA2NDAsICJoZWlnaHQiOiAzNjAsCiAic2Vzc2lvbiI6ICJ2bGMtMjAyNS0wMi0xOC0yMyIsICJvYmplY3RzIjogW3sibGFiZWwiOiAicG90aG9sZSIsICJ5b2xvIjogW3hjLCB5YywgdywgaF19XX0KYGBgCgoqKktp4buDbSB0cmE6KiogaW4gc+G7kSDhuqNuaC9ib3ggdOG7q25nIHNwbGl0OyB24bq9IDYg4bqjbmggY8OzIGJveCBsw6puIMSR4buDIGNo4bqvYyBi4bqhbiBoaeG7g3UgbmjDo24uIFjDoWMgbmjhuq1uIGtow7RuZyBgc2Vzc2lvbmAgbsOgbyB4deG6pXQgaGnhu4duIOG7nyBoYWkgc3BsaXQuCgoqKkPDonUgaOG7j2k6KiogdsOsIHNhbyBjaGlhIHRoZW8gYHNlc3Npb25gIG3DoCBraMO0bmcgY2hpYSBuZ+G6q3Ugbmhpw6puPyDEkGnhu4F1IGfDrCB44bqjeSByYSB24bubaSBGMSBu4bq/dSBjaGlhIG5n4bqrdSBuaGnDqm4/CgojIyBCxrDhu5tjIDIgwrcgTmjDo24gWU9MTyDihpIgSlNPTiB0aGFuZyAw4oCTMTAwMAoKYGBgCmjDoG0geW9sb190b19ib3gobGFiZWwsIHlvbG8pOgogICAgeGMsIHljLCB3LCBoID0geW9sbyAgICAgICAgICAgICAgICAgICAgICAgICAgIyBjaHXhuqluIGhvw6EgWzAsMV0sIHTDom0gKyBrw61jaCB0aMaw4bubYwogICAgeDEgPSAoeGMgLSB3LzIpICogMTAwMDsgICB5MSA9ICh5YyAtIGgvMikgKiAxMDAwCiAgICB4MiA9ICh4YyArIHcvMikgKiAxMDAwOyAgIHkyID0gKHljICsgaC8yKSAqIDEwMDAKICAgIGzDoG0gdHLDsm4gdGjDoG5oIHPhu5Egbmd1ecOqbiwga+G6uXAgdsOgbyBbMCwgMTAwMF0KICAgIHRy4bqjIHsibGFiZWwiOiBsYWJlbCwgImJib3hfMmQiOiBbeDEsIHkxLCB4MiwgeTJdfQoKaMOgbSBvYmplY3RzX3RvX2Fuc3dlcihvYmplY3RzKToKICAgIGJveGVzID0gW3lvbG9fdG9fYm94KC4uLikgY2hvIG3hu5dpIG9iamVjdF0KICAgIHPhuq9wIGJveGVzIHRoZW8gKHkxLCB4MSkgICAgICAgICAgICAgICAgICAgICAgIyB0csOqbuKGkmTGsOG7m2ksIHRyw6Fp4oaScGjhuqNpOiB0aOG7qSB04buxIOG7lW4gxJHhu4tuaCBjaG8gbW9kZWwgaOG7jWMKICAgIHRy4bqjIGpzb24uZHVtcHMoYm94ZXMsIHNlcGFyYXRvcnM9KCIsIiwgIjoiKSkgIyAiW10iIG7hur91IGtow7RuZyBjw7Mgb2JqZWN0CmBgYAoKVmnhur90ICoqcHJvbXB0KiogecOqdSBj4bqndSBtb2RlbCBjaOG7iSB0cuG6oyBKU09OIGxpc3QsIG7DqnUgMyBs4bubcCB2w6AgbsOzaSByw7UgInRoYW5nIDDigJMxMDAwIi4KCioqS2nhu4NtIHRyYToqKiB24bubaSBt4buZdCDhuqNuaCwgY2h1eeG7g24gbmjDo24gc2FuZyBKU09OIHLhu5NpIHF1eSBuZ8aw4bujYyB24buBIHBpeGVsIHbDoCB24bq9IGzhuqFpOyBib3ggcGjhuqNpIHRyw7luZyBib3ggZ+G7kWMuCgoqKkPDonUgaOG7j2k6KiogdsOsIHNhbyBRd2VuMy1WTCBkw7luZyB0aGFuZyAw4oCTMTAwMCBjaOG7qSBraMO0bmcgZMO5bmcgcGl4ZWwgdHV54buHdCDEkeG7kWk/IChTbGlkZSAwNCwgIsSR4bqndSByYSBncm91bmRpbmciKS4KCiMjIELGsOG7m2MgMyDCtyBFMDogemVyby1zaG90CgpgYGAKcXVhbnRpemF0aW9uID0gQml0c0FuZEJ5dGVzQ29uZmlnKGxvYWRfaW5fNGJpdD1UcnVlLCBibmJfNGJpdF9xdWFudF90eXBlPSJuZjQiLAogICAgICAgICAgICAgICAgICBibmJfNGJpdF9jb21wdXRlX2R0eXBlPWZsb2F0MTYsIGJuYl80Yml0X3VzZV9kb3VibGVfcXVhbnQ9VHJ1ZSwKICAgICAgICAgICAgICAgICAgbGxtX2ludDhfc2tpcF9tb2R1bGVzPVsidmlzdWFsIiwgImxtX2hlYWQiXSkKbW9kZWwgPSBRd2VuM1ZMRm9yQ29uZGl0aW9uYWxHZW5lcmF0aW9uLmZyb21fcHJldHJhaW5lZCgKICAgICAgICAgICAgIlF3ZW4vUXdlbjMtVkwtNEItSW5zdHJ1Y3QiLCBxdWFudGl6YXRpb25fY29uZmlnPXF1YW50aXphdGlvbiwKICAgICAgICAgICAgZHR5cGU9ZmxvYXQxNiwgZGV2aWNlX21hcD17IiI6IDB9KQpwcm9jZXNzb3IgPSBBdXRvUHJvY2Vzc29yLmZyb21fcHJldHJhaW5lZCguLi4pOyAgcHJvY2Vzc29yLnRva2VuaXplci5wYWRkaW5nX3NpZGUgPSAibGVmdCIKCnbhu5tpIG3hu5dpIGJhdGNoIOG6o25oIHRlc3QgKGJhdGNoIDQpOgogICAgbWVzc2FnZXMgPSBbeyJyb2xlIjogInVzZXIiLCAiY29udGVudCI6IFt7InR5cGUiOiAiaW1hZ2UiLCAiaW1hZ2UiOiBpbWd9LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB7InR5cGUiOiAidGV4dCIsICAidGV4dCI6IFBST01QVH1dfV0KICAgIGlucHV0cyA9IHByb2Nlc3Nvci5hcHBseV9jaGF0X3RlbXBsYXRlKG1lc3NhZ2VzLCB0b2tlbml6ZT1UcnVlLCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJldHVybl9kaWN0PVRydWUsIHJldHVybl90ZW5zb3JzPSJwdCIsIHBhZGRpbmc9VHJ1ZSkKICAgIG91dCA9IG1vZGVsLmdlbmVyYXRlKCoqaW5wdXRzLCBtYXhfbmV3X3Rva2Vucz0yNTYsIGRvX3NhbXBsZT1GYWxzZSkKICAgIHJhdyA9IGRlY29kZShvdXRbOiwgxJHhu5lfZMOgaV9wcm9tcHQ6XSwgc2tpcF9zcGVjaWFsX3Rva2Vucz1UcnVlKQogICAgZ2hpIHsiaW1hZ2VfaWQiOiAuLi4sICJyYXciOiByYXd9IHbDoG8gd29yay9lMC5qc29ubCAgICAgICMgbeG7l2kgZMOybmcgbeG7mXQg4bqjbmgKYGBgCgpDaOG6pW06CgpgYGBiYXNoCnB5dGhvbiBzY3JpcHRzL3Njb3JlLnB5IC0tcHJlZCB3b3JrL2UwLmpzb25sIC0tbmFtZSBFMCAtLW91dCByZXBvcnQvZXZpZGVuY2UvZTAuanNvbgpgYGAKCj4gYGxsbV9pbnQ4X3NraXBfbW9kdWxlcz1bInZpc3VhbCIsImxtX2hlYWQiXWAgZ2nhu68gVmlUIChr4buDIGPhuqMgcHJvamVjdG9yKSDhu58gZnAxNi4gTuG6v3UgVmlUIGLhu4sgbMaw4bujbmcgdOG7rSBob8OhIDQtYml0LCB0cuG7jW5nIHPhu5EgY+G7p2EgbWVyZ2VyIGzDoCB0cuG7jW5nIHPhu5EgbMaw4bujbmcgdOG7rSB2w6AgKipraMO0bmcgdGjhu4MgaHXhuqVuIGx1eeG7h24gxJHhuqd5IMSR4bunKiog4bufIEUyL0UzLgoKKipLaeG7g20gdHJhOioqIGBwYXJzZV92YWxpZF9yYXRlYCBsw6AgYmFvIG5oacOqdT8gSW4gNSBvdXRwdXQgdGjDtCDEkeG7gyB4ZW0gbW9kZWwgdHLhuqMgZ8OsIChjw7MgYuG7jWMgYGBganNvbiBraMO0bmc/IGPDsyBjaOG7ryB0aOG7q2Ega2jDtG5nPykuCgoqKkPDonUgaOG7j2k6KiogaGFpIGzhu5dpIHBo4buVIGJp4bq/biBuaOG6pXQgY+G7p2EgRTAgbMOgIGfDrCAoc2FpIHRoYW5nLCBi4buLYSBib3gsIHNhaSBs4bubcCwgSlNPTiBo4buPbmcpPyBLw6htIG3hu5l0IHbDrSBk4bulIG91dHB1dC4KCiMjIELGsOG7m2MgNCDCtyBWw60gZOG7pSBodeG6pW4gbHV54buHbiB2w6AgbWFzayBsb3NzCgpgYGAKaMOgbSBlbmNvZGVfZXhhbXBsZShpbWFnZSwgcHJvbXB0LCBhbnN3ZXIpOgogICAgcHJvbXB0X2VuYyA9IHByb2Nlc3Nvci5hcHBseV9jaGF0X3RlbXBsYXRlKFt1c2VyXSwgYWRkX2dlbmVyYXRpb25fcHJvbXB0PVRydWUsCiAgICAgICAgICAgICAgICAgICAgIHRva2VuaXplPVRydWUsIHJldHVybl9kaWN0PVRydWUsIHJldHVybl90ZW5zb3JzPSJwdCIpCiAgICBmdWxsX2VuYyAgID0gcHJvY2Vzc29yLmFwcGx5X2NoYXRfdGVtcGxhdGUoW3VzZXIsIGFzc2lzdGFudChhbnN3ZXIpXSwgYWRkX2dlbmVyYXRpb25fcHJvbXB0PUZhbHNlLCAuLi4pCiAgICBuID0gxJHhu5kgZMOgaSBwcm9tcHRfZW5jLmlucHV0X2lkcwogICAgS0nhu4JNIFRSQSBmdWxsX2VuYy5pbnB1dF9pZHNbMCwgOm5dID09IHByb21wdF9lbmMuaW5wdXRfaWRzWzBdICAgICMgcHJvbXB0IGzDoCB0aeG7gW4gdOG7kSBj4bunYSBjaHXhu5dpIMSR4bqneSDEkeG7pwogICAgbGFiZWxzID0gYuG6o24gc2FvIGPhu6dhIGZ1bGxfZW5jLmlucHV0X2lkczsgIGxhYmVsc1s6LCA6bl0gPSAtMTAwICAgIyBjaOG7iSBo4buNYyBwaOG6p24gdHLhuqMgbOG7nWkKICAgIHRy4bqjIGZ1bGxfZW5jIGvDqG0gbGFiZWxzCgpow6BtIGNvbGxhdGUoYmF0Y2gpOgogICAgcGFkIGlucHV0X2lkcyAvIGF0dGVudGlvbl9tYXNrIC8gbW1fdG9rZW5fdHlwZV9pZHMgLyBsYWJlbHMgxJHhur9uIGPDuW5nIMSR4buZIGTDoGkgKGxhYmVscyDEkeG7h20gPSAtMTAwKQogICAgcGl4ZWxfdmFsdWVzLCBpbWFnZV9ncmlkX3RodzogbuG7kWkgKGNhdCkgdGhlbyBjaGnhu4F1IDAsIEtIw5RORyBwYWQKYGBgCgoqKktp4buDbSB0cmE6KiogduG7m2kgbeG7mXQgdsOtIGThu6UsIGluIHPhu5EgdG9rZW4g4bqjbmggKHBo4bqjaSBsw6AgKioyMjAqKiBjaG8g4bqjbmggNjQww5czNjApLCB2w6AgZ2nhuqNpIG3DoyByacOqbmcgY8OhYyB0b2tlbiBjw7MgYGxhYmVscyAhPSAtMTAwYDoga+G6v3QgcXXhuqMgcGjhuqNpIMSRw7puZyBsw6AgSlNPTiDEkcOhcCDDoW4gKGPhu5luZyBgPHxpbV9lbmR8PmApLgoKKipDw6J1IGjhu49pOioqIHbDrCBzYW8ga2jDtG5nIHTDrW5oIGxvc3MgdHLDqm4gcHJvbXB0IHbDoCB0b2tlbiDhuqNuaD8gxJBp4buBdSBnw6wgeOG6o3kgcmEgbuG6v3UgdMOtbmg/CgojIyBCxrDhu5tjIDUgwrcgU2V0dXAgTG9SQSAvIMSRw7NuZyBixINuZyAoRTEsIEUyLCBFMykKCmBgYApow6BtIGFwcGx5X3NldHVwKG1vZGVsLCBzZXR1cCk6CiAgICBt4buNaSB0aGFtIHPhu5E6IHJlcXVpcmVzX2dyYWQgPSBGYWxzZQogICAgbW9kZWwuZW5hYmxlX2lucHV0X3JlcXVpcmVfZ3JhZHMoKSAgICAgICAgICAgICMgY+G6p24gY2hvIGdyYWRpZW50IGNoZWNrcG9pbnRpbmcKICAgIG3hu6VjIHRpw6p1IExvUkEgKHJlZ2V4IHRyw6puIHTDqm4gbW9kdWxlKToKICAgICAgICBMTE0gKEUxLCBFMiwgRTMpOiBtb2RlbC5sYW5ndWFnZV9tb2RlbC5sYXllcnMuTi5zZWxmX2F0dG4ue3Esayx2LG99X3Byb2oKICAgICAgICAgICAgICAgICAgICAgICAgICBtb2RlbC5sYW5ndWFnZV9tb2RlbC5sYXllcnMuTi5tbHAue2dhdGUsdXAsZG93bn1fcHJvaiAgICAgICMgMzYgw5cgNyA9IDI1MiBtb2R1bGUKICAgICAgICBWaVQgKGNo4buJIEUzKTogICAgIG1vZGVsLnZpc3VhbC5ibG9ja3MuTi5hdHRuLntxa3YscHJvan0KICAgICAgICAgICAgICAgICAgICAgICAgICBtb2RlbC52aXN1YWwuYmxvY2tzLk4ubWxwLntsaW5lYXJfZmMxLGxpbmVhcl9mYzJ9ICAgICAgICAgICMgMjQgw5cgNCA9IDk2IG1vZHVsZQogICAgcGVmdF9tb2RlbCA9IGdldF9wZWZ0X21vZGVsKG1vZGVsLCBMb3JhQ29uZmlnKHI9MTYsIGxvcmFfYWxwaGE9MzIsIGxvcmFfZHJvcG91dD0wLjA1LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRhcmdldF9tb2R1bGVzPTxyZWdleD4pKQogICAgbuG6v3Ugc2V0dXAgbMOgIEUyIGhv4bq3YyBFMzoKICAgICAgICB24bubaSBt4buNaSB0aGFtIHPhu5EgY8OzIHTDqm4gY2jhu6lhICJ2aXN1YWwubWVyZ2VyIiBob+G6t2MgInZpc3VhbC5kZWVwc3RhY2tfbWVyZ2VyX2xpc3QiOgogICAgICAgICAgICDEkcawYSB24buBIGZsb2F0MzI7ICByZXF1aXJlc19ncmFkID0gVHJ1ZQogICAgaW4gdOG7lW5nIHPhu5EgdGhhbSBz4buRIGh14bqlbiBsdXnhu4duCmBgYAoKR+G7o2kgw70gcXV5IG3DtCAoxJHhu4MgYuG6oW4gxJHhu5FpIGNoaeG6v3UpOiBMTE0gKyBlbWJlZGRpbmcg4omIIDQsMEI7IFZpVCDiiYggNDE1TTsgdHJvbmcgVmlULCBgdmlzdWFsLm1lcmdlcmAg4omIIDI3TSB2w6AgYHZpc3VhbC5kZWVwc3RhY2tfbWVyZ2VyX2xpc3RgIOKJiCA4Mk0gKERlZXBTdGFjayBs4bqleSDEkeG6t2MgdHLGsG5nIOG7nyBs4bubcCA1LCAxMSwgMTcgY+G7p2EgVmlUKS4gVOG7lW5nIHRoYW0gc+G7kSBRd2VuMy1WTC00QiDiiYggNCw0NEIuCgoqKktp4buDbSB0cmE6KiogYGxlbihtYXRjaGVkX21vZHVsZXMpYCBwaOG6o2kgbMOgIDI1MiAoRTEsIEUyKSB2w6AgMzQ4IChFMykuIFPhu5EgdGhhbSBz4buRIGh14bqlbiBsdXnhu4duIGPhu6dhIEUyIHBo4bqjaSBs4bubbiBoxqFuIEUxIGtob+G6o25nIDEwOU0uCgoqKkPDonUgaOG7j2k6KiogdsOsIHNhbyBtZXJnZXIgcGjhuqNpIGh14bqlbiBsdXnhu4duIOG7nyBmbG9hdDMyIHRyb25nIGtoaSBjw6FjIHRy4buNbmcgc+G7kSBjw7JuIGzhuqFpIOG7nyBmcDE2PyBWw6wgc2FvIGtow7RuZyB0aOG7gyBodeG6pW4gbHV54buHbiDEkeG6p3kgxJHhu6cgbWVyZ2VyIG7hur91IFZpVCBi4buLIGzGsOG7o25nIHThu60gaG/DoSA0LWJpdD8KCiMjIELGsOG7m2MgNiDCtyBWw7JuZyBodeG6pW4gbHV54buHbgoKYGBgCnThu5FpIMawdSA9IEFkYW1XKGPDoWMgdGhhbSBz4buRIHJlcXVpcmVzX2dyYWQsIGxyPTJlLTQsIHdlaWdodF9kZWNheT0wKQps4buLY2ggbHI6IHdhcm11cCA1JSBixrDhu5tjIMSR4bqndSwgc2F1IMSRw7MgY29zaW5lIHbhu4EgMApzY2FsZXIgPSBHcmFkU2NhbGVyKCJjdWRhIikgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgZnAxNgpi4bqtdCBncmFkaWVudCBjaGVja3BvaW50aW5nICh1c2VfcmVlbnRyYW50PUZhbHNlKTsgbW9kZWwuY29uZmlnLnVzZV9jYWNoZSA9IEZhbHNlCmzhurdwIMSR4bq/biBraGkgxJHhu6cgbWF4X3N0ZXBzID0gMzAwIGLGsOG7m2MgdOG7kWkgxrB1OgogICAgbOG6t3AgNCBs4bqnbiAodMOtY2ggbHXhu7kgZ3JhZGllbnQpLCBt4buXaSBs4bqnbiBt4buZdCBiYXRjaCAyIOG6o25oOgogICAgICAgIHbhu5tpIGF1dG9jYXN0KGZwMTYpOiAgbG9zcyA9IG1vZGVsKCoqYmF0Y2gpLmxvc3MgLyA0CiAgICAgICAgbuG6v3UgbG9zcyBraMO0bmcgaOG7r3UgaOG6oW46IEThu6pORyB2w6AgYsOhbyAobmd1eSBjxqEgdHLDoG4gc+G7kSBmcDE2KQogICAgICAgIHNjYWxlci5zY2FsZShsb3NzKS5iYWNrd2FyZCgpCiAgICBzY2FsZXIudW5zY2FsZV8ob3B0KTsgY2xpcF9ncmFkX25vcm1fKHBhcmFtcywgMS4wKTsgc2NhbGVyLnN0ZXAob3B0KTsgc2NhbGVyLnVwZGF0ZSgpOyBvcHQuemVyb19ncmFkKCkKICAgIG3hu5dpIDEwMCBixrDhu5tjOiBsxrB1IGFkYXB0ZXIgIFbDgCAgbMawdSByacOqbmcgY8OhYyB0aGFtIHPhu5EgcHJvamVjdG9yIMSRw6MgaHXhuqVuIGx1eeG7h24gKEUyLCBFMykKZ2hpIHJ1bi5qc29uOiB0cmFpbmFibGVfcGFyYW1zLCBwZWFrX3ZyYW1fZ2IgKHRvcmNoLmN1ZGEubWF4X21lbW9yeV9hbGxvY2F0ZWQpLCB0cmFpbl9taW51dGVzCmBgYAoKPiBgbW9kZWwuc2F2ZV9wcmV0cmFpbmVkKClgIGPhu6dhIHBlZnQgKipjaOG7iSBsxrB1IGFkYXB0ZXIgTG9SQSoqLiBQcm9qZWN0b3IgKEUyL0UzKSDEkcaw4bujYyBodeG6pW4gbHV54buHbiDEkeG6p3kgxJHhu6csICoqa2jDtG5nIG7hurFtIHRyb25nIGFkYXB0ZXIqKjogbuG6v3UgcXXDqm4gbMawdSByacOqbmcsIGzDumMgc3V5IGx14bqtbiBi4bqhbiBz4bq9IG7huqFwIGzhuqFpIHByb2plY3RvciBn4buRYyB2w6Aga+G6v3QgcXXhuqMgRTIvRTMgdHLDtG5nIGdp4buRbmcgRTEuCj4gQ2jDqXAgYWRhcHRlciBzYW5nIEdvb2dsZSBEcml2ZSBzYXUgbeG7l2kgbOG6p24gY2jhuqF5OiBzZXNzaW9uIG1p4buFbiBwaMOtIGPDsyB0aOG7gyBi4buLIG5n4bqvdC4KCioqS2nhu4NtIHRyYToqKiBsb3NzIGdp4bqjbSDEkeG7gXUgdsOgIGjhu691IGjhuqFuOyBnaGkgbOG6oWkgVlJBTSDEkeG7iW5oIHbDoCB0aOG7nWkgZ2lhbi4KCioqQ8OidSBo4buPaToqKiBzZXR1cCBuw6BvIGTDuW5nIG5oaeG7gXUgVlJBTSBuaOG6pXQgdsOgIHbDrCBzYW8gKGfhu6NpIMO9OiBncmFkaWVudCBxdWEgVmlULCB0aGFtIHPhu5EgZmxvYXQzMiBj4bunYSBwcm9qZWN0b3IpPwoKIyMgQsaw4bubYyA3IMK3IFN1eSBsdeG6rW4gdsOgIGNo4bqlbSBFMeKAk0UzCgpgYGAKbuG6oXAgbW9kZWwgZ+G7kWMgKGPDuW5nIGPhuqV1IGjDrG5oIDQtYml0IG5oxrAgYsaw4bubYyAzKSDihpIgbuG6oXAgYWRhcHRlciBMb1JBIOKGkiBu4bqhcCBs4bqhaSB0aGFtIHPhu5EgcHJvamVjdG9yIChu4bq/dSBjw7MpCnN1eSBsdeG6rW4gdHLDqm4gdGVzdCBuaMawIGLGsOG7m2MgMyDihpIgd29yay9lMS5qc29ubApgYGAKCmBgYGJhc2gKcHl0aG9uIHNjcmlwdHMvc2NvcmUucHkgLS1wcmVkIHdvcmsvZTEuanNvbmwgLS1uYW1lIEUxIC0tb3V0IHJlcG9ydC9ldmlkZW5jZS9lMS5qc29uIFwKICAgIC0tbWV0YSB0cmFpbmFibGVfcGFyYW1zPTxz4buRPiAtLW1ldGEgcGVha192cmFtX2diPTxz4buRPiAtLW1ldGEgdHJhaW5fbWludXRlcz08c+G7kT4KYGBgCgpN4bq5bzogZ+G7oSBs4buXaSB0csOqbiA0MCDhuqNuaCBj4bunYSBgdmFsYCAoYC0tc3BsaXQgdmFsYCk7IGNo4buJIGNo4bqlbSBgdGVzdGAgbeG7mXQgbOG6p24gY2hvIG3hu5dpIHNldHVwLgoKKipLaeG7g20gdHJhOioqIEUxIHBo4bqjaSBoxqFuIGjhurNuIEUwLiBO4bq/dSBraMO0bmcsIGtp4buDbSB0cmEgbWFzayBsb3NzIChixrDhu5tjIDQpIHbDoCB0aGFuZyB0b+G6oSDEkeG7mSAoYsaw4bubYyAyKS4KCioqQ8OidSBo4buPaToqKiBs4bqtcCBi4bqjbmcgRTDigJNFMyB0cm9uZyBgUkVQT1JULm1kYCAoRjEsIENJOTUsIEZQL+G6o25oLCBGMSB04burbmcgbOG7m3AsIFZSQU0sIHRo4budaSBnaWFuKS4gU2V0dXAgbsOgbyDEkcOhbmcgxJHhu5NuZyB0aeG7gW4gbmjhuqV0PyBDaMOqbmggbOG7h2NoIGPDsyB2xrDhu6N0IGtob+G6o25nIHRpbiBj4bqteSBraMO0bmc/CgojIyBCxrDhu5tjIDggwrcgRTQ6IGFibGF0aW9uIChjaOG7jW4gbeG7mXQpCgrEkOG7lWkgKirEkcO6bmcgbeG7mXQgYmnhur9uKiogc28gduG7m2kgc2V0dXAgdOG7kXQgbmjhuqV0IGPhu6dhIEUx4oCTRTMgKHhlbSBgY29uZmlncy9lNC55YW1sYCkgdsOgIGNo4bqheSBj4bqjIGhhaSBnacOhIHRy4buLICht4buZdCB0cm9uZyBoYWkgY8OzIHRo4buDIGNow61uaCBsw6Agc2V0dXAgZ+G7kWMpOgoKfCBCaeG6v24gfCBDw6FjaCDEkeG7lWkgfAp8LS0tfC0tLXwKfCBgcmFua2AgfCByID0gNCB2w6AgciA9IDY0ICjEkeG6t3QgYWxwaGEgPSAywrdyKSB8CnwgYGRhdGFfc2l6ZWAgfCAxMCUgdsOgIDEwMCUg4bqjbmggdHJhaW4gfAp8IGBjb29yZF9zY2FsZWAgfCBgbm9ybTEwMDBgIHbDoCBgcGl4ZWxgOiDEkeG7lWkgKipj4bqjKiogbmjDo24sIHByb21wdCwgdsOgIGNo4bqlbSB24bubaSBgLS1jb29yZCBwaXhlbGAgfAoKTMawdTogYHJlcG9ydC9ldmlkZW5jZS9lNF88Ymnhur9uPl88Z2nDoSB0cuG7iz4uanNvbmAuCgoqKkPDonUgaOG7j2k6Kioga+G6v3QgbHXhuq1uIGPhu6dhIGLhuqFuIGPDsyDEkcOhbmcgdGluIGtow7RuZyBraGkgdGVzdCBjaOG7iSBjw7MgMzExIOG6o25oPyAoTmjDrG4gQ0k5NS4pCgojIyBCxrDhu5tjIDkgwrcgRTUgKHR14buzIGNo4buNbiwgKzEwIMSRaeG7g20pOiBHUlBPIHbhu5tpIHJld2FyZCBJb1UKCmBgYApow6BtIHJld2FyZChjb21wbGV0aW9uLCBndF9ib3hlcyk6CiAgICBwcmVkID0gcGFyc2Vfb3V0cHV0KGNvbXBsZXRpb24pICAgICAgICAgICAgICMgZMO5bmcgxJHGsOG7o2Mgc2NyaXB0cy5zY29yZS5wYXJzZV9vdXRwdXQKICAgIG7hur91IHByZWQgbMOgIE5vbmU6IHRy4bqjIDAuMCAgICAgICAgICAgICAgICAgICAjIEpTT04gaOG7j25nCiAgICBu4bq/dSBraMO0bmcgY8OzIGd0IHbDoCBraMO0bmcgY8OzIHByZWQ6IHRy4bqjIDEuMAogICAgZ2jDqXAgxJHDtGkgdGhhbSBsYW0gdGhlbyBJb1UgZ2nhuqNtIGThuqduLCBjw7luZyBs4bubcCwgY2jhu4kgbmjhuq1uIElvVSDiiaUgMC41CiAgICB04buVbmdfaW91ID0gdOG7lW5nIElvVSBjw6FjIGPhurdwIGto4bubcAogICAgdHLhuqMgMC4yICsgMC44ICogdOG7lW5nX2lvdSAvIG1heChz4buRX2d0LCBz4buRX3ByZWQsIDEpICAgICMgYm94IGLhu4thIGLhu4sgcGjhuqF0IHF1YSBt4bqrdSBz4buRCmBgYAoKVmnhur90ICoqdGVzdCBjaG8gcmV3YXJkIHRyxrDhu5tjIGtoaSBjaOG6oXkgR1JQTyoqIChob8OgbiBo4bqjbywgSlNPTiBo4buPbmcsIHNhaSBs4bubcCwgdGjhu6thIGJveCwgcuG7l25nKS4gSMOgbSByZXdhcmQgbMOgIFB5dGhvbiB0aHXhuqduIG7Dqm4gdGVzdCDEkcaw4bujYyBtw6Aga2jDtG5nIGPhuqduIEdQVS4KCkdSUE86IGto4bufaSB04bqhbyB04burIGFkYXB0ZXIgdOG7kXQgbmjhuqV0LCBgbnVtX2dlbmVyYXRpb25zPTRgLCBgbWF4X2NvbXBsZXRpb25fbGVuZ3RoPTEyOGAsIGtob+G6o25nIDYwIGLGsOG7m2MsIGB1c2VfdmxsbT1GYWxzZWAuIETDuW5nIGB0cmwuR1JQT1RyYWluZXJgICjEkcOjIGdoaW0gYHRybD09MS45LjJgOyBow6N5IMSR4buNYyBkb2NzIGPhu6dhIMSRw7puZyBwaGnDqm4gYuG6o24gbsOgeSkuCgoqKkPDonUgaOG7j2k6KiogY8OzIGThuqV1IGhp4buHdSByZXdhcmQgaGFja2luZyBraMO0bmcgKG1vZGVsIHRy4bqjIGBbXWAgaG/hurdjIG3hu5l0IGJveCBraOG7lW5nIGzhu5MpPyBTbyB24bubaSBTRlQsIEYxIHRoYXkgxJHhu5VpIHRo4bq/IG7DoG8gdsOgIGPDsyB2xrDhu6N0IENJOTUga2jDtG5nPwoKIyMgS2hpIGfhurdwIHPhu7EgY+G7kQoKfCBUcmnhu4d1IGNo4bupbmcgfCBOZ3V5w6puIG5ow6JuIHRoxrDhu51uZyBn4bq3cCB8IEPDoWNoIHjhu60gbMO9IHwKfC0tLXwtLS18LS0tfAp8IGBDVURBIG91dCBvZiBtZW1vcnlgIHwgYmF0Y2ggcXXDoSBs4bubbiwgcXXDqm4gZ3JhZGllbnQgY2hlY2twb2ludGluZyB8IGBiYXRjaF9zaXplPTFgLCBgZ3JhZF9hY2N1bT04YCwgYuG6rXQgZ3JhZGllbnQgY2hlY2twb2ludGluZyB8CnwgTG9zcyBgbmFuYC9gaW5mYCB8IHRyw6BuIHPhu5EgZnAxNiB8IGdp4bqjbSBscjsgdGjhu60gYGJhdGNoX3NpemU9MWA7IGtp4buDbSB0cmEgYuG6oW4gxJHDoyBtYXNrIMSRw7puZyB8CnwgRjEgRTIvRTMg4omIIEUxIGNow61uaCB4w6FjIHwgcXXDqm4gbuG6oXAgbOG6oWkgcHJvamVjdG9yIGzDumMgc3V5IGx14bqtbiB8IGzGsHUgdsOgIG7huqFwIGBleHRyYV90cmFpbmFibGVgIChixrDhu5tjIDYpIHwKfCBgcGFyc2VfdmFsaWRfcmF0ZWAgdGjhuqVwIHwgbW9kZWwgbOG6t3AgdsO0IGjhuqFuIC8gYuG7iyBj4bqvdCB8IHTEg25nIGBtYXhfbmV3X3Rva2Vuc2AsIGtp4buDbSB0cmEgdG9rZW4ga+G6v3QgdGjDumMgYDx8aW1fZW5kfD5gIHRyb25nIG5ow6NuIHwKfCBGMSDiiYggMCBtw6AgSlNPTiBo4bujcCBs4buHIHwgc2FpIHRoYW5nIHRv4bqhIMSR4buZIChwaXhlbCB0aGF5IHbDrCAw4oCTMTAwMCkgfCB4ZW0gbOG6oWkgYsaw4bubYyAyIHbDoCBgLS1jb29yZGAgfAp8IE3huqV0IHNlc3Npb24gZ2nhu69hIGNo4burbmcgfCBDb2xhYiBmcmVlIG5n4bqvdCB8IGNow6lwIGFkYXB0ZXIgbMOqbiBEcml2ZSBzYXUgbeG7l2kgbOG6p24gfAo=", "RUBRIC.md": "IyBUaGFuZyDEkWnhu4NtIExhYiBEYXkgMTAgKDEwMCDEkWnhu4NtICsgMTAgYm9udXMpCgo+IELDoGkgY8OhIG5ow6JuLCBu4buZcCBxdWEgKipyZXBvIEdpdEh1YiBjw7RuZyBraGFpKiouIFhlbSBbYFJFQURNRS5tZGBdKFJFQURNRS5tZCkgdsOgIFtgR1VJREUubWRgXShHVUlERS5tZCkuCgojIyBDw6FjaCBjaOG6pW0KCi0gR2nhuqNuZyB2acOqbiBjbG9uZSByZXBvIChjb21taXQgbeG7m2kgbmjhuqV0ICoqdHLGsOG7m2MgaOG6oW4gbuG7mXAqKiksICoqZ2hpIMSRw6ggYHNjcmlwdHMvYCB2w6AgYHRlc3RzL2AgYuG6sW5nIGLhuqNuIGfhu5FjIGPhu6dhIMSR4buBKiosIGNo4bqheSBgc2NyaXB0cy9jaGVja19zdWJtaXNzaW9uLnB5YCwgdsOgIMSR4buNYyBgcmVwb3J0L2V2aWRlbmNlLyouanNvbmAgZG8gYHNjb3JlLnB5YCBzaW5oIHJhLgotIEdp4bqjbmcgdmnDqm4gxJHhu5FpIGNoaeG6v3UgYsOhbyBjw6FvICoqZ2nhu69hIGPDoWMgcmVwbyoqIHbDoCBjw7MgdGjhu4MgZ+G7jWkgKip24bqlbiDEkcOhcCA1IHBow7p0KiogxJHhu4MgeMOhYyBuaOG6rW4gYuG6oW4gaGnhu4N1IMSRaeG7gXUgYuG6oW4gdmnhur90LgoKfCBQaOG6p24gfCDEkGnhu4NtIHwgTuG7mWkgZHVuZyB8CnwtLS18LS0tfC0tLXwKfCBBLiBL4bq/dCBxdeG6oyBjaOG6oXkgKHThu7EgxJHhu5luZykgfCA0MCB8IFhlbSBi4bqjbmcgQSBiw6puIGTGsOG7m2kgfAp8IEIuIFRow60gbmdoaeG7h20gdsOgIGLhurFuZyBjaOG7qW5nIHwgMjAgfCBYZW0gYuG6o25nIEIgYsOqbiBkxrDhu5tpIHwKfCBDLiBCw6FvIGPDoW8gfCAzMCB8IDEwIGPDonUgw5cgMyDEkWnhu4NtLiBN4buXaSBjw6J1IHBo4bqjaSAqKsSRw7puZyBi4bqjbiBjaOG6pXQqKiB2w6AgKipkw7luZyBz4buRIGPhu6dhIGNow61uaCBi4bqhbioqIChraMO0bmcgY2jDqXAgdMOgaSBsaeG7h3UpLiBDw6J1IDMsIDQsIDYsIDggY2jhuqVtIGvhu7kgbmjhuqV0IHwKfCBELiBHaXQgdsOgIHRyw6xuaCBiw6B5IHwgMTAgfCDiiaUgNSBjb21taXQgdGhlbyBixrDhu5tjLCByZXBvIHPhuqFjaCAoa2jDtG5nIGThu68gbGnhu4d1L2tob8OhL3Ry4buNbmcgc+G7kSk6IDUgwrcgYGNoZWNrX3N1Ym1pc3Npb24ucHlgIGtow7RuZyBjw7JuIEZBSUw6IDUgfAoKIyMgQS4gS+G6v3QgcXXhuqMgY2jhuqF5ICg0MCDEkWnhu4NtLCB04buxIMSR4buZbmcpCgp8IFRpw6p1IGNow60gfCDEkGnhu4NtIHwgxJDhuqF0IGtoaSB8CnwtLS18LS0tfC0tLXwKfCBgZTAuanNvbmDigJNgZTMuanNvbmAgaOG7o3AgbOG7hyB8IDEyIHwgxJDhu6cgNCBmaWxlLCBgc3BsaXRgIGzDoCBgdGVzdGAsIGBtaXNzaW5nX3ByZWRpY3Rpb25zYCA9IDAsIGPDuW5nIGBkYXRhc2V0X3ZlcnNpb25gIHbDoCBraOG7m3AgYGRhdGEvc3BsaXRzL21ldGEuanNvbmAgfAp8IGBtZXRhYCBj4bunYSBFMeKAk0UzIHwgNiB8IEPDsyBgdHJhaW5hYmxlX3BhcmFtc2AsIGBwZWFrX3ZyYW1fZ2JgLCBgdHJhaW5fbWludXRlc2AgbMOgIHPhu5EgZMawxqFuZyB8CnwgT3V0cHV0IGjhu6NwIGzhu4cgY+G7p2EgRTEgfCA0IHwgYHBhcnNlX3ZhbGlkX3JhdGVgIGPhu6dhIEUxIOKJpSAwLDk1IHwKfCBFMSB0aOG6rXQgc+G7sSBo4buNYyDEkcaw4bujYyB8IDYgfCBGMSBj4bunYSBFMSBoxqFuIEYxIGPhu6dhIEUwIMOtdCBuaOG6pXQgMCwxMCB8CnwgRTQgfCA2IHwgQ8OzIMOtdCBuaOG6pXQgbeG7mXQgZmlsZSBgZTQqLmpzb25gIGjhu6NwIGzhu4cgfAp8IELhurFuZyBjaOG7qW5nIG5o4bqldCBxdcOhbiB8IDYgfCBgY2hlY2tfc3VibWlzc2lvbi5weWAga2jDtG5nIGLDoW8gRkFJTCDhu58gY8OhYyBt4bulYyBi4bqxbmcgY2jhu6luZyB8CgojIyBCLiBUaMOtIG5naGnhu4dtIHbDoCBi4bqxbmcgY2jhu6luZyAoMjAgxJFp4buDbSkKCnwgVGnDqnUgY2jDrSB8IMSQaeG7g20gfCDEkOG6oXQgxJHhu6cgxJFp4buDbSBraGkgfAp8LS0tfC0tLXwtLS18CnwgQuG6o25nIHThu5VuZyBo4bujcCBFMOKAk0UzIHwgNiB8IMSQ4bunIEYxLCBDSTk1LCBGUC/huqNuaCwgVlJBTSwgdGjhu51pIGdpYW4sIEYxIHThu6tuZyBs4bubcCwga2jhu5twIGPDoWMgZmlsZSBi4bqxbmcgY2jhu6luZyB8Cnwg4bqibmggbWluaCBob+G6oSB8IDQgfCDiiaUgNiDhuqNuaCBk4buxIMSRb8OhbiDEkcO6bmcvc2FpIGPDsyBjaMO6IHRow61jaCAoYm94IGThu7EgxJFvw6FuIHbDoCBib3ggxJHDum5nKSB8CnwgRTQgc+G6oWNoIHwgNiB8IENo4buJIMSR4buVaSAqKm3hu5l0KiogYmnhur9uLCBjaOG6oXkgY+G6oyBoYWkgZ2nDoSB0cuG7iywga+G6v3QgbHXhuq1uIGPDsyBuaMOsbiBDSTk1IHwKfCBOaOG6rXQga8O9IHTDoGkgbmd1ecOqbiB8IDQgfCBWUkFNIHbDoCB0aOG7nWkgZ2lhbiB0cm9uZyBiw6FvIGPDoW8ga2jhu5twIGBtZXRhYCB8CgojIyBCb251cyBFNSAoKzEwLCB04buVbmcga2jDtG5nIHbGsOG7o3QgcXXDoSAxMTApCgpIw6BtIHJld2FyZCDEkcO6bmcgdsOgIGPDsyB0ZXN0OiA0IMK3IGNo4bqheSBHUlBPIGPDsyBsb2c6IDMgwrcgcGjDom4gdMOtY2ggcmV3YXJkIGhhY2tpbmcgdsOgIHNvIHbhu5tpIFNGVCBi4bqxbmcgc+G7kSBrw6htIENJOTU6IDMuCgojIyBUcuG7qyDEkWnhu4NtCgotIFPhu61hIGBzY3JpcHRzL2AgaG/hurdjIGB0ZXN0cy9gIGPhu6dhIMSR4buBOiAqKuKIkjEwKiouCi0gQ2jDqXAgYsOgaSAoc+G7kSBsaeG7h3UgaG/hurdjIHbEg24gYuG6o24gZ2nhu5FuZyBuaGF1IGdp4buvYSBjw6FjIHJlcG8pOiAqKjAgxJFp4buDbSBjaG8gY+G6oyBoYWkgYsOqbioqLgotIENvbW1pdCBk4buvIGxp4buHdSwga2hvw6EgS2FnZ2xlL0hGLCBob+G6t2MgdHLhu41uZyBz4buROiBt4buXaSBt4bulYyB0cuG7qyAyIMSRaeG7g20g4bufIHBo4bqnbiBELgo=", "REPORT.md": "SOG7jSB0w6puOiA8xJFp4buBbiBo4buNIHTDqm4+CmBkYXRhc2V0X3ZlcnNpb25gOiA8xJFp4buBbiwgbOG6pXkgdOG7qyBkYXRhL3NwbGl0cy9tZXRhLmpzb24+CgojIyBC4bqjbmcgdOG7lW5nIGjhu6NwIChz4buRIGPhu6dhIGLhuqFuLCBs4bqleSB04burIGByZXBvcnQvZXZpZGVuY2UvKi5qc29uYCkKCnwgTcOjIHwgU2V0dXAgfCBUaGFtIHPhu5EgaHXhuqVuIGx1eeG7h24gfCBWUkFNIMSR4buJbmggKEdCKSB8IFBow7p0IHwgRjEgfCBDSTk1IHwgRlAv4bqjbmggfCBGMSBwb3Rob2xlIHwgRjEgY3JhY2sgfCBGMSBtYW5ob2xlIHwKfC0tLXwtLS18LS0tfC0tLXwtLS18LS0tfC0tLXwtLS18LS0tfC0tLXwtLS18CnwgRTAgfCB6ZXJvLXNob3QgfCAwIHwgfCB8IHwgfCB8IHwgfCB8CnwgRTEgfCBMb1JBIExMTSB8IHwgfCB8IHwgfCB8IHwgfCB8CnwgRTIgfCArIHByb2plY3RvciB8IHwgfCB8IHwgfCB8IHwgfCB8CnwgRTMgfCArIExvUkEgVmlUIHwgfCB8IHwgfCB8IHwgfCB8IHwKfCBFNCB8IChnaGkgYmnhur9uIHbDoCBnacOhIHRy4buLKSB8IHwgfCB8IHwgfCB8IHwgfCB8CgojIyAxMCBjw6J1IGjhu49pICh0cuG6oyBs4budaSBi4bqxbmcgKipz4buRIGPhu6dhIGLhuqFuKiosIGtow7RuZyBjaMOpcCBs4bqhaSB0w6BpIGxp4buHdSkKCiMjIyBDw6J1IDEg4oCUIEtp4bq/biB0csO6YzogVmlULCBwcm9qZWN0b3IgdsOgIExMTSBj4bunYSBRd2VuMy1WTC00QiBjw7MgYmFvIG5oacOqdSB0aGFtIHPhu5EgbeG7l2kga2jhu5FpLCB2w6AgRTEvRTIvRTMgbeG7l2kgY8OhaSBj4bqtcCBuaOG6rXQga2jhu5FpIG7DoG8/CjzEkWnhu4FuPgoKIyMjIEPDonUgMiDigJQgRTA6IGBwYXJzZV92YWxpZF9yYXRlYCBi4bqxbmcgYmFvIG5oacOqdSwgdsOgIGhhaSBs4buXaSBwaOG7lSBiaeG6v24gbmjhuqV0IGPhu6dhIG1vZGVsIGfhu5FjIGzDoCBnw6wgKGvDqG0gbeG7mXQgdsOtIGThu6Ugb3V0cHV0IHRow7QpPwo8xJFp4buBbj4KCiMjIyBDw6J1IDMg4oCUIFRoYW5nIDDigJMxMDAwOiBu4bq/dSBjaOG6pW0gb3V0cHV0IHRoZW8gcGl4ZWwgKGhv4bq3YyBuZ8aw4bujYyBs4bqhaSkgdGjDrCBGMSByYSBzYW8gdsOgIHbDrCBzYW8/IELhuqFuIMSRw6Mga2nhu4NtIGNo4bupbmcgxJFp4buBdSBuw6B5IHRo4bq/IG7DoG8/CjzEkWnhu4FuPgoKIyMjIEPDonUgNCDigJQgTWFzayBsb3NzOiB2w6wgc2FvIGNo4buJIHTDrW5oIGxvc3MgdHLDqm4gcGjhuqduIHRy4bqjIGzhu51pPyDEkGnhu4F1IGfDrCB44bqjeSByYSBu4bq/dSB0w61uaCB0csOqbiBj4bqjIHByb21wdCB2w6AgdG9rZW4g4bqjbmg/CjzEkWnhu4FuPgoKIyMjIEPDonUgNSDigJQgVsOsIHNhbyBraMO0bmcgdGjhu4MgaHXhuqVuIGx1eeG7h24gxJHhuqd5IMSR4bunIHByb2plY3RvciBu4bq/dSBWaVQgYuG7iyBsxrDhu6NuZyB04butIGhvw6EgNC1iaXQsIHbDoCBi4bqhbiDEkcOjIHjhu60gbMO9IHRo4bq/IG7DoG8/IFbDrCBzYW8gcHJvamVjdG9yIHBo4bqjaSDhu58gZmxvYXQzMj8KPMSRaeG7gW4+CgojIyMgQ8OidSA2IOKAlCBTcGxpdCB0aGVvIHBoacOqbiBxdWF5OiB2w6wgc2FvIGNoaWEgbmfhuqt1IG5oacOqbiBz4bq9IGNobyBGMSBjYW8gZ2nhuqM/IFPhu5Eg4bqjbmggdsOgIHPhu5EgYm94IHThu6tuZyBs4bubcCDhu58gbeG7l2kgc3BsaXQgbMOgIGJhbyBuaGnDqnU/CjzEkWnhu4FuPgoKIyMjIEPDonUgNyDigJQgU28gc8OhbmggRTEvRTIvRTM6IG3hu58gcHJvamVjdG9yIHbDoCBMb1JBIHRyw6puIFZpVCBjw7MgxJHDoW5nIGtow7RuZyAoRjEsIENJOTUsIFZSQU0sIHRo4budaSBnaWFuKT8KPMSRaeG7gW4+CgojIyMgQ8OidSA4IOKAlCBU4bqtcCB0ZXN0IGPDsyAzMTEg4bqjbmg6IGtob+G6o25nIHRpbiBj4bqteSBy4buZbmcgdGjhur8gbsOgbywgdsOgIMSRaeG7gXUgxJHDsyBnaeG7m2kgaOG6oW4ga+G6v3QgbHXhuq1uIHNvIHPDoW5oIGPDoWMgc2V0dXAgcmEgc2FvPwo8xJFp4buBbj4KCiMjIyBDw6J1IDkg4oCUIFBow6JuIHTDrWNoIGzhu5dpIHRoZW8gbOG7m3A6IGzhu5twIG7DoG8ga8OpbSBuaOG6pXQgdsOgIHbDrCBzYW8gKGvDrWNoIHRoxrDhu5tjIGJveCwgc+G7kSBt4bqrdSwgbmfhu68gbmdoxKlhKT8gS8OobSDhuqNuaCBtaW5oIGhv4bqhLgo8xJFp4buBbj4KCiMjIyBDw6J1IDEwIOKAlCBFNDogYuG6oW4gxJHhu5VpIGJp4bq/biBuw6BvLCBr4bq/dCBxdeG6oyB0aOG6vyBuw6BvLCB2w6Aga+G6v3QgbHXhuq1uIGPDsyDEkcOhbmcgdGluIGtow7RuZz8KPMSRaeG7gW4+CgojIyBOaMOsbiBs4bqhaQoKPMSRaeG7gW46IMSRaeG7gXUgYuG6oW4gY2jGsGEgaGnhu4N1LCBs4buXaSBk4buvIGxp4buHdSBob+G6t2MgbcO0IGjDrG5oIGLhuqFuIGfhurdwIG3DoCBjw7RuZyBj4bulIGNo4bqlbSBjaMawYSBi4bqvdCDEkcaw4bujYz4K", "requirements-colab.txt": "IyBEw7luZyB0b3JjaC90b3JjaHZpc2lvbiBjw7Mgc+G6tW4gY+G7p2EgQ29sYWIuIFBoacOqbiBi4bqjbiDEkcOjIGtp4buDbSB0cmEgY8OgaSDEkeG6t3QvaW1wb3J0IG5nw6B5IDIwMjYtMTAtMTEuCnRyYW5zZm9ybWVycz09NS4xOS4wCnBlZnQ9PTAuMjEuMgp0cmw9PTEuOS4yCmFjY2VsZXJhdGU9PTEuMTUuMApiaXRzYW5kYnl0ZXM9PTAuNTAuMgpweXlhbWwKcGlsbG93CmthZ2dsZQo=", "pytest.ini": "W3B5dGVzdF0KcHl0aG9ucGF0aCA9IC4KdGVzdHBhdGhzID0gdGVzdHMK", "scripts/__init__.py": "", "scripts/check_submission.py": "IiIiS2nhu4NtIHRyYSBiw6BpIG7hu5lwIExhYiBEYXkgMTAgdHLGsOG7m2Mga2hpIG7hu5lwLiBLSMOUTkcgc+G7rWEgZmlsZSBuw6B5LiIiIgppbXBvcnQganNvbgppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKUkVRVUlSRUQgPSAoImUwIiwgImUxIiwgImUyIiwgImUzIikKVFJBSU5FRCA9ICgiZTEiLCAiZTIiLCAiZTMiKQpNRVRBX0tFWVMgPSAoInRyYWluYWJsZV9wYXJhbXMiLCAicGVha192cmFtX2diIiwgInRyYWluX21pbnV0ZXMiKQpGT1JCSURERU5fUFJFRklYID0gKCJkYXRhLyIsICJ3b3JrLyIpCkZPUkJJRERFTl9OQU1FUyA9ICgiLmVudiIsKQpGT1JCSURERU5fU1VGRklYID0gKCIuc2FmZXRlbnNvcnMiLCAiLmJpbiIsICIucHQiLCAiLmNrcHQiKQpNSU5fQ09NTUlUUyA9IDUKCgpkZWYgX2dpdChyb290LCAqYXJncyk6CiAgICByZXR1cm4gc3VicHJvY2Vzcy5ydW4oWyJnaXQiLCAiLUMiLCBzdHIocm9vdCksICphcmdzXSwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlKQoKCmRlZiBydW5fY2hlY2tzKHJvb3QpOgogICAgcm9vdCA9IFBhdGgocm9vdCkKICAgIHJlcyA9IFtdCgogICAgZGVmIGFkZChzdCwgbmFtZSwgZGV0YWlsPSIiKToKICAgICAgICByZXMuYXBwZW5kKChzdCwgbmFtZSwgZGV0YWlsKSkKCiAgICBldl9kaXIgPSByb290IC8gInJlcG9ydCIgLyAiZXZpZGVuY2UiCiAgICBldiA9IHt9CiAgICBmb3Iga2V5IGluIFJFUVVJUkVEOgogICAgICAgIHAgPSBldl9kaXIgLyBmIntrZXl9Lmpzb24iCiAgICAgICAgaWYgbm90IHAuZXhpc3RzKCk6CiAgICAgICAgICAgIGFkZCgiRkFJTCIsIGYiZXZpZGVuY2Uge2tleX0iLCBmInRoaeG6v3UgcmVwb3J0L2V2aWRlbmNlL3trZXl9Lmpzb24iKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHRyeToKICAgICAgICAgICAgZCA9IGpzb24ubG9hZHMocC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3IgYXMgZToKICAgICAgICAgICAgYWRkKCJGQUlMIiwgZiJldmlkZW5jZSB7a2V5fSIsIGYiSlNPTiBo4buPbmc6IHtlfSIpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgZXZba2V5XSA9IGQKICAgICAgICBwcm9ibGVtcyA9IFtdCiAgICAgICAgaWYgZC5nZXQoInNjb3JlX3ZlcnNpb24iKSAhPSAxOgogICAgICAgICAgICBwcm9ibGVtcy5hcHBlbmQoInNjb3JlX3ZlcnNpb24gIT0gMSIpCiAgICAgICAgaWYgZC5nZXQoInNwbGl0IikgIT0gInRlc3QiOgogICAgICAgICAgICBwcm9ibGVtcy5hcHBlbmQoInNwbGl0ICE9IHRlc3QiKQogICAgICAgIGlmIGQuZ2V0KCJtaXNzaW5nX3ByZWRpY3Rpb25zIiwgMSkgIT0gMDoKICAgICAgICAgICAgcHJvYmxlbXMuYXBwZW5kKGYibWlzc2luZ19wcmVkaWN0aW9ucz17ZC5nZXQoJ21pc3NpbmdfcHJlZGljdGlvbnMnKX0iKQogICAgICAgIGFkZCgiRkFJTCIgaWYgcHJvYmxlbXMgZWxzZSAiUEFTUyIsIGYiZXZpZGVuY2Uge2tleX0iLCAiOyAiLmpvaW4ocHJvYmxlbXMpKQogICAgZTQgPSBzb3J0ZWQoZXZfZGlyLmdsb2IoImU0Ki5qc29uIikpIGlmIGV2X2Rpci5leGlzdHMoKSBlbHNlIFtdCiAgICBpZiBlNDoKICAgICAgICBmb3IgcCBpbiBlNDoKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgZXZbcC5zdGVtXSA9IGpzb24ubG9hZHMocC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgICAgIGV4Y2VwdCBWYWx1ZUVycm9yOgogICAgICAgICAgICAgICAgYWRkKCJGQUlMIiwgZiJldmlkZW5jZSB7cC5zdGVtfSIsICJKU09OIGjhu49uZyIpCiAgICAgICAgYWRkKCJQQVNTIiwgImV2aWRlbmNlIGU0IiwgZiJ7bGVuKGU0KX0gZmlsZSIpCiAgICBlbHNlOgogICAgICAgIGFkZCgiRkFJTCIsICJldmlkZW5jZSBlNCIsICJj4bqnbiDDrXQgbmjhuqV0IG3hu5l0IGZpbGUgZTQqLmpzb24iKQoKICAgIGZvciBrZXkgaW4gVFJBSU5FRDoKICAgICAgICBpZiBrZXkgbm90IGluIGV2OgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIG1ldGEgPSBldltrZXldLmdldCgibWV0YSIsIHt9KQogICAgICAgIGJhZCA9IFtrIGZvciBrIGluIE1FVEFfS0VZUwogICAgICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShtZXRhLmdldChrKSwgKGludCwgZmxvYXQpKSBvciBpc2luc3RhbmNlKG1ldGEuZ2V0KGspLCBib29sKQogICAgICAgICAgICAgICBvciBtZXRhW2tdIDw9IDBdCiAgICAgICAgYWRkKCJGQUlMIiBpZiBiYWQgZWxzZSAiUEFTUyIsIGYie2tleX0gbWV0YSIsICIsICIuam9pbihiYWQpKQoKICAgIHZlcnMgPSB7ZC5nZXQoImRhdGFzZXRfdmVyc2lvbiIpIGZvciBkIGluIGV2LnZhbHVlcygpfQogICAgYWRkKCJQQVNTIiBpZiBsZW4odmVycykgPD0gMSBlbHNlICJGQUlMIiwgImRhdGFzZXRfdmVyc2lvbiBuaOG6pXQgcXXDoW4iLCBzdHIoc29ydGVkKG1hcChzdHIsIHZlcnMpKSkpCiAgICBtZXRhX3BhdGggPSByb290IC8gImRhdGEiIC8gInNwbGl0cyIgLyAibWV0YS5qc29uIgogICAgaWYgbWV0YV9wYXRoLmV4aXN0cygpOgogICAgICAgIG0gPSBqc29uLmxvYWRzKG1ldGFfcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgb2sgPSB2ZXJzIDw9IHttLmdldCgiZGF0YXNldF92ZXJzaW9uIil9CiAgICAgICAgYWRkKCJQQVNTIiBpZiBvayBlbHNlICJGQUlMIiwgImto4bubcCBkYXRhL3NwbGl0cy9tZXRhLmpzb24iLAogICAgICAgICAgICAiIiBpZiBvayBlbHNlICJi4bqxbmcgY2jhu6luZyBraMO0bmcgc2luaCB04burIHNwbGl0IGNodeG6qW4gKGNo4bqheSBs4bqhaSBwcmVwYXJlX2RhdGEucHkpIikKICAgIGVsc2U6CiAgICAgICAgYWRkKCJXQVJOIiwgImto4bubcCBkYXRhL3NwbGl0cy9tZXRhLmpzb24iLCAia2jDtG5nIGPDsyBkYXRhL3NwbGl0cy9tZXRhLmpzb24gxJHhu4MgxJHhu5FpIGNoaeG6v3UiKQoKICAgIHJwID0gcm9vdCAvICJSRVBPUlQubWQiCiAgICBpZiBub3QgcnAuZXhpc3RzKCk6CiAgICAgICAgYWRkKCJGQUlMIiwgIlJFUE9SVC5tZCIsICJ0aGnhur91IFJFUE9SVC5tZCIpCiAgICBlbHNlOgogICAgICAgIGFkZCgiRkFJTCIgaWYgIjzEkWnhu4FuIiBpbiBycC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikgZWxzZSAiUEFTUyIsCiAgICAgICAgICAgICJSRVBPUlQubWQgxJHDoyDEkWnhu4FuIiwgImPDsm4gY2h14buXaSAnPMSRaeG7gW4nIikKCiAgICBpZiAocm9vdCAvICIuZ2l0IikuZXhpc3RzKCk6CiAgICAgICAgbiA9IF9naXQocm9vdCwgInJldi1saXN0IiwgIi0tY291bnQiLCAiSEVBRCIpCiAgICAgICAgY291bnQgPSBpbnQobi5zdGRvdXQuc3RyaXAoKSBvciAwKSBpZiBuLnJldHVybmNvZGUgPT0gMCBlbHNlIDAKICAgICAgICBhZGQoIlBBU1MiIGlmIGNvdW50ID49IE1JTl9DT01NSVRTIGVsc2UgIkZBSUwiLCAiZ2l0IGNvbW1pdHMiLCBmIntjb3VudH0ve01JTl9DT01NSVRTfSIpCiAgICAgICAgZmlsZXMgPSBfZ2l0KHJvb3QsICJscy1maWxlcyIpLnN0ZG91dC5zcGxpdGxpbmVzKCkKICAgICAgICBiYWQgPSBbZiBmb3IgZiBpbiBmaWxlcyBpZiBmLnN0YXJ0c3dpdGgoRk9SQklEREVOX1BSRUZJWCkKICAgICAgICAgICAgICAgb3IgZi5zcGxpdCgiLyIpWy0xXSBpbiBGT1JCSURERU5fTkFNRVMgb3IgZi5lbmRzd2l0aChGT1JCSURERU5fU1VGRklYKV0KICAgICAgICBhZGQoIkZBSUwiIGlmIGJhZCBlbHNlICJQQVNTIiwgImdpdCBmb3JiaWRkZW4gZmlsZXMiLCAiLCAiLmpvaW4oYmFkWzo1XSkpCiAgICBlbHNlOgogICAgICAgIGFkZCgiV0FSTiIsICJnaXQiLCAia2jDtG5nIHBo4bqjaSBnaXQgcmVwbyIpCiAgICByZXR1cm4gcmVzCgoKZGVmIG1haW4oKToKICAgIHJvb3QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXQogICAgcmVzID0gcnVuX2NoZWNrcyhyb290KQogICAgZm9yIHN0LCBuYW1lLCBkZXRhaWwgaW4gcmVzOgogICAgICAgIHByaW50KGYiW3tzdH1dIHtuYW1lfSIgKyAoZiIg4oCUIHtkZXRhaWx9IiBpZiBkZXRhaWwgZWxzZSAiIikpCiAgICBmYWlscyA9IHN1bSgxIGZvciByIGluIHJlcyBpZiByWzBdID09ICJGQUlMIikKICAgIHByaW50KGYiXG57ZmFpbHN9IEZBSUwsIHtzdW0oMSBmb3IgciBpbiByZXMgaWYgclswXSA9PSAnV0FSTicpfSBXQVJOIikKICAgIHN5cy5leGl0KDEgaWYgZmFpbHMgZWxzZSAwKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK", "scripts/labcommon.py": "IiIiSOG6sW5nIHPhu5EgdsOgIGjDoG0gSS9PIGTDuW5nIGNodW5nIGNobyBjw6FjIGPDtG5nIGPhu6UgY+G7p2EgbGFiIERheSAxMC4iIiIKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpDTEFTU0VTID0gKCJwb3Rob2xlIiwgImNyYWNrIiwgIm1hbmhvbGUiKQoKCmRlZiByZWFkX2pzb25sKHBhdGgpOgogICAgcm93cyA9IFtdCiAgICB3aXRoIG9wZW4ocGF0aCwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICBmb3IgbGluZSBpbiBmOgogICAgICAgICAgICBsaW5lID0gbGluZS5zdHJpcCgpCiAgICAgICAgICAgIGlmIGxpbmU6CiAgICAgICAgICAgICAgICByb3dzLmFwcGVuZChqc29uLmxvYWRzKGxpbmUpKQogICAgcmV0dXJuIHJvd3MKCgpkZWYgd3JpdGVfanNvbmwocGF0aCwgcm93cyk6CiAgICBwYXRoID0gUGF0aChwYXRoKQogICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgd2l0aCBvcGVuKHBhdGgsICJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICBmb3IgciBpbiByb3dzOgogICAgICAgICAgICBmLndyaXRlKGpzb24uZHVtcHMociwgZW5zdXJlX2FzY2lpPUZhbHNlKSArICJcbiIpCg==", "scripts/make_notebook.py": "IiIiU2luaCBsYWJfZGF5MTAuaXB5bmIgKGtodW5nKS4gQ2jhuqF5OiBweXRob24gc2NyaXB0cy9tYWtlX25vdGVib29rLnB5IiIiCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKCmRlZiBtZChzKToKICAgIHJldHVybiB7ImNlbGxfdHlwZSI6ICJtYXJrZG93biIsICJtZXRhZGF0YSI6IHt9LCAic291cmNlIjogcy5zdHJpcCgiXG4iKS5zcGxpdGxpbmVzKFRydWUpfQoKCmRlZiBjb2RlKHMpOgogICAgcmV0dXJuIHsiY2VsbF90eXBlIjogImNvZGUiLCAibWV0YWRhdGEiOiB7fSwgImV4ZWN1dGlvbl9jb3VudCI6IE5vbmUsICJvdXRwdXRzIjogW10sCiAgICAgICAgICAgICJzb3VyY2UiOiBzLnN0cmlwKCJcbiIpLnNwbGl0bGluZXMoVHJ1ZSl9CgoKQ0VMTFMgPSBbCiAgICBtZCgiIyBMYWIgRGF5IDEwOiBGaW5ldHVuZSBRd2VuMy1WTC00QiAoTG9SQSkgxJHhu4MgxJHhu4tuaCB24buLIGjGsCBo4bqhaSBt4bq3dCDEkcaw4budbmdcbiIKICAgICAgICLEkOG7jWMgYFJFQURNRS5tZGAgdsOgIGBHVUlERS5tZGAgdHLGsOG7m2MuIE5vdGVib29rIG7DoHkgY2jhu4kgbMOgICoqa2h1bmcqKjogYuG6oW4gdOG7sSB2aeG6v3QgbcOjIHRoZW8gcHNldWRvIGNvZGUgdHJvbmcgR1VJREUuIiksCiAgICBtZCgiIyMgMC4gTcO0aSB0csaw4budbmcgKENvbGFiIFQ0KSIpLAogICAgY29kZSgiIyBSdW50aW1lIOKGkiBDaGFuZ2UgcnVudGltZSB0eXBlIOKGkiBUNCBHUFVcbiIKICAgICAgICAgIiFudmlkaWEtc21pIC1MXG4iCiAgICAgICAgICIhcGlwIC1xIGluc3RhbGwgLXIgcmVxdWlyZW1lbnRzLWNvbGFiLnR4dCIpLAogICAgY29kZSgiIyBLYWdnbGU6IGTDuW5nIENvbGFiIFNlY3JldHMgKEtBR0dMRV9VU0VSTkFNRSwgS0FHR0xFX0tFWSkuIEtIw5RORyB2aeG6v3Qga2hvw6EgdsOgbyBub3RlYm9vay5cbiIKICAgICAgICAgImltcG9ydCBvc1xuIgogICAgICAgICAiZnJvbSBnb29nbGUuY29sYWIgaW1wb3J0IHVzZXJkYXRhXG4iCiAgICAgICAgICJvcy5lbnZpcm9uWydLQUdHTEVfVVNFUk5BTUUnXSA9IHVzZXJkYXRhLmdldCgnS0FHR0xFX1VTRVJOQU1FJylcbiIKICAgICAgICAgIm9zLmVudmlyb25bJ0tBR0dMRV9LRVknXSA9IHVzZXJkYXRhLmdldCgnS0FHR0xFX0tFWScpIiksCiAgICBtZCgiIyMgMS4gROG7ryBsaeG7h3U6IHThuqNpIHbDoCBjaGlhIHNwbGl0IiksCiAgICBjb2RlKCIhcHl0aG9uIHNjcmlwdHMvcHJlcGFyZV9kYXRhLnB5IC0ta2FnZ2xlIiksCiAgICBjb2RlKCIjIFlPVVIgQ09ERSBIRVJFOiDEkeG7jWMgZGF0YS9zcGxpdHMvdHJhaW4uanNvbmwsIHhlbSB2w6BpIG3huqt1LCB24bq9IGJveCBsw6puIOG6o25oIiksCiAgICBtZCgiIyMgMi4gTmjDo246IFlPTE8g4oaSIEpTT04gdGhhbmcgMOKAkzEwMDAiKSwKICAgIGNvZGUoIiMgWU9VUiBDT0RFIEhFUkU6IHZp4bq/dCB5b2xvX3RvX2JveCgpIHbDoCBvYmplY3RzX3RvX2Fuc3dlcigpIChHVUlERSBixrDhu5tjIDIpIiksCiAgICBtZCgiIyMgMy4gRTA6IHplcm8tc2hvdCIpLAogICAgY29kZSgiIyBZT1VSIENPREUgSEVSRTogbuG6oXAgbW9kZWwgNC1iaXQsIHN1eSBsdeG6rW4gdHLDqm4gdGVzdCDihpIgd29yay9lMC5qc29ubCAoR1VJREUgYsaw4bubYyAzKSIpLAogICAgY29kZSgiIXB5dGhvbiBzY3JpcHRzL3Njb3JlLnB5IC0tcHJlZCB3b3JrL2UwLmpzb25sIC0tbmFtZSBFMCAtLW91dCByZXBvcnQvZXZpZGVuY2UvZTAuanNvbiIpLAogICAgbWQoIiMjIDQuIEThu7FuZyBk4buvIGxp4buHdSBodeG6pW4gbHV54buHbjogY2hhdCB0ZW1wbGF0ZSB2w6AgbWFzayBsb3NzIiksCiAgICBjb2RlKCIjIFlPVVIgQ09ERSBIRVJFOiBlbmNvZGVfZXhhbXBsZSgpIHbDoCBjb2xsYXRlKCkgKEdVSURFIGLGsOG7m2MgNCkiKSwKICAgIG1kKCIjIyA1LiBTZXR1cCBMb1JBIC8gxJHDs25nIGLEg25nIGNobyBFMSwgRTIsIEUzIiksCiAgICBjb2RlKCIjIFlPVVIgQ09ERSBIRVJFOiBhcHBseV9zZXR1cChtb2RlbCwgc2V0dXApIChHVUlERSBixrDhu5tjIDUpIiksCiAgICBtZCgiIyMgNi4gVsOybmcgaHXhuqVuIGx1eeG7h24iKSwKICAgIGNvZGUoIiMgWU9VUiBDT0RFIEhFUkU6IHbDsm5nIHRyYWluLCBsxrB1IGFkYXB0ZXIgKyBydW4uanNvbiAoR1VJREUgYsaw4bubYyA2KSIpLAogICAgbWQoIiMjIDcuIFN1eSBsdeG6rW4gdsOgIGNo4bqlbSBFMeKAk0UzIiksCiAgICBjb2RlKCIjIFlPVVIgQ09ERSBIRVJFOiBzdXkgbHXhuq1uIHRlc3Qg4oaSIHdvcmsvZU4uanNvbmwgcuG7k2kgY2jhuqF5IHNjcmlwdHMvc2NvcmUucHkgduG7m2kgLS1tZXRhIChHVUlERSBixrDhu5tjIDcpIiksCiAgICBtZCgiIyMgOC4gRTQgKGFibGF0aW9uIHThu7EgY2jhu41uKSB2w6AgRTUgKHR14buzIGNo4buNbiwgR1JQTykiKSwKICAgIGNvZGUoIiMgWU9VUiBDT0RFIEhFUkU6IEdVSURFIGLGsOG7m2MgOCB2w6AgOSIpLApdCgoKZGVmIG1haW4oKToKICAgIG5iID0geyJjZWxscyI6IENFTExTLAogICAgICAgICAgIm1ldGFkYXRhIjogeyJhY2NlbGVyYXRvciI6ICJHUFUiLCAiY29sYWIiOiB7InByb3ZlbmFuY2UiOiBbXX0sCiAgICAgICAgICAgICAgICAgICAgICAgImtlcm5lbHNwZWMiOiB7Im5hbWUiOiAicHl0aG9uMyIsICJkaXNwbGF5X25hbWUiOiAiUHl0aG9uIDMifX0sCiAgICAgICAgICAibmJmb3JtYXQiOiA0LCAibmJmb3JtYXRfbWlub3IiOiA1fQogICAgcGF0aCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdIC8gImxhYl9kYXkxMC5pcHluYiIKICAgIHBhdGgud3JpdGVfdGV4dChqc29uLmR1bXBzKG5iLCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0xKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHByaW50KCJ3cm90ZSIsIHBhdGgpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=", "scripts/prepare_data.py": "IiIiVOG6o2kgZGF0YXNldCByb2FkIGRhbWFnZSwga2nhu4NtIHRyYSwgY2hpYSBzcGxpdCB0aGVvIHBoacOqbiBxdWF5IChj4buRIMSR4buLbmgpLiBLSMOUTkcgc+G7rWEgZmlsZSBuw6B5LiIiIgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IHNodXRpbAppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmltcG9ydCB1cmxsaWIucmVxdWVzdAppbXBvcnQgemlwZmlsZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0pKQoKZnJvbSBQSUwgaW1wb3J0IEltYWdlICAjIG5vcWE6IEU0MDIKCmZyb20gc2NyaXB0cy5sYWJjb21tb24gaW1wb3J0IENMQVNTRVMsIHdyaXRlX2pzb25sICAjIG5vcWE6IEU0MDIKZnJvbSBzY3JpcHRzLnNwbGl0dGluZyBpbXBvcnQgKCAgIyBub3FhOiBFNDAyCiAgICBNSU5fQ0xBU1NfQk9YRVMsIE1JTl9GUkFDLCBTRUVELCBTUExJVFMsIGFzc2lnbl9zcGxpdHMsIGRhdGFzZXRfdmVyc2lvbiwgc2Vzc2lvbl9rZXksCikKCktBR0dMRV9TTFVHID0gImxvcmVuem9hcmNpb25pL3JvYWQtZGFtYWdlLWRhdGFzZXQtcG90aG9sZXMtY3JhY2tzLWFuZC1tYW5ob2xlcyIKCgpkZWYgX3BhcnNlX2xhYmVsX2ZpbGUocGF0aCwgZmlsZW5hbWUpOgogICAgb2JqcyA9IFtdCiAgICBmb3IgbG4sIGxpbmUgaW4gZW51bWVyYXRlKHBhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpLnNwbGl0bGluZXMoKSwgMSk6CiAgICAgICAgaWYgbm90IGxpbmUuc3RyaXAoKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBwYXJ0cyA9IGxpbmUuc3BsaXQoKQogICAgICAgIHRyeToKICAgICAgICAgICAgaWYgbGVuKHBhcnRzKSA8IDU6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yCiAgICAgICAgICAgIGNpZCA9IGludChwYXJ0c1swXSkKICAgICAgICAgICAgeGMsIHljLCB3LCBoID0gKGZsb2F0KHYpIGZvciB2IGluIHBhcnRzWzE6NV0pCiAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3I6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ7ZmlsZW5hbWV9OiBtYWxmb3JtZWQgbGFiZWwgbGluZSB7bG59OiB7bGluZSFyfSIpIGZyb20gTm9uZQogICAgICAgIGlmIG5vdCAwIDw9IGNpZCA8IGxlbihDTEFTU0VTKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIntmaWxlbmFtZX06IGNsYXNzIGlkIHtjaWR9IG91dHNpZGUgMC4ue2xlbihDTEFTU0VTKSAtIDF9IChsaW5lIHtsbn0pIikKICAgICAgICBpZiBub3QgYWxsKDAuMCA8PSB2IDw9IDEuMCBmb3IgdiBpbiAoeGMsIHljLCB3LCBoKSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ7ZmlsZW5hbWV9OiB2YWx1ZSBvdXQgb2YgcmFuZ2UgWzAsMV0gb24gbGluZSB7bG59OiB7bGluZSFyfSIpCiAgICAgICAgb2Jqcy5hcHBlbmQoeyJsYWJlbCI6IENMQVNTRVNbY2lkXSwgInlvbG8iOiBbeGMsIHljLCB3LCBoXX0pCiAgICByZXR1cm4gb2JqcwoKCmRlZiBsb2FkX3JlY29yZHMoc3JjKToKICAgIHNyYyA9IFBhdGgoc3JjKQogICAgaW1hZ2VzID0gc29ydGVkKHAgZm9yIHAgaW4gKHNyYyAvICJpbWFnZXMiKS5pdGVyZGlyKCkgaWYgcC5zdWZmaXgubG93ZXIoKSA9PSAiLmpwZyIpCiAgICBpZiBub3QgaW1hZ2VzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJubyAuanBnIHVuZGVyIHtzcmMgLyAnaW1hZ2VzJ30iKQogICAgcmVjb3JkcyA9IFtdCiAgICBmb3IgcCBpbiBpbWFnZXM6CiAgICAgICAgbGFiZWwgPSBzcmMgLyAibGFiZWxzLVlPTE8iIC8gZiJ7cC5zdGVtfS50eHQiCiAgICAgICAgaWYgbm90IGxhYmVsLmV4aXN0cygpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYie3AubmFtZX06IG1pc3NpbmcgbGFiZWwgZmlsZSB7bGFiZWwubmFtZX0iKQogICAgICAgIHdpdGggSW1hZ2Uub3BlbihwKSBhcyBpbToKICAgICAgICAgICAgd2lkdGgsIGhlaWdodCA9IGltLnNpemUKICAgICAgICByZWNvcmRzLmFwcGVuZCh7CiAgICAgICAgICAgICJpbWFnZV9pZCI6IHAuc3RlbSwgImZpbGVuYW1lIjogcC5uYW1lLCAid2lkdGgiOiB3aWR0aCwgImhlaWdodCI6IGhlaWdodCwKICAgICAgICAgICAgInNlc3Npb24iOiBzZXNzaW9uX2tleShwLm5hbWUpLCAib2JqZWN0cyI6IF9wYXJzZV9sYWJlbF9maWxlKGxhYmVsLCBwLm5hbWUpLAogICAgICAgIH0pCiAgICByZXR1cm4gcmVjb3JkcwoKCmRlZiBfZ3JvdXBfc3RhdHMocmVjb3Jkcyk6CiAgICBncm91cHMgPSB7fQogICAgZm9yIHIgaW4gcmVjb3JkczoKICAgICAgICBnID0gZ3JvdXBzLnNldGRlZmF1bHQoclsic2Vzc2lvbiJdLCB7ImltYWdlcyI6IDAsICJib3hlcyI6IHtjOiAwIGZvciBjIGluIENMQVNTRVN9fSkKICAgICAgICBnWyJpbWFnZXMiXSArPSAxCiAgICAgICAgZm9yIG8gaW4gclsib2JqZWN0cyJdOgogICAgICAgICAgICBnWyJib3hlcyJdW29bImxhYmVsIl1dICs9IDEKICAgIHJldHVybiBncm91cHMKCgpkZWYgcHJlcGFyZShzcmMsIG91dCwgc2VlZD1TRUVELCBtaW5fZnJhYz1NSU5fRlJBQywgbWluX2NsYXNzX2JveGVzPU1JTl9DTEFTU19CT1hFUyk6CiAgICBzcmMsIG91dCA9IFBhdGgoc3JjKSwgUGF0aChvdXQpCiAgICByZWNvcmRzID0gbG9hZF9yZWNvcmRzKHNyYykKICAgIGdyb3VwcyA9IF9ncm91cF9zdGF0cyhyZWNvcmRzKQogICAgYXNzaWduLCBzYWx0ID0gYXNzaWduX3NwbGl0cyhncm91cHMsIHNlZWQ9c2VlZCwgbWluX2ZyYWM9bWluX2ZyYWMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1pbl9jbGFzc19ib3hlcz1taW5fY2xhc3NfYm94ZXMpCiAgICAob3V0IC8gImltYWdlcyIpLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHBlcl9zcGxpdCA9IHtzOiBbXSBmb3IgcyBpbiBTUExJVFN9CiAgICBjb3VudHMgPSB7czogeyJpbWFnZXMiOiAwLCAiYm94ZXMiOiB7YzogMCBmb3IgYyBpbiBDTEFTU0VTfX0gZm9yIHMgaW4gU1BMSVRTfQogICAgZm9yIHIgaW4gcmVjb3JkczoKICAgICAgICBzID0gYXNzaWduW3JbInNlc3Npb24iXV0KICAgICAgICBzaHV0aWwuY29weTIoc3JjIC8gImltYWdlcyIgLyByWyJmaWxlbmFtZSJdLCBvdXQgLyAiaW1hZ2VzIiAvIHJbImZpbGVuYW1lIl0pCiAgICAgICAgcGVyX3NwbGl0W3NdLmFwcGVuZCh7CiAgICAgICAgICAgICJpbWFnZV9pZCI6IHJbImltYWdlX2lkIl0sICJpbWFnZV9wYXRoIjogZiJpbWFnZXMve3JbJ2ZpbGVuYW1lJ119IiwKICAgICAgICAgICAgIndpZHRoIjogclsid2lkdGgiXSwgImhlaWdodCI6IHJbImhlaWdodCJdLAogICAgICAgICAgICAic2Vzc2lvbiI6IHJbInNlc3Npb24iXSwgIm9iamVjdHMiOiByWyJvYmplY3RzIl0sCiAgICAgICAgfSkKICAgICAgICBjb3VudHNbc11bImltYWdlcyJdICs9IDEKICAgICAgICBmb3IgbyBpbiByWyJvYmplY3RzIl06CiAgICAgICAgICAgIGNvdW50c1tzXVsiYm94ZXMiXVtvWyJsYWJlbCJdXSArPSAxCiAgICBmb3IgcyBpbiBTUExJVFM6CiAgICAgICAgd3JpdGVfanNvbmwob3V0IC8gInNwbGl0cyIgLyBmIntzfS5qc29ubCIsIHBlcl9zcGxpdFtzXSkKICAgIG1ldGEgPSB7CiAgICAgICAgImRhdGFzZXRfdmVyc2lvbiI6IGRhdGFzZXRfdmVyc2lvbihbKHJbImltYWdlX2lkIl0sIGFzc2lnbltyWyJzZXNzaW9uIl1dKSBmb3IgciBpbiByZWNvcmRzXSksCiAgICAgICAgInNlZWQiOiBzZWVkLCAic2FsdCI6IHNhbHQsICJjbGFzc2VzIjogbGlzdChDTEFTU0VTKSwgImNvdW50cyI6IGNvdW50cywgImdyb3VwcyI6IGFzc2lnbiwKICAgIH0KICAgIChvdXQgLyAic3BsaXRzIiAvICJtZXRhLmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgIGpzb24uZHVtcHMobWV0YSwgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MiksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICByZXR1cm4gbWV0YQoKCmRlZiBfZmluZF9yb290KGJhc2UpOgogICAgZm9yIHAgaW4gW1BhdGgoYmFzZSldICsgc29ydGVkKFBhdGgoYmFzZSkucmdsb2IoIioiKSk6CiAgICAgICAgaWYgcC5pc19kaXIoKSBhbmQgKHAgLyAiaW1hZ2VzIikuaXNfZGlyKCkgYW5kIChwIC8gImxhYmVscy1ZT0xPIikuaXNfZGlyKCk6CiAgICAgICAgICAgIHJldHVybiBwCiAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmImNhbm5vdCBmaW5kIGltYWdlcy8gYW5kIGxhYmVscy1ZT0xPLyB1bmRlciB7YmFzZX0iKQoKCmRlZiBfZmV0Y2goYXJncywgd29yayk6CiAgICB3b3JrLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGlmIGFyZ3Muc291cmNlOgogICAgICAgIHJldHVybiBfZmluZF9yb290KGFyZ3Muc291cmNlKQogICAgaWYgYXJncy56aXA6CiAgICAgICAgeiA9IGFyZ3MuemlwCiAgICAgICAgaWYgei5zdGFydHN3aXRoKCgiaHR0cDovLyIsICJodHRwczovLyIpKToKICAgICAgICAgICAgZGVzdCA9IHdvcmsgLyAiZGF0YXNldC56aXAiCiAgICAgICAgICAgIHVybGxpYi5yZXF1ZXN0LnVybHJldHJpZXZlKHosIGRlc3QpCiAgICAgICAgICAgIHogPSBzdHIoZGVzdCkKICAgICAgICB3aXRoIHppcGZpbGUuWmlwRmlsZSh6KSBhcyB6ZjoKICAgICAgICAgICAgemYuZXh0cmFjdGFsbCh3b3JrIC8gInJhdyIpCiAgICAgICAgcmV0dXJuIF9maW5kX3Jvb3Qod29yayAvICJyYXciKQogICAgc3VicHJvY2Vzcy5ydW4oWyJrYWdnbGUiLCAiZGF0YXNldHMiLCAiZG93bmxvYWQiLCAiLWQiLCBLQUdHTEVfU0xVRywgIi1wIiwKICAgICAgICAgICAgICAgICAgICBzdHIod29yayAvICJyYXciKSwgIi0tdW56aXAiXSwgY2hlY2s9VHJ1ZSkKICAgIHJldHVybiBfZmluZF9yb290KHdvcmsgLyAicmF3IikKCgpkZWYgbWFpbihhcmd2PU5vbmUpOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgZyA9IGFwLmFkZF9tdXR1YWxseV9leGNsdXNpdmVfZ3JvdXAocmVxdWlyZWQ9VHJ1ZSkKICAgIGcuYWRkX2FyZ3VtZW50KCItLXNvdXJjZSIsIGhlbHA9InRoxrAgbeG7pWMgxJHDoyBnaeG6o2kgbsOpbiAoY8OzIGltYWdlcy8gdsOgIGxhYmVscy1ZT0xPLykiKQogICAgZy5hZGRfYXJndW1lbnQoIi0temlwIiwgaGVscD0ixJHGsOG7nW5nIGThuqtuIGhv4bq3YyBVUkwgZmlsZSB6aXAgY+G7p2EgZGF0YXNldCIpCiAgICBnLmFkZF9hcmd1bWVudCgiLS1rYWdnbGUiLCBhY3Rpb249InN0b3JlX3RydWUiLAogICAgICAgICAgICAgICAgICAgaGVscD0idOG6o2kgYuG6sW5nIEthZ2dsZSBDTEkgKGPhuqduIEtBR0dMRV9VU0VSTkFNRS9LQUdHTEVfS0VZKSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgZGVmYXVsdD0iZGF0YSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0td29yayIsIGRlZmF1bHQ9IndvcmsiKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoYXJndikKICAgIHJvb3QgPSBfZmV0Y2goYXJncywgUGF0aChhcmdzLndvcmspKQogICAgbWV0YSA9IHByZXBhcmUocm9vdCwgYXJncy5vdXQpCiAgICBwcmludChmImRhdGFzZXRfdmVyc2lvbiA6IHttZXRhWydkYXRhc2V0X3ZlcnNpb24nXX0iKQogICAgZm9yIHMgaW4gU1BMSVRTOgogICAgICAgIGMgPSBtZXRhWyJjb3VudHMiXVtzXQogICAgICAgIHByaW50KGYiICB7czo1c30ge2NbJ2ltYWdlcyddOjVkfSDhuqNuaCAgYm94ZXM9e2NbJ2JveGVzJ119IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==", "scripts/score.py": "IiIiVHLhu41uZyB0w6BpIGNo4bqlbSBsYWIgRGF5IDEwOiBwYXJzZSBvdXRwdXQsIEYxQElvVTAuNSwgQ0kgYm9vdHN0cmFwLiBLSMOUTkcgc+G7rWEgZmlsZSBuw6B5LiIiIgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IHJhbmRvbQppbXBvcnQgcmUKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0pKQoKZnJvbSBzY3JpcHRzLmxhYmNvbW1vbiBpbXBvcnQgQ0xBU1NFUywgcmVhZF9qc29ubCAgIyBub3FhOiBFNDAyCgpTQ09SRV9WRVJTSU9OID0gMQpJT1VfVEhSID0gMC41Cl9GRU5DRSA9IHJlLmNvbXBpbGUociJgYGAoPzpqc29uKT9ccyooLio/KWBgYCIsIHJlLlMpCgoKZGVmIF9pc19udW0odik6CiAgICByZXR1cm4gaXNpbnN0YW5jZSh2LCAoaW50LCBmbG9hdCkpIGFuZCBub3QgaXNpbnN0YW5jZSh2LCBib29sKSBhbmQgbWF0aC5pc2Zpbml0ZSh2KQoKCmRlZiBwYXJzZV9vdXRwdXQocmF3KToKICAgICIiIlRy4bqjIHbhu4EgbGlzdFt7ImxhYmVsIiwiYmJveCJ9XSBo4bujcCBs4buHLCBbXSBu4bq/dSAnW10nLCBob+G6t2MgTm9uZSBu4bq/dSBo4buPbmcuIiIiCiAgICB0ZXh0ID0gKHJhdyBvciAiIikuc3RyaXAoKQogICAgbSA9IF9GRU5DRS5zZWFyY2godGV4dCkKICAgIGlmIG06CiAgICAgICAgdGV4dCA9IG0uZ3JvdXAoMSkuc3RyaXAoKQogICAgdHJ5OgogICAgICAgIG9iaiA9IGpzb24ubG9hZHModGV4dCkKICAgIGV4Y2VwdCAoVmFsdWVFcnJvciwgVHlwZUVycm9yKToKICAgICAgICBpLCBqID0gdGV4dC5maW5kKCJbIiksIHRleHQucmZpbmQoIl0iKQogICAgICAgIGlmIGkgPCAwIG9yIGogPD0gaToKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICB0cnk6CiAgICAgICAgICAgIG9iaiA9IGpzb24ubG9hZHModGV4dFtpOmogKyAxXSkKICAgICAgICBleGNlcHQgKFZhbHVlRXJyb3IsIFR5cGVFcnJvcik6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICBpZiBub3QgaXNpbnN0YW5jZShvYmosIGxpc3QpOgogICAgICAgIHJldHVybiBOb25lCiAgICBvdXQgPSBbXQogICAgZm9yIG8gaW4gb2JqOgogICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKG8sIGRpY3QpOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIGxhYiwgYmIgPSBvLmdldCgibGFiZWwiKSwgby5nZXQoImJib3hfMmQiKQogICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKGxhYiwgc3RyKSBvciBub3QgaXNpbnN0YW5jZShiYiwgbGlzdCkgb3IgbGVuKGJiKSAhPSA0OgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIGlmIG5vdCBhbGwoX2lzX251bSh2KSBmb3IgdiBpbiBiYik6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgeDEsIHkxLCB4MiwgeTIgPSAoZmxvYXQodikgZm9yIHYgaW4gYmIpCiAgICAgICAgaWYgeDIgPD0geDEgb3IgeTIgPD0geTE6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgb3V0LmFwcGVuZCh7ImxhYmVsIjogbGFiLCAiYmJveCI6ICh4MSwgeTEsIHgyLCB5Mil9KQogICAgcmV0dXJuIG91dAoKCmRlZiBpb3VfeHl4eShhLCBiKToKICAgIGl3ID0gbWF4KDAuMCwgbWluKGFbMl0sIGJbMl0pIC0gbWF4KGFbMF0sIGJbMF0pKQogICAgaWggPSBtYXgoMC4wLCBtaW4oYVszXSwgYlszXSkgLSBtYXgoYVsxXSwgYlsxXSkpCiAgICBpbnRlciA9IGl3ICogaWgKICAgIHVuaW9uID0gKGFbMl0gLSBhWzBdKSAqIChhWzNdIC0gYVsxXSkgKyAoYlsyXSAtIGJbMF0pICogKGJbM10gLSBiWzFdKSAtIGludGVyCiAgICByZXR1cm4gaW50ZXIgLyB1bmlvbiBpZiB1bmlvbiA+IDAgZWxzZSAwLjAKCgpkZWYgX2d0X2JveGVzKGVudHJ5KToKICAgIG91dCA9IFtdCiAgICBmb3IgbyBpbiBlbnRyeVsib2JqZWN0cyJdOgogICAgICAgIHhjLCB5YywgdywgaCA9IG9bInlvbG8iXQogICAgICAgIG91dC5hcHBlbmQoeyJsYWJlbCI6IG9bImxhYmVsIl0sICJiYm94IjogKAogICAgICAgICAgICAoeGMgLSB3IC8gMikgKiAxMDAwLCAoeWMgLSBoIC8gMikgKiAxMDAwLCAoeGMgKyB3IC8gMikgKiAxMDAwLCAoeWMgKyBoIC8gMikgKiAxMDAwKX0pCiAgICByZXR1cm4gb3V0CgoKZGVmIF90b19ub3JtMTAwMChwcmVkLCBjb29yZCwgd2lkdGgsIGhlaWdodCk6CiAgICBpZiBjb29yZCA9PSAibm9ybTEwMDAiOgogICAgICAgIHJldHVybiBwcmVkCiAgICByZXR1cm4gW3sibGFiZWwiOiBwWyJsYWJlbCJdLCAiYmJveCI6ICgKICAgICAgICBwWyJiYm94Il1bMF0gLyB3aWR0aCAqIDEwMDAsIHBbImJib3giXVsxXSAvIGhlaWdodCAqIDEwMDAsCiAgICAgICAgcFsiYmJveCJdWzJdIC8gd2lkdGggKiAxMDAwLCBwWyJiYm94Il1bM10gLyBoZWlnaHQgKiAxMDAwKX0gZm9yIHAgaW4gcHJlZF0KCgpkZWYgX21hdGNoKGd0LCBwcmVkKToKICAgIHBlciA9IHtjOiBbMCwgMCwgMF0gZm9yIGMgaW4gQ0xBU1NFU30KICAgIHBlclsib3RoZXIiXSA9IFswLCAwLCAwXQogICAgaW91cyA9IFtdCiAgICBmb3IgbGFiIGluIHNldChDTEFTU0VTKSB8IHtwWyJsYWJlbCJdIGZvciBwIGluIHByZWR9OgogICAgICAgIGcgPSBbeFsiYmJveCJdIGZvciB4IGluIGd0IGlmIHhbImxhYmVsIl0gPT0gbGFiXQogICAgICAgIHAgPSBbeFsiYmJveCJdIGZvciB4IGluIHByZWQgaWYgeFsibGFiZWwiXSA9PSBsYWJdCiAgICAgICAgcGFpcnMgPSBzb3J0ZWQoKChpb3VfeHl4eShhLCBiKSwgaSwgaikgZm9yIGksIGEgaW4gZW51bWVyYXRlKGcpCiAgICAgICAgICAgICAgICAgICAgICAgIGZvciBqLCBiIGluIGVudW1lcmF0ZShwKSksIHJldmVyc2U9VHJ1ZSkKICAgICAgICB1c2VkX2csIHVzZWRfcCwgdHAgPSBzZXQoKSwgc2V0KCksIDAKICAgICAgICBmb3IgdiwgaSwgaiBpbiBwYWlyczoKICAgICAgICAgICAgaWYgdiA8IElPVV9USFI6CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICBpZiBpIGluIHVzZWRfZyBvciBqIGluIHVzZWRfcDoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHVzZWRfZy5hZGQoaSkKICAgICAgICAgICAgdXNlZF9wLmFkZChqKQogICAgICAgICAgICB0cCArPSAxCiAgICAgICAgICAgIGlvdXMuYXBwZW5kKHYpCiAgICAgICAga2V5ID0gbGFiIGlmIGxhYiBpbiBwZXIgZWxzZSAib3RoZXIiCiAgICAgICAgcGVyW2tleV1bMF0gKz0gdHAKICAgICAgICBwZXJba2V5XVsxXSArPSBsZW4ocCkgLSB0cAogICAgICAgIHBlcltrZXldWzJdICs9IGxlbihnKSAtIHRwCiAgICByZXR1cm4gcGVyLCBpb3VzCgoKZGVmIF9wcmYodHAsIGZwLCBmbik6CiAgICBwID0gdHAgLyAodHAgKyBmcCkgaWYgdHAgKyBmcCBlbHNlIDAuMAogICAgciA9IHRwIC8gKHRwICsgZm4pIGlmIHRwICsgZm4gZWxzZSAwLjAKICAgIGYgPSAyICogcCAqIHIgLyAocCArIHIpIGlmIHAgKyByIGVsc2UgMC4wCiAgICByZXR1cm4geyJwcmVjaXNpb24iOiBwLCAicmVjYWxsIjogciwgImYxIjogZiwgInRwIjogdHAsICJmcCI6IGZwLCAiZm4iOiBmbn0KCgpkZWYgX2Jvb3RzdHJhcF9mMShyb3dzLCBuX2Jvb3QsIHNlZWQpOgogICAgcm5nLCBuLCB2YWxzID0gcmFuZG9tLlJhbmRvbShzZWVkKSwgbGVuKHJvd3MpLCBbXQogICAgZm9yIF8gaW4gcmFuZ2Uobl9ib290KToKICAgICAgICB0cCA9IGZwID0gZm4gPSAwCiAgICAgICAgZm9yIF8gaW4gcmFuZ2Uobik6CiAgICAgICAgICAgIHIgPSByb3dzW3JuZy5yYW5kcmFuZ2UobildCiAgICAgICAgICAgIHRwLCBmcCwgZm4gPSB0cCArIHJbMF0sIGZwICsgclsxXSwgZm4gKyByWzJdCiAgICAgICAgdmFscy5hcHBlbmQoX3ByZih0cCwgZnAsIGZuKVsiZjEiXSkKICAgIHZhbHMuc29ydCgpCiAgICByZXR1cm4gW3ZhbHNbaW50KDAuMDI1ICogbl9ib290KV0sIHZhbHNbbWF4KGludCgwLjk3NSAqIG5fYm9vdCkgLSAxLCAwKV1dCgoKZGVmIGV2YWx1YXRlKGVudHJpZXMsIHByZWRzLCBjb29yZD0ibm9ybTEwMDAiLCBuX2Jvb3Q9MTAwMCwgc2VlZD0wKToKICAgIHBlcl90b3QgPSB7YzogWzAsIDAsIDBdIGZvciBjIGluICgqQ0xBU1NFUywgIm90aGVyIil9CiAgICByb3dzLCBpb3VzLCB2YWxpZCwgbWlzc2luZyA9IFtdLCBbXSwgMCwgMAogICAgZm9yIGUgaW4gZW50cmllczoKICAgICAgICByYXcgPSBwcmVkcy5nZXQoZVsiaW1hZ2VfaWQiXSkKICAgICAgICBwYXJzZWQgPSBOb25lCiAgICAgICAgaWYgcmF3IGlzIE5vbmU6CiAgICAgICAgICAgIG1pc3NpbmcgKz0gMQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHBhcnNlZCA9IHBhcnNlX291dHB1dChyYXcpCiAgICAgICAgcHJlZCA9IFtdCiAgICAgICAgaWYgcGFyc2VkIGlzIG5vdCBOb25lOgogICAgICAgICAgICB2YWxpZCArPSAxCiAgICAgICAgICAgIHByZWQgPSBfdG9fbm9ybTEwMDAocGFyc2VkLCBjb29yZCwgZVsid2lkdGgiXSwgZVsiaGVpZ2h0Il0pCiAgICAgICAgcGVyLCBpID0gX21hdGNoKF9ndF9ib3hlcyhlKSwgcHJlZCkKICAgICAgICBpb3VzICs9IGkKICAgICAgICByb3dzLmFwcGVuZCgoc3VtKHZbMF0gZm9yIHYgaW4gcGVyLnZhbHVlcygpKSwgc3VtKHZbMV0gZm9yIHYgaW4gcGVyLnZhbHVlcygpKSwKICAgICAgICAgICAgICAgICAgICAgc3VtKHZbMl0gZm9yIHYgaW4gcGVyLnZhbHVlcygpKSkpCiAgICAgICAgZm9yIGMsIHYgaW4gcGVyLml0ZW1zKCk6CiAgICAgICAgICAgIGZvciBrIGluIHJhbmdlKDMpOgogICAgICAgICAgICAgICAgcGVyX3RvdFtjXVtrXSArPSB2W2tdCiAgICBuID0gbGVuKGVudHJpZXMpCiAgICB0cCwgZnAsIGZuID0gKHN1bShyW2tdIGZvciByIGluIHJvd3MpIGZvciBrIGluIHJhbmdlKDMpKQogICAgcmV0dXJuIHsKICAgICAgICAibl9pbWFnZXMiOiBuLAogICAgICAgICJwYXJzZV92YWxpZF9yYXRlIjogdmFsaWQgLyBuIGlmIG4gZWxzZSAwLjAsCiAgICAgICAgIm1pc3NpbmdfcHJlZGljdGlvbnMiOiBtaXNzaW5nLAogICAgICAgICJtaWNybyI6IF9wcmYodHAsIGZwLCBmbiksCiAgICAgICAgInBlcl9jbGFzcyI6IHtjOiBfcHJmKCp2KSBmb3IgYywgdiBpbiBwZXJfdG90Lml0ZW1zKCl9LAogICAgICAgICJtZWFuX2lvdSI6IHN1bShpb3VzKSAvIGxlbihpb3VzKSBpZiBpb3VzIGVsc2UgMC4wLAogICAgICAgICJmcF9wZXJfaW1hZ2UiOiBmcCAvIG4gaWYgbiBlbHNlIDAuMCwKICAgICAgICAiZjFfY2k5NSI6IF9ib290c3RyYXBfZjEocm93cywgbl9ib290LCBzZWVkKSBpZiBuIGVsc2UgWzAuMCwgMC4wXSwKICAgIH0KCgpkZWYgX3BhcnNlX21ldGEoaXRlbXMpOgogICAgb3V0ID0ge30KICAgIGZvciBpdCBpbiBpdGVtcyBvciBbXToKICAgICAgICBrLCBfLCB2ID0gaXQucGFydGl0aW9uKCI9IikKICAgICAgICB0cnk6CiAgICAgICAgICAgIG91dFtrXSA9IGZsb2F0KHYpIGlmICIuIiBpbiB2IGVsc2UgaW50KHYpCiAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3I6CiAgICAgICAgICAgIG91dFtrXSA9IHYKICAgIHJldHVybiBvdXQKCgpkZWYgbWFpbihhcmd2PU5vbmUpOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXByZWQiLCByZXF1aXJlZD1UcnVlKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNwbGl0cy1kaXIiLCBkZWZhdWx0PSJkYXRhL3NwbGl0cyIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc3BsaXQiLCBkZWZhdWx0PSJ0ZXN0IiwgY2hvaWNlcz1bInZhbCIsICJ0ZXN0Il0pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY29vcmQiLCBkZWZhdWx0PSJub3JtMTAwMCIsIGNob2ljZXM9WyJub3JtMTAwMCIsICJwaXhlbCJdKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW5hbWUiLCByZXF1aXJlZD1UcnVlKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW91dCIsIHJlcXVpcmVkPVRydWUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbWV0YSIsIGFjdGlvbj0iYXBwZW5kIiwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJrZXk9dmFsdWUgKHRyYWluYWJsZV9wYXJhbXMsIHBlYWtfdnJhbV9nYiwgdHJhaW5fbWludXRlc+KApikiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW4tYm9vdCIsIHR5cGU9aW50LCBkZWZhdWx0PTEwMDApCiAgICBhID0gYXAucGFyc2VfYXJncyhhcmd2KQoKICAgIHNkID0gUGF0aChhLnNwbGl0c19kaXIpCiAgICBlbnRyaWVzID0gcmVhZF9qc29ubChzZCAvIGYie2Euc3BsaXR9Lmpzb25sIikKICAgIG1ldGEgPSBqc29uLmxvYWRzKChzZCAvICJtZXRhLmpzb24iKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBwcmVkcyA9IHt9CiAgICBmb3Igcm93IGluIHJlYWRfanNvbmwoYS5wcmVkKToKICAgICAgICBpZiByb3dbImltYWdlX2lkIl0gaW4gcHJlZHM6CiAgICAgICAgICAgIHN5cy5leGl0KGYiZHVwbGljYXRlIGltYWdlX2lkIGluIHByZWRpY3Rpb25zOiB7cm93WydpbWFnZV9pZCddfSIpCiAgICAgICAgcHJlZHNbcm93WyJpbWFnZV9pZCJdXSA9IHJvd1sicmF3Il0KICAgIGV4dHJhID0gc2V0KHByZWRzKSAtIHtlWyJpbWFnZV9pZCJdIGZvciBlIGluIGVudHJpZXN9CiAgICBpZiBleHRyYToKICAgICAgICBwcmludChmImPhuqNuaCBiw6FvOiBi4buPIHF1YSB7bGVuKGV4dHJhKX0gaW1hZ2VfaWQga2jDtG5nIHRodeG7mWMgc3BsaXQge2Euc3BsaXR9IiwgZmlsZT1zeXMuc3RkZXJyKQoKICAgIHJlcyA9IGV2YWx1YXRlKGVudHJpZXMsIHByZWRzLCBjb29yZD1hLmNvb3JkLCBuX2Jvb3Q9YS5uX2Jvb3QpCiAgICByZXMudXBkYXRlKHsKICAgICAgICAic2NvcmVfdmVyc2lvbiI6IFNDT1JFX1ZFUlNJT04sICJuYW1lIjogYS5uYW1lLAogICAgICAgICJkYXRhc2V0X3ZlcnNpb24iOiBtZXRhWyJkYXRhc2V0X3ZlcnNpb24iXSwgInNwbGl0IjogYS5zcGxpdCwgImNvb3JkIjogYS5jb29yZCwKICAgICAgICAibWV0YSI6IF9wYXJzZV9tZXRhKGEubWV0YSksCiAgICB9KQogICAgb3V0ID0gUGF0aChhLm91dCkKICAgIG91dC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgb3V0LndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXMsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgbSA9IHJlc1sibWljcm8iXQogICAgcHJpbnQoZiJ7YS5uYW1lfSBbe2Euc3BsaXR9XSBuPXtyZXNbJ25faW1hZ2VzJ119IHZhbGlkPXtyZXNbJ3BhcnNlX3ZhbGlkX3JhdGUnXTouM2Z9ICIKICAgICAgICAgIGYiUD17bVsncHJlY2lzaW9uJ106LjNmfSBSPXttWydyZWNhbGwnXTouM2Z9IEYxPXttWydmMSddOi4zZn0gIgogICAgICAgICAgZiJDSTk1PXtyZXNbJ2YxX2NpOTUnXVswXTouM2Z9LXtyZXNbJ2YxX2NpOTUnXVsxXTouM2Z9IG1lYW5Jb1U9e3Jlc1snbWVhbl9pb3UnXTouM2Z9ICIKICAgICAgICAgIGYiRlAvaW1nPXtyZXNbJ2ZwX3Blcl9pbWFnZSddOi4yZn0iKQogICAgZm9yIGMsIHYgaW4gcmVzWyJwZXJfY2xhc3MiXS5pdGVtcygpOgogICAgICAgIGlmIGMgIT0gIm90aGVyIiBvciB2WyJmcCJdOgogICAgICAgICAgICBwcmludChmIiAgIHtjOjhzfSBQPXt2WydwcmVjaXNpb24nXTouM2Z9IFI9e3ZbJ3JlY2FsbCddOi4zZn0gRjE9e3ZbJ2YxJ106LjNmfSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=", "scripts/splitting.py": "IiIiR29tIOG6o25oIHRoZW8gcGhpw6puIHF1YXkgdsOgIGfDoW4gc3BsaXQgY+G7kSDEkeG7i25oIChjaOG7kW5nIHLDsiBy4buJIGdp4buvYSBjw6FjIGZyYW1lIGxp4buBbiBr4buBKS4iIiIKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IHJlCgpTUExJVFMgPSAoInRyYWluIiwgInZhbCIsICJ0ZXN0IikKVEFSR0VUID0geyJ0cmFpbiI6IDAuNzAsICJ2YWwiOiAwLjE1LCAidGVzdCI6IDAuMTV9Ck1JTl9GUkFDID0geyJ0cmFpbiI6IDAuNTAsICJ2YWwiOiAwLjEwLCAidGVzdCI6IDAuMTB9Ck1JTl9DTEFTU19CT1hFUyA9IDIwClNFRUQgPSAiZGF5MTAiCgpfUEFUVEVSTlMgPSBbCiAgICAocmUuY29tcGlsZShyIl52bGNzbmFwWy1fXShcZHs0fS1cZHsyfS1cZHsyfSktKFxkezJ9KWhcZHsyfW1cZHsyfXNcZCtcLmpwZyQiKSwKICAgICBsYW1iZGEgbTogZiJ2bGMte20uZ3JvdXAoMSl9LXttLmdyb3VwKDIpfSIpLAogICAgKHJlLmNvbXBpbGUociJeKFxkezh9KV9cZHs2fVwuanBnJCIpLAogICAgIGxhbWJkYSBtOiBmInBob25lLXttLmdyb3VwKDEpfSIpLAogICAgKHJlLmNvbXBpbGUociJedmxjc25hcC0oXGQrKSg/Oi1bQS1aYS16XSspP1wuanBnJCIpLAogICAgIGxhbWJkYSBtOiBmIm51bS17aW50KG0uZ3JvdXAoMSkpIC8vIDUwfSIpLApdCgoKZGVmIHNlc3Npb25fa2V5KGZpbGVuYW1lKToKICAgIGZvciBwYXQsIGZuIGluIF9QQVRURVJOUzoKICAgICAgICBtID0gcGF0Lm1hdGNoKGZpbGVuYW1lKQogICAgICAgIGlmIG06CiAgICAgICAgICAgIHJldHVybiBmbihtKQogICAgcmFpc2UgVmFsdWVFcnJvcihmInVucmVjb2duaXNlZCBpbWFnZSBmaWxlbmFtZToge2ZpbGVuYW1lfSIpCgoKZGVmIF9hdHRlbXB0KGdyb3Vwcywgc2VlZCwgc2FsdCk6CiAgICBvcmRlciA9IHNvcnRlZCgKICAgICAgICBncm91cHMsCiAgICAgICAga2V5PWxhbWJkYSBrOiBoYXNobGliLnNoYTI1NihmIntzZWVkfTp7c2FsdH06e2t9Ii5lbmNvZGUoKSkuaGV4ZGlnZXN0KCksCiAgICApCiAgICB0b3RhbCA9IHN1bShnWyJpbWFnZXMiXSBmb3IgZyBpbiBncm91cHMudmFsdWVzKCkpCiAgICBjdXIgPSB7czogMCBmb3IgcyBpbiBTUExJVFN9CiAgICBvdXQgPSB7fQogICAgZm9yIGsgaW4gb3JkZXI6CiAgICAgICAgcyA9IG1heChTUExJVFMsIGtleT1sYW1iZGEgczogVEFSR0VUW3NdICogdG90YWwgLSBjdXJbc10pCiAgICAgICAgb3V0W2tdID0gcwogICAgICAgIGN1cltzXSArPSBncm91cHNba11bImltYWdlcyJdCiAgICByZXR1cm4gb3V0CgoKZGVmIF9pc192YWxpZChncm91cHMsIGFzc2lnbiwgbWluX2ZyYWMsIG1pbl9jbGFzc19ib3hlcyk6CiAgICB0b3RhbCA9IHN1bShnWyJpbWFnZXMiXSBmb3IgZyBpbiBncm91cHMudmFsdWVzKCkpCiAgICBmb3IgcyBpbiBTUExJVFM6CiAgICAgICAgaW1ncyA9IHN1bShncm91cHNba11bImltYWdlcyJdIGZvciBrLCB2IGluIGFzc2lnbi5pdGVtcygpIGlmIHYgPT0gcykKICAgICAgICBpZiBpbWdzIDwgbWluX2ZyYWNbc10gKiB0b3RhbCBvciBpbWdzID09IDA6CiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgY2xhc3NlcyA9IHtjIGZvciBnIGluIGdyb3Vwcy52YWx1ZXMoKSBmb3IgYyBpbiBnWyJib3hlcyJdfQogICAgZm9yIHMgaW4gKCJ2YWwiLCAidGVzdCIpOgogICAgICAgIGZvciBjIGluIGNsYXNzZXM6CiAgICAgICAgICAgIG4gPSBzdW0oZ3JvdXBzW2tdWyJib3hlcyJdLmdldChjLCAwKSBmb3IgaywgdiBpbiBhc3NpZ24uaXRlbXMoKSBpZiB2ID09IHMpCiAgICAgICAgICAgIGlmIG4gPCBtaW5fY2xhc3NfYm94ZXM6CiAgICAgICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgIHJldHVybiBUcnVlCgoKZGVmIGFzc2lnbl9zcGxpdHMoZ3JvdXBzLCBzZWVkPVNFRUQsIG1pbl9mcmFjPU1JTl9GUkFDLAogICAgICAgICAgICAgICAgICBtaW5fY2xhc3NfYm94ZXM9TUlOX0NMQVNTX0JPWEVTLCBtYXhfc2FsdD0yMDAwKToKICAgICIiIkfDoW4gbeG7l2kgcGhpw6puIHbDoG8gbeG7mXQgc3BsaXQuIEPDuW5nIChncm91cHMsIHNlZWQpIGx1w7RuIHJhIGPDuW5nIGvhur90IHF14bqjLiIiIgogICAgZm9yIHNhbHQgaW4gcmFuZ2UobWF4X3NhbHQpOgogICAgICAgIGEgPSBfYXR0ZW1wdChncm91cHMsIHNlZWQsIHNhbHQpCiAgICAgICAgaWYgX2lzX3ZhbGlkKGdyb3VwcywgYSwgbWluX2ZyYWMsIG1pbl9jbGFzc19ib3hlcyk6CiAgICAgICAgICAgIHJldHVybiBhLCBzYWx0CiAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJubyB2YWxpZCBzcGxpdCBmb3Igc2VlZCB7c2VlZCFyfSB3aXRoaW4ge21heF9zYWx0fSBzYWx0cyIpCgoKZGVmIGRhdGFzZXRfdmVyc2lvbihpbWFnZV9zcGxpdHMpOgogICAgaCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIGZvciBpbWFnZV9pZCwgc3BsaXQgaW4gc29ydGVkKGltYWdlX3NwbGl0cyk6CiAgICAgICAgaC51cGRhdGUoZiJ8e2ltYWdlX2lkfTp7c3BsaXR9Ii5lbmNvZGUoKSkKICAgIHJldHVybiAiZDEwLSIgKyBoLmhleGRpZ2VzdCgpWzoxMF0K", "tests/test_check_submission.py": "aW1wb3J0IGpzb24KaW1wb3J0IHN1YnByb2Nlc3MKCmZyb20gc2NyaXB0cy5jaGVja19zdWJtaXNzaW9uIGltcG9ydCBydW5fY2hlY2tzCgoKZGVmIF9ldihuYW1lLCB2ZXI9ImQxMC14IiwgdHJhaW5lZD1UcnVlLCAqKm92ZXIpOgogICAgZCA9IHsic2NvcmVfdmVyc2lvbiI6IDEsICJuYW1lIjogbmFtZSwgImRhdGFzZXRfdmVyc2lvbiI6IHZlciwKICAgICAgICAgInNwbGl0IjogInRlc3QiLCAibWlzc2luZ19wcmVkaWN0aW9ucyI6IDAsICJtaWNybyI6IHsiZjEiOiAwLjV9LAogICAgICAgICAibWV0YSI6IHsidHJhaW5hYmxlX3BhcmFtcyI6IDFlNiwgInBlYWtfdnJhbV9nYiI6IDkuMCwgInRyYWluX21pbnV0ZXMiOiAxMi4wfQogICAgICAgICBpZiB0cmFpbmVkIGVsc2Uge319CiAgICBkLnVwZGF0ZShvdmVyKQogICAgcmV0dXJuIGQKCgpkZWYgX3NldHVwKHJvb3QsIGV2cywgcmVwb3J0PSJI4buNIHTDqm46IEFcblxuTuG7mWkgZHVuZyDEkcOjIMSRaeG7gW4uXG4iLCBtZXRhPVRydWUpOgogICAgZXYgPSByb290IC8gInJlcG9ydCIgLyAiZXZpZGVuY2UiCiAgICBldi5ta2RpcihwYXJlbnRzPVRydWUpCiAgICBmb3IgZm5hbWUsIGQgaW4gZXZzLml0ZW1zKCk6CiAgICAgICAgKGV2IC8gZm5hbWUpLndyaXRlX3RleHQoanNvbi5kdW1wcyhkKSkKICAgIChyb290IC8gIlJFUE9SVC5tZCIpLndyaXRlX3RleHQocmVwb3J0LCBlbmNvZGluZz0idXRmLTgiKQogICAgaWYgbWV0YToKICAgICAgICAocm9vdCAvICJkYXRhIiAvICJzcGxpdHMiKS5ta2RpcihwYXJlbnRzPVRydWUpCiAgICAgICAgKHJvb3QgLyAiZGF0YSIgLyAic3BsaXRzIiAvICJtZXRhLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoeyJkYXRhc2V0X3ZlcnNpb24iOiAiZDEwLXgifSkpCgoKZGVmIF9nb29kKCk6CiAgICByZXR1cm4geyJlMC5qc29uIjogX2V2KCJFMCIsIHRyYWluZWQ9RmFsc2UpLCAiZTEuanNvbiI6IF9ldigiRTEiKSwgImUyLmpzb24iOiBfZXYoIkUyIiksCiAgICAgICAgICAgICJlMy5qc29uIjogX2V2KCJFMyIpLCAiZTRfcmFuay5qc29uIjogX2V2KCJFNCIpfQoKCmRlZiBfaGFzKHJlcywgc3RhdHVzLCB0ZXh0KToKICAgIHJldHVybiBhbnkoc3QgPT0gc3RhdHVzIGFuZCB0ZXh0IGluIG5hbWUgZm9yIHN0LCBuYW1lLCBfIGluIHJlcykKCgpkZWYgdGVzdF9hbGxfZ29vZF9oYXNfbm9fZmFpbCh0bXBfcGF0aCk6CiAgICBfc2V0dXAodG1wX3BhdGgsIF9nb29kKCkpCiAgICByZXMgPSBydW5fY2hlY2tzKHRtcF9wYXRoKQogICAgYXNzZXJ0IG5vdCBbciBmb3IgciBpbiByZXMgaWYgclswXSA9PSAiRkFJTCJdLCByZXMKCgpkZWYgdGVzdF9taXNzaW5nX3JlcXVpcmVkX2V2aWRlbmNlX2ZhaWxzKHRtcF9wYXRoKToKICAgIGcgPSBfZ29vZCgpOyBkZWwgZ1siZTIuanNvbiJdCiAgICBfc2V0dXAodG1wX3BhdGgsIGcpCiAgICBhc3NlcnQgX2hhcyhydW5fY2hlY2tzKHRtcF9wYXRoKSwgIkZBSUwiLCAiZTIiKQoKCmRlZiB0ZXN0X21pc3NpbmdfZTRfZmFpbHModG1wX3BhdGgpOgogICAgZyA9IF9nb29kKCk7IGRlbCBnWyJlNF9yYW5rLmpzb24iXQogICAgX3NldHVwKHRtcF9wYXRoLCBnKQogICAgYXNzZXJ0IF9oYXMocnVuX2NoZWNrcyh0bXBfcGF0aCksICJGQUlMIiwgImU0IikKCgpkZWYgdGVzdF9kYXRhc2V0X3ZlcnNpb25fbWlzbWF0Y2hfYmV0d2Vlbl9maWxlc19mYWlscyh0bXBfcGF0aCk6CiAgICBnID0gX2dvb2QoKTsgZ1siZTMuanNvbiJdID0gX2V2KCJFMyIsIHZlcj0iZDEwLW90aGVyIikKICAgIF9zZXR1cCh0bXBfcGF0aCwgZykKICAgIGFzc2VydCBfaGFzKHJ1bl9jaGVja3ModG1wX3BhdGgpLCAiRkFJTCIsICJkYXRhc2V0X3ZlcnNpb24iKQoKCmRlZiB0ZXN0X3ZlcnNpb25fbm90X21hdGNoaW5nX2xvY2FsX21ldGFfZmFpbHModG1wX3BhdGgpOgogICAgZyA9IHtrOiBfZXYodlsibmFtZSJdLCB2ZXI9ImQxMC16enoiLCB0cmFpbmVkPXZbIm1ldGEiXSAhPSB7fSkgZm9yIGssIHYgaW4gX2dvb2QoKS5pdGVtcygpfQogICAgX3NldHVwKHRtcF9wYXRoLCBnKQogICAgYXNzZXJ0IF9oYXMocnVuX2NoZWNrcyh0bXBfcGF0aCksICJGQUlMIiwgIm1ldGEuanNvbiIpCgoKZGVmIHRlc3RfbWlzc2luZ190cmFpbmVkX21ldGFfZmFpbHModG1wX3BhdGgpOgogICAgZyA9IF9nb29kKCk7IGdbImUxLmpzb24iXSA9IF9ldigiRTEiLCB0cmFpbmVkPUZhbHNlKQogICAgX3NldHVwKHRtcF9wYXRoLCBnKQogICAgYXNzZXJ0IF9oYXMocnVuX2NoZWNrcyh0bXBfcGF0aCksICJGQUlMIiwgImUxIG1ldGEiKQoKCmRlZiB0ZXN0X21pc3NpbmdfcHJlZGljdGlvbnNfZmFpbHModG1wX3BhdGgpOgogICAgZyA9IF9nb29kKCk7IGdbImUxLmpzb24iXSA9IF9ldigiRTEiLCBtaXNzaW5nX3ByZWRpY3Rpb25zPTMpCiAgICBfc2V0dXAodG1wX3BhdGgsIGcpCiAgICBhc3NlcnQgX2hhcyhydW5fY2hlY2tzKHRtcF9wYXRoKSwgIkZBSUwiLCAiZTEiKQoKCmRlZiB0ZXN0X3JlcG9ydF9wbGFjZWhvbGRlcl9mYWlscyh0bXBfcGF0aCk6CiAgICBfc2V0dXAodG1wX3BhdGgsIF9nb29kKCksIHJlcG9ydD0iPMSRaeG7gW4gY8OidSB0cuG6oyBs4budaT5cbiIpCiAgICBhc3NlcnQgX2hhcyhydW5fY2hlY2tzKHRtcF9wYXRoKSwgIkZBSUwiLCAiUkVQT1JUIikKCgpkZWYgdGVzdF9jb3JydXB0X2V2aWRlbmNlX2pzb25fZmFpbHNfbm90X2NyYXNoZXModG1wX3BhdGgpOgogICAgX3NldHVwKHRtcF9wYXRoLCBfZ29vZCgpKQogICAgKHRtcF9wYXRoIC8gInJlcG9ydCIgLyAiZXZpZGVuY2UiIC8gImUxLmpzb24iKS53cml0ZV90ZXh0KCJ7bm90IGpzb24iKQogICAgYXNzZXJ0IF9oYXMocnVuX2NoZWNrcyh0bXBfcGF0aCksICJGQUlMIiwgImUxIikKCgpkZWYgdGVzdF9ub19naXRfaXNfd2FybmluZ19vbmx5KHRtcF9wYXRoKToKICAgIF9zZXR1cCh0bXBfcGF0aCwgX2dvb2QoKSkKICAgIGFzc2VydCBfaGFzKHJ1bl9jaGVja3ModG1wX3BhdGgpLCAiV0FSTiIsICJnaXQiKQoKCmRlZiB0ZXN0X2dpdF9jb21taXRfY291bnRfYW5kX2ZvcmJpZGRlbl9maWxlcyh0bXBfcGF0aCk6CiAgICBfc2V0dXAodG1wX3BhdGgsIF9nb29kKCkpCgogICAgZGVmIHJ1bigqYSk6CiAgICAgICAgc3VicHJvY2Vzcy5ydW4oWyJnaXQiLCAiLUMiLCBzdHIodG1wX3BhdGgpLCAqYV0sIGNoZWNrPVRydWUsIGNhcHR1cmVfb3V0cHV0PVRydWUpCgogICAgcnVuKCJpbml0IiwgIi1xIik7IHJ1bigiY29uZmlnIiwgInVzZXIuZW1haWwiLCAidEB0Iik7IHJ1bigiY29uZmlnIiwgInVzZXIubmFtZSIsICJ0IikKICAgICh0bXBfcGF0aCAvICJtb2RlbC5zYWZldGVuc29ycyIpLndyaXRlX3RleHQoIngiKQogICAgcnVuKCJhZGQiLCAiLUEiKTsgcnVuKCJjb21taXQiLCAiLXFtIiwgIm9uZSIpCiAgICByZXMgPSBydW5fY2hlY2tzKHRtcF9wYXRoKQogICAgYXNzZXJ0IF9oYXMocmVzLCAiRkFJTCIsICJjb21taXRzIikgYW5kIF9oYXMocmVzLCAiRkFJTCIsICJmb3JiaWRkZW4iKQo=", "tests/test_configs_and_docs.py": "aW1wb3J0IGpzb24KaW1wb3J0IHJlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHlhbWwKClJPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXQpET0NTID0ge246IChST09UIC8gbikucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgZm9yIG4gaW4gKCJSRUFETUUubWQiLCAiR1VJREUubWQiLCAiUlVCUklDLm1kIiwgIlJFUE9SVC5tZCIpfQoKCmRlZiBfY2ZnKG4pOgogICAgcmV0dXJuIHlhbWwuc2FmZV9sb2FkKChST09UIC8gImNvbmZpZ3MiIC8gZiJle259LnlhbWwiKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCgoKZGVmIHRlc3RfYWxsX2NvbmZpZ3NfcGFyc2VfYW5kX2hhdmVfbmFtZSgpOgogICAgZm9yIG4gaW4gcmFuZ2UoNik6CiAgICAgICAgZCA9IF9jZmcobikKICAgICAgICBhc3NlcnQgZFsibmFtZSJdID09IGYiRXtufSIgYW5kIGRbImRlc2NyaXB0aW9uIl0KCgpkZWYgdGVzdF90cmFpbmVkX2NvbmZpZ3NfZGlmZmVyX29ubHlfaW5fdGhlX3ZhcmlhYmxlKCk6CiAgICBjZmdzID0ge246IF9jZmcobikgZm9yIG4gaW4gKDEsIDIsIDMpfQogICAgZm9yIGsgaW4gKCJsciIsICJiYXRjaF9zaXplIiwgImdyYWRfYWNjdW0iLCAibWF4X3N0ZXBzIiwgInNlZWQiKToKICAgICAgICBhc3NlcnQgbGVuKHtjW2tdIGZvciBjIGluIGNmZ3MudmFsdWVzKCl9KSA9PSAxLCBrCiAgICBhc3NlcnQgW2NmZ3Nbbl1bInRyYWluX3Byb2plY3RvciJdIGZvciBuIGluICgxLCAyLCAzKV0gPT0gW0ZhbHNlLCBUcnVlLCBUcnVlXQogICAgYXNzZXJ0IFtjZmdzW25dWyJsb3JhIl1bInRhcmdldHMiXSBmb3IgbiBpbiAoMSwgMiwgMyldID09IFsibGxtIiwgImxsbSIsICJsbG0rdml0Il0KCgpkZWYgdGVzdF9ub3RlYm9va19zZWN0aW9uc19hbmRfbm9fc2VjcmV0cygpOgogICAgbmIgPSBqc29uLmxvYWRzKChST09UIC8gImxhYl9kYXkxMC5pcHluYiIpLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIGFzc2VydCBuYlsibmJmb3JtYXQiXSA9PSA0CiAgICBoZWFkcyA9IFtjWyJzb3VyY2UiXVswXSBmb3IgYyBpbiBuYlsiY2VsbHMiXSBpZiBjWyJjZWxsX3R5cGUiXSA9PSAibWFya2Rvd24iIGFuZCBjWyJzb3VyY2UiXV0KICAgIGZvciBpIGluIHJhbmdlKDkpOgogICAgICAgIGFzc2VydCBhbnkoaC5zdGFydHN3aXRoKGYiIyMge2l9LiIpIGZvciBoIGluIGhlYWRzKSwgaQogICAgdGV4dCA9IGpzb24uZHVtcHMobmIpCiAgICBhc3NlcnQgbm90IHJlLnNlYXJjaChyIktBR0dMRV9LRVknP1xdP1xzKj1ccypbJ1wiXVswLTlhLWZdezIwLH0iLCB0ZXh0KQoKCmRlZiB0ZXN0X25vX3BsYWNlaG9sZGVyc19pbl9yZWFkbWVfZ3VpZGVfcnVicmljKCk6CiAgICBmb3IgbiBpbiAoIlJFQURNRS5tZCIsICJHVUlERS5tZCIsICJSVUJSSUMubWQiKToKICAgICAgICBhc3NlcnQgbm90IHJlLnNlYXJjaChyIlxiVEJEXGJ8PMSRaeG7gW4iLCBET0NTW25dKSwgbgoKCmRlZiB0ZXN0X3JlcG9ydF90ZW1wbGF0ZV9oYXNfc3R1ZGVudF9tYXJrZXJzX2FuZF90ZW5fcXVlc3Rpb25zKCk6CiAgICBhc3NlcnQgIjzEkWnhu4FuIiBpbiBET0NTWyJSRVBPUlQubWQiXQogICAgYXNzZXJ0IGxlbihyZS5maW5kYWxsKHIiXiMjIyBDw6J1IFxkKyIsIERPQ1NbIlJFUE9SVC5tZCJdLCByZS5NKSkgPT0gMTAKCgpkZWYgdGVzdF9kb2NzX21lbnRpb25fcmVhbF9jb21tYW5kc19hbmRfdmVyaWZpZWRfZmFjdHMoKToKICAgIGcgPSBET0NTWyJHVUlERS5tZCJdCiAgICBmb3IgdG9rZW4gaW4gKCJwcmVwYXJlX2RhdGEucHkiLCAic2NvcmUucHkiLCAiYmJveF8yZCIsICIw4oCTMTAwMCIsICJsbG1faW50OF9za2lwX21vZHVsZXMiLAogICAgICAgICAgICAgICAgICAidmlzdWFsLm1lcmdlciIsICJ2aXN1YWwuZGVlcHN0YWNrX21lcmdlcl9saXN0IiwgImxhbmd1YWdlX21vZGVsIiwKICAgICAgICAgICAgICAgICAgInJlcG9ydC9ldmlkZW5jZSIsICIyMjAiLCAiMjUyIiwgIjM0OCIpOgogICAgICAgIGFzc2VydCB0b2tlbiBpbiBnLCB0b2tlbgogICAgYXNzZXJ0ICJjaGVja19zdWJtaXNzaW9uLnB5IiBpbiBET0NTWyJSRUFETUUubWQiXQoKCmRlZiB0ZXN0X3J1YnJpY19zdW1zX3RvXzEwMCgpOgogICAgcHRzID0gW2ludCh4KSBmb3IgeCBpbiByZS5maW5kYWxsKHIiXlx8XHMqW0EtRF1cLltefF0qXHxccyooXGQrKVxzKlx8IiwgRE9DU1siUlVCUklDLm1kIl0sIHJlLk0pXQogICAgYXNzZXJ0IHN1bShwdHMpID09IDEwMCwgcHRzCiAgICB0ZXh0ID0gRE9DU1siUlVCUklDLm1kIl0KICAgIGZvciBoZWFkLCBueHQsIHRvdGFsIGluICgoIiMjIEEuIiwgIiMjIEIuIiwgNDApLCAoIiMjIEIuIiwgIiMjIEJvbnVzIiwgMjApKToKICAgICAgICBzZWN0aW9uID0gdGV4dFt0ZXh0LmluZGV4KGhlYWQpOnRleHQuaW5kZXgobnh0KV0KICAgICAgICBzdWIgPSBbaW50KHgpIGZvciB4IGluIHJlLmZpbmRhbGwociJeXHxbXnxcbl0qXHxccyooXGQrKVxzKlx8Iiwgc2VjdGlvbiwgcmUuTSldCiAgICAgICAgYXNzZXJ0IHN1bShzdWIpID09IHRvdGFsLCAoaGVhZCwgc3ViKQoKCmRlZiB0ZXN0X3JlYWRtZV9zcGxpdF90YWJsZV9tYXRjaGVzX3JlYWxfc3BsaXRfc2l6ZXMoKToKICAgICMgc+G7kSBsaeG7h3UgxJHDoyDEkW8gYuG6sW5nIHByZXBhcmVfZGF0YS5weSB0csOqbiBk4buvIGxp4buHdSB0aOG6rXQgKGRhdGFzZXRfdmVyc2lvbiBkMTAtMjFlNDA3MDMwMSkKICAgIGZvciBuIGluICgiMS40MjIiLCAiMjc2IiwgIjMxMSIpOgogICAgICAgIGFzc2VydCBuIGluIERPQ1NbIlJFQURNRS5tZCJdCg==", "tests/test_prepare_data.py": "aW1wb3J0IGpzb24KaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHB5dGVzdApmcm9tIFBJTCBpbXBvcnQgSW1hZ2UKCmZyb20gc2NyaXB0cy5sYWJjb21tb24gaW1wb3J0IHJlYWRfanNvbmwKZnJvbSBzY3JpcHRzLnByZXBhcmVfZGF0YSBpbXBvcnQgX2dyb3VwX3N0YXRzLCBsb2FkX3JlY29yZHMsIHByZXBhcmUKZnJvbSBzY3JpcHRzLnNwbGl0dGluZyBpbXBvcnQgYXNzaWduX3NwbGl0cwoKRklMRVMgPSBbCiAgICAidmxjc25hcC0yMDI1LTAyLTE4LTE3aDAwbTAwczAwMS5qcGciLCAidmxjc25hcC0yMDI1LTAyLTE4LTE4aDAwbTAwczAwMS5qcGciLAogICAgInZsY3NuYXAtMjAyNS0wMi0xOS0xNGgwMG0wMHMwMDEuanBnIiwgIjIwMjUwMjE5XzE2NDY0OS5qcGciLAogICAgIjIwMjUwMjIzXzEwMTAxMC5qcGciLCAidmxjc25hcC0wMDAwMS5qcGciLCAidmxjc25hcC0wMDA2MC5qcGciLCAidmxjc25hcC0wMDEyMC5qcGciLApdCkxBQkVMID0gIjAgMC41IDAuNSAwLjIgMC4yXG4xIDAuMyAwLjMgMC4xIDAuMVxuMiAwLjcgMC43IDAuMSAwLjFcbiIKTk9fTUlOID0geyJ0cmFpbiI6IDAsICJ2YWwiOiAwLCAidGVzdCI6IDB9CgoKZGVmIF9tYWtlX2RhdGFzZXQocm9vdDogUGF0aCwgZmlsZXMsIGxhYmVscyk6CiAgICAocm9vdCAvICJpbWFnZXMiKS5ta2RpcihwYXJlbnRzPVRydWUpCiAgICAocm9vdCAvICJsYWJlbHMtWU9MTyIpLm1rZGlyKHBhcmVudHM9VHJ1ZSkKICAgIGZvciBuYW1lIGluIGZpbGVzOgogICAgICAgIEltYWdlLm5ldygiUkdCIiwgKDY0MCwgMzYwKSwgKDEwLCAxMCwgMTApKS5zYXZlKHJvb3QgLyAiaW1hZ2VzIiAvIG5hbWUpCiAgICBmb3Igc3RlbSwgdGV4dCBpbiBsYWJlbHMuaXRlbXMoKToKICAgICAgICAocm9vdCAvICJsYWJlbHMtWU9MTyIgLyBmIntzdGVtfS50eHQiKS53cml0ZV90ZXh0KHRleHQsIGVuY29kaW5nPSJ1dGYtOCIpCgoKZGVmIF9sYWJlbHMoZmlsZXMsIHRleHQ9TEFCRUwpOgogICAgcmV0dXJuIHtQYXRoKGYpLnN0ZW06IHRleHQgZm9yIGYgaW4gZmlsZXN9CgoKZGVmIHRlc3RfbG9hZF9yZWNvcmRzX3BhcnNlc19vYmplY3RzX2FuZF9zZXNzaW9ucyh0bXBfcGF0aCk6CiAgICBfbWFrZV9kYXRhc2V0KHRtcF9wYXRoLCBGSUxFUywgX2xhYmVscyhGSUxFUykpCiAgICByZWNzID0gbG9hZF9yZWNvcmRzKHRtcF9wYXRoKQogICAgYXNzZXJ0IGxlbihyZWNzKSA9PSBsZW4oRklMRVMpCiAgICByID0gbmV4dCh4IGZvciB4IGluIHJlY3MgaWYgeFsiZmlsZW5hbWUiXSA9PSAiMjAyNTAyMTlfMTY0NjQ5LmpwZyIpCiAgICBhc3NlcnQgKHJbIndpZHRoIl0sIHJbImhlaWdodCJdLCByWyJzZXNzaW9uIl0pID09ICg2NDAsIDM2MCwgInBob25lLTIwMjUwMjE5IikKICAgIGFzc2VydCBbb1sibGFiZWwiXSBmb3IgbyBpbiByWyJvYmplY3RzIl1dID09IFsicG90aG9sZSIsICJjcmFjayIsICJtYW5ob2xlIl0KICAgIGFzc2VydCByWyJvYmplY3RzIl1bMF1bInlvbG8iXSA9PSBbMC41LCAwLjUsIDAuMiwgMC4yXQoKCkBweXRlc3QubWFyay5wYXJhbWV0cml6ZSgiYmFkLG1zZyIsIFsKICAgICgiNSAwLjUgMC41IDAuMiAwLjJcbiIsICJjbGFzcyIpLAogICAgKCIwIDEuNSAwLjUgMC4yIDAuMlxuIiwgInJhbmdlIiksCiAgICAoIjAgMC41IDAuNVxuIiwgIm1hbGZvcm1lZCIpLAogICAgKCIwIGEgYiBjIGRcbiIsICJtYWxmb3JtZWQiKSwKXSkKZGVmIHRlc3RfYmFkX2xhYmVsX2xpbmVzX25hbWVfdGhlX2ZpbGUodG1wX3BhdGgsIGJhZCwgbXNnKToKICAgIGxhYmVscyA9IF9sYWJlbHMoRklMRVMpCiAgICBsYWJlbHNbUGF0aChGSUxFU1swXSkuc3RlbV0gPSBiYWQKICAgIF9tYWtlX2RhdGFzZXQodG1wX3BhdGgsIEZJTEVTLCBsYWJlbHMpCiAgICB3aXRoIHB5dGVzdC5yYWlzZXMoVmFsdWVFcnJvcikgYXMgZToKICAgICAgICBsb2FkX3JlY29yZHModG1wX3BhdGgpCiAgICBhc3NlcnQgRklMRVNbMF0gaW4gc3RyKGUudmFsdWUpIGFuZCBtc2cgaW4gc3RyKGUudmFsdWUpCgoKZGVmIHRlc3RfbWlzc2luZ19sYWJlbF9maWxlX2lzX2xvdWQodG1wX3BhdGgpOgogICAgbGFiZWxzID0gX2xhYmVscyhGSUxFUykKICAgIGRlbCBsYWJlbHNbUGF0aChGSUxFU1sxXSkuc3RlbV0KICAgIF9tYWtlX2RhdGFzZXQodG1wX3BhdGgsIEZJTEVTLCBsYWJlbHMpCiAgICB3aXRoIHB5dGVzdC5yYWlzZXMoVmFsdWVFcnJvciwgbWF0Y2g9RklMRVNbMV0pOgogICAgICAgIGxvYWRfcmVjb3Jkcyh0bXBfcGF0aCkKCgpkZWYgdGVzdF91bmtub3duX2ZpbGVuYW1lX2lzX2xvdWQodG1wX3BhdGgpOgogICAgZmlsZXMgPSBGSUxFUyArIFsiSU1HXzAwMDEuanBnIl0KICAgIF9tYWtlX2RhdGFzZXQodG1wX3BhdGgsIGZpbGVzLCBfbGFiZWxzKGZpbGVzKSkKICAgIHdpdGggcHl0ZXN0LnJhaXNlcyhWYWx1ZUVycm9yLCBtYXRjaD0iSU1HXzAwMDEuanBnIik6CiAgICAgICAgbG9hZF9yZWNvcmRzKHRtcF9wYXRoKQoKCmRlZiB0ZXN0X3ByZXBhcmVfd3JpdGVzX3NwbGl0c19tZXRhX2FuZF9pbWFnZXNfYW5kX2lzX3JlcHJvZHVjaWJsZSh0bXBfcGF0aCk6CiAgICBzcmMgPSB0bXBfcGF0aCAvICJzcmMiCiAgICBfbWFrZV9kYXRhc2V0KHNyYywgRklMRVMsIF9sYWJlbHMoRklMRVMpKQogICAgbTEgPSBwcmVwYXJlKHNyYywgdG1wX3BhdGggLyAibzEiLCBtaW5fZnJhYz1OT19NSU4sIG1pbl9jbGFzc19ib3hlcz0wKQogICAgbTIgPSBwcmVwYXJlKHNyYywgdG1wX3BhdGggLyAibzIiLCBtaW5fZnJhYz1OT19NSU4sIG1pbl9jbGFzc19ib3hlcz0wKQogICAgYXNzZXJ0IG0xWyJkYXRhc2V0X3ZlcnNpb24iXSA9PSBtMlsiZGF0YXNldF92ZXJzaW9uIl0gICAgICAgIyBj4buRIMSR4buLbmggY2hvIGPhuqMgbOG7m3AKICAgIG91dCA9IHRtcF9wYXRoIC8gIm8xIgogICAgcm93cyA9IHtzOiByZWFkX2pzb25sKG91dCAvICJzcGxpdHMiIC8gZiJ7c30uanNvbmwiKSBmb3IgcyBpbiAoInRyYWluIiwgInZhbCIsICJ0ZXN0Iil9CiAgICBhc3NlcnQgc3VtKGxlbih2KSBmb3IgdiBpbiByb3dzLnZhbHVlcygpKSA9PSBsZW4oRklMRVMpCiAgICBzZXNzaW9ucyA9IHt9CiAgICBmb3IgcywgdiBpbiByb3dzLml0ZW1zKCk6CiAgICAgICAgZm9yIHIgaW4gdjoKICAgICAgICAgICAgc2Vzc2lvbnMuc2V0ZGVmYXVsdChyWyJzZXNzaW9uIl0sIHNldCgpKS5hZGQocykKICAgICAgICAgICAgYXNzZXJ0IChvdXQgLyByWyJpbWFnZV9wYXRoIl0pLmV4aXN0cygpCiAgICBhc3NlcnQgYWxsKGxlbih2KSA9PSAxIGZvciB2IGluIHNlc3Npb25zLnZhbHVlcygpKSAgICAgICAgICAjIGtow7RuZyBwaGnDqm4gbsOgbyDhu58gaGFpIHNwbGl0CiAgICBvbl9kaXNrID0ganNvbi5sb2Fkcygob3V0IC8gInNwbGl0cyIgLyAibWV0YS5qc29uIikucmVhZF90ZXh0KCkpCiAgICBhc3NlcnQgb25fZGlza1siZGF0YXNldF92ZXJzaW9uIl0gPT0gbTFbImRhdGFzZXRfdmVyc2lvbiJdCgoKQHB5dGVzdC5tYXJrLnNraXBpZihub3Qgb3MuZW52aXJvbi5nZXQoIkQxMF9SQVciKSwgcmVhc29uPSLEkeG6t3QgRDEwX1JBVz08dGjGsCBt4bulYyBkYXRhc2V0PiDEkeG7gyBjaOG6oXkiKQpkZWYgdGVzdF9yZWFsX2RhdGFfZGVmYXVsdF9zcGxpdF9pc192YWxpZCgpOgogICAgZ3JvdXBzID0gX2dyb3VwX3N0YXRzKGxvYWRfcmVjb3Jkcyhvcy5lbnZpcm9uWyJEMTBfUkFXIl0pKQogICAgYXNzaWduLCBfID0gYXNzaWduX3NwbGl0cyhncm91cHMpICAgICAgICAjIG7DqW0gUnVudGltZUVycm9yIG7hur91IGtow7RuZyBjaGlhIMSRxrDhu6NjCiAgICBhc3NlcnQgc2V0KGFzc2lnbi52YWx1ZXMoKSkgPT0geyJ0cmFpbiIsICJ2YWwiLCAidGVzdCJ9Cg==", "tests/test_score.py": "aW1wb3J0IGpzb24KCmltcG9ydCBweXRlc3QKCmZyb20gc2NyaXB0cyBpbXBvcnQgc2NvcmUKZnJvbSBzY3JpcHRzLnNjb3JlIGltcG9ydCBldmFsdWF0ZSwgaW91X3h5eHksIHBhcnNlX291dHB1dAoKCmRlZiBfYm94KGxhYmVsLCB4MSwgeTEsIHgyLCB5Mik6CiAgICByZXR1cm4geyJsYWJlbCI6IGxhYmVsLCAiYmJveF8yZCI6IFt4MSwgeTEsIHgyLCB5Ml19CgoKZGVmIHRlc3RfcGFyc2VfcGxhaW5fZmVuY2VkX2FuZF9zdXJyb3VuZGVkKCk6CiAgICBzID0ganNvbi5kdW1wcyhbX2JveCgicG90aG9sZSIsIDEwLCAyMCwgMzAsIDQwKV0pCiAgICB3YW50ID0gW3sibGFiZWwiOiAicG90aG9sZSIsICJiYm94IjogKDEwLjAsIDIwLjAsIDMwLjAsIDQwLjApfV0KICAgIGFzc2VydCBwYXJzZV9vdXRwdXQocykgPT0gd2FudAogICAgYXNzZXJ0IHBhcnNlX291dHB1dChmImBgYGpzb25cbntzfVxuYGBgIikgPT0gd2FudAogICAgYXNzZXJ0IHBhcnNlX291dHB1dChmIkhlcmUgeW91IGdvOiB7c30gSG9wZSB0aGlzIGhlbHBzLiIpID09IHdhbnQKCgpkZWYgdGVzdF9wYXJzZV9lbXB0eV9saXN0X2lzX3ZhbGlkX2FuZF9lbXB0eSgpOgogICAgYXNzZXJ0IHBhcnNlX291dHB1dCgiW10iKSA9PSBbXQoKCkBweXRlc3QubWFyay5wYXJhbWV0cml6ZSgicmF3IiwgWwogICAgIiIsICJub3QganNvbiIsICJ7fSIsICd7ImxhYmVsIjoicG90aG9sZSJ9JywgIlsxLDJdIiwKICAgICdbeyJsYWJlbCI6InBvdGhvbGUifV0nLAogICAgJ1t7ImxhYmVsIjoicG90aG9sZSIsImJib3hfMmQiOlsxLDIsM119XScsCiAgICAnW3sibGFiZWwiOiJwb3Rob2xlIiwiYmJveF8yZCI6WzUsNSw1LDldfV0nLAogICAgJ1t7ImxhYmVsIjoicG90aG9sZSIsImJib3hfMmQiOlsxLDksNSw5XX1dJywKICAgICdbeyJsYWJlbCI6InBvdGhvbGUiLCJiYm94XzJkIjpbIjEiLCIyIiwiMyIsIjQiXX1dJywKICAgICdbeyJsYWJlbCI6InBvdGhvbGUiLCJiYm94XzJkIjpbdHJ1ZSwyLDMsNF19XScsCiAgICAnW3sibGFiZWwiOiJwb3Rob2xlIiwiYmJveF8yZCI6W05hTiwyLDMsNF19XScsCiAgICAnW3sibGFiZWwiOiJwb3Rob2xlIiwiYmJveF8yZCI6WzEsMixJbmZpbml0eSw0XX1dJywKICAgICdbeyJsYWJlbCI6NywiYmJveF8yZCI6WzEsMiwzLDRdfV0nLApdKQpkZWYgdGVzdF9wYXJzZV9yZWplY3RzX21hbGZvcm1lZF93aXRob3V0X3JhaXNpbmcocmF3KToKICAgIGFzc2VydCBwYXJzZV9vdXRwdXQocmF3KSBpcyBOb25lCgoKZGVmIHRlc3RfcGFyc2Vfbm9uZV9pbnB1dF9kb2VzX25vdF9yYWlzZSgpOgogICAgYXNzZXJ0IHBhcnNlX291dHB1dChOb25lKSBpcyBOb25lCgoKZGVmIHRlc3RfaW91KCk6CiAgICBhc3NlcnQgaW91X3h5eHkoKDAsIDAsIDEwLCAxMCksICgwLCAwLCAxMCwgMTApKSA9PSAxLjAKICAgIGFzc2VydCBpb3VfeHl4eSgoMCwgMCwgMTAsIDEwKSwgKDIwLCAyMCwgMzAsIDMwKSkgPT0gMC4wCiAgICBhc3NlcnQgaW91X3h5eHkoKDAsIDAsIDEwLCAxMCksICg1LCAwLCAxNSwgMTApKSA9PSBweXRlc3QuYXBwcm94KDEgLyAzKQoKCmRlZiBfZW50cnkoaW1hZ2VfaWQsIG9ianMsIHc9NjQwLCBoPTM2MCk6CiAgICByZXR1cm4geyJpbWFnZV9pZCI6IGltYWdlX2lkLCAid2lkdGgiOiB3LCAiaGVpZ2h0IjogaCwgIm9iamVjdHMiOiBvYmpzfQoKCmRlZiBfb2JqKGxhYmVsLCB4YywgeWMsIHcsIGgpOgogICAgcmV0dXJuIHsibGFiZWwiOiBsYWJlbCwgInlvbG8iOiBbeGMsIHljLCB3LCBoXX0KCgpHVF9PTkUgPSBbX29iaigicG90aG9sZSIsIDAuNSwgMC41LCAwLjIsIDAuMildICAgICAgICAgICMgMC0xMDAwOiAoNDAwLDQwMCw2MDAsNjAwKQpQRVJGRUNUID0ganNvbi5kdW1wcyhbX2JveCgicG90aG9sZSIsIDQwMCwgNDAwLCA2MDAsIDYwMCldKQoKCmRlZiB0ZXN0X3BlcmZlY3RfcHJlZGljdGlvbigpOgogICAgciA9IGV2YWx1YXRlKFtfZW50cnkoImEiLCBHVF9PTkUpXSwgeyJhIjogUEVSRkVDVH0sIG5fYm9vdD0yMCkKICAgIGFzc2VydCByWyJtaWNybyJdWyJmMSJdID09IDEuMCBhbmQgclsibWVhbl9pb3UiXSA9PSBweXRlc3QuYXBwcm94KDEuMCkKICAgIGFzc2VydCByWyJwYXJzZV92YWxpZF9yYXRlIl0gPT0gMS4wIGFuZCByWyJtaXNzaW5nX3ByZWRpY3Rpb25zIl0gPT0gMAogICAgYXNzZXJ0IHJbInBlcl9jbGFzcyJdWyJwb3Rob2xlIl1bInRwIl0gPT0gMQoKCmRlZiB0ZXN0X3dyb25nX2NsYXNzX2lzX2ZwX2FuZF9mbigpOgogICAgcmF3ID0ganNvbi5kdW1wcyhbX2JveCgiY3JhY2siLCA0MDAsIDQwMCwgNjAwLCA2MDApXSkKICAgIHIgPSBldmFsdWF0ZShbX2VudHJ5KCJhIiwgR1RfT05FKV0sIHsiYSI6IHJhd30sIG5fYm9vdD0yMCkKICAgIGFzc2VydCAoclsibWljcm8iXVsidHAiXSwgclsibWljcm8iXVsiZnAiXSwgclsibWljcm8iXVsiZm4iXSkgPT0gKDAsIDEsIDEpCgoKZGVmIHRlc3RfZHVwbGljYXRlX3ByZWRpY3Rpb25fY291bnRzX29uZV90cF9vbmVfZnAoKToKICAgIHJhdyA9IGpzb24uZHVtcHMoW19ib3goInBvdGhvbGUiLCA0MDAsIDQwMCwgNjAwLCA2MDApXSAqIDIpCiAgICByID0gZXZhbHVhdGUoW19lbnRyeSgiYSIsIEdUX09ORSldLCB7ImEiOiByYXd9LCBuX2Jvb3Q9MjApCiAgICBhc3NlcnQgKHJbIm1pY3JvIl1bInRwIl0sIHJbIm1pY3JvIl1bImZwIl0sIHJbIm1pY3JvIl1bImZuIl0pID09ICgxLCAxLCAwKQoKCmRlZiB0ZXN0X2ludmFsaWRfYW5kX21pc3Npbmdfb3V0cHV0cygpOgogICAgZW50cmllcyA9IFtfZW50cnkoImEiLCBHVF9PTkUpLCBfZW50cnkoImIiLCBHVF9PTkUpLCBfZW50cnkoImMiLCBHVF9PTkUpXQogICAgciA9IGV2YWx1YXRlKGVudHJpZXMsIHsiYSI6IFBFUkZFQ1QsICJiIjogImdhcmJhZ2UifSwgbl9ib290PTIwKQogICAgYXNzZXJ0IHJbInBhcnNlX3ZhbGlkX3JhdGUiXSA9PSBweXRlc3QuYXBwcm94KDEgLyAzKQogICAgYXNzZXJ0IHJbIm1pc3NpbmdfcHJlZGljdGlvbnMiXSA9PSAxCiAgICBhc3NlcnQgclsibWljcm8iXVsicmVjYWxsIl0gPT0gcHl0ZXN0LmFwcHJveCgxIC8gMykKCgpkZWYgdGVzdF9leHRyYV9wcmVkaWN0aW9uX2lkc19pZ25vcmVkKCk6CiAgICByID0gZXZhbHVhdGUoW19lbnRyeSgiYSIsIEdUX09ORSldLCB7ImEiOiBQRVJGRUNULCAienp6IjogUEVSRkVDVH0sIG5fYm9vdD0yMCkKICAgIGFzc2VydCByWyJuX2ltYWdlcyJdID09IDEKCgpkZWYgdGVzdF9waXhlbF92c19ub3JtMTAwMF9zY2FsZV90cmFwKCk6CiAgICByYXcgPSBqc29uLmR1bXBzKFtfYm94KCJwb3Rob2xlIiwgMjU2LCAxNDQsIDM4NCwgMjE2KV0pICAgIyDEkcO6bmcgbuG6v3UgbMOgIHBpeGVsIGPhu6dhIOG6o25oIDY0MHgzNjAKICAgIHB4ID0gZXZhbHVhdGUoW19lbnRyeSgiYSIsIEdUX09ORSldLCB7ImEiOiByYXd9LCBjb29yZD0icGl4ZWwiLCBuX2Jvb3Q9MjApCiAgICBubSA9IGV2YWx1YXRlKFtfZW50cnkoImEiLCBHVF9PTkUpXSwgeyJhIjogcmF3fSwgY29vcmQ9Im5vcm0xMDAwIiwgbl9ib290PTIwKQogICAgYXNzZXJ0IHB4WyJtaWNybyJdWyJmMSJdID09IDEuMAogICAgYXNzZXJ0IG5tWyJtaWNybyJdWyJmMSJdID09IDAuMCBhbmQgbm1bInBhcnNlX3ZhbGlkX3JhdGUiXSA9PSAxLjAKCgpkZWYgdGVzdF91bmtub3duX2xhYmVsX2NvdW50c19hc19mcF9vdGhlcigpOgogICAgcmF3ID0ganNvbi5kdW1wcyhbX2JveCgicG90aG9sZSIsIDQwMCwgNDAwLCA2MDAsIDYwMCksIF9ib3goInB1ZGRsZSIsIDEsIDEsIDUwLCA1MCldKQogICAgciA9IGV2YWx1YXRlKFtfZW50cnkoImEiLCBHVF9PTkUpXSwgeyJhIjogcmF3fSwgbl9ib290PTIwKQogICAgYXNzZXJ0IHJbIm1pY3JvIl1bImZwIl0gPT0gMSBhbmQgclsicGVyX2NsYXNzIl1bIm90aGVyIl1bImZwIl0gPT0gMQoKCmRlZiB0ZXN0X2Jvb3RzdHJhcF9jaV9jb250YWluc19wb2ludF9lc3RpbWF0ZV9hbmRfaXNfc2VlZGVkKCk6CiAgICBlbnRyaWVzID0gW19lbnRyeShzdHIoaSksIEdUX09ORSkgZm9yIGkgaW4gcmFuZ2UoMjApXQogICAgcHJlZHMgPSB7c3RyKGkpOiAoUEVSRkVDVCBpZiBpICUgMiBlbHNlICJbXSIpIGZvciBpIGluIHJhbmdlKDIwKX0KICAgIHIxID0gZXZhbHVhdGUoZW50cmllcywgcHJlZHMsIG5fYm9vdD0yMDAsIHNlZWQ9MCkKICAgIHIyID0gZXZhbHVhdGUoZW50cmllcywgcHJlZHMsIG5fYm9vdD0yMDAsIHNlZWQ9MCkKICAgIGxvLCBoaSA9IHIxWyJmMV9jaTk1Il0KICAgIGFzc2VydCBsbyA8PSByMVsibWljcm8iXVsiZjEiXSA8PSBoaSBhbmQgcjFbImYxX2NpOTUiXSA9PSByMlsiZjFfY2k5NSJdCgoKZGVmIF93cml0ZShwLCByb3dzKToKICAgIHAucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHAud3JpdGVfdGV4dCgiIi5qb2luKGpzb24uZHVtcHMocikgKyAiXG4iIGZvciByIGluIHJvd3MpLCBlbmNvZGluZz0idXRmLTgiKQoKCmRlZiB0ZXN0X2NsaV9lbmRfdG9fZW5kX3dyaXRlc19qc29uX3dpdGhfbWV0YSh0bXBfcGF0aCk6CiAgICBzcCA9IHRtcF9wYXRoIC8gInNwbGl0cyIKICAgIF93cml0ZShzcCAvICJ0ZXN0Lmpzb25sIiwgW19lbnRyeSgiYSIsIEdUX09ORSldKQogICAgKHNwIC8gIm1ldGEuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyh7ImRhdGFzZXRfdmVyc2lvbiI6ICJkMTAtYWJjIn0pKQogICAgcHJlZCA9IHRtcF9wYXRoIC8gInAuanNvbmwiCiAgICBfd3JpdGUocHJlZCwgW3siaW1hZ2VfaWQiOiAiYSIsICJyYXciOiBQRVJGRUNUfV0pCiAgICBvdXQgPSB0bXBfcGF0aCAvICJldiIgLyAiZTEuanNvbiIKICAgIHNjb3JlLm1haW4oWyItLXByZWQiLCBzdHIocHJlZCksICItLXNwbGl0cy1kaXIiLCBzdHIoc3ApLCAiLS1uYW1lIiwgIkUxIiwKICAgICAgICAgICAgICAgICItLW91dCIsIHN0cihvdXQpLCAiLS1tZXRhIiwgInRyYWluYWJsZV9wYXJhbXM9MTAwMCIsCiAgICAgICAgICAgICAgICAiLS1tZXRhIiwgInBlYWtfdnJhbV9nYj03LjUiLCAiLS1tZXRhIiwgIm5vdGU9aGVsbG8iXSkKICAgIGQgPSBqc29uLmxvYWRzKG91dC5yZWFkX3RleHQoKSkKICAgIGFzc2VydCBkWyJuYW1lIl0gPT0gIkUxIiBhbmQgZFsiZGF0YXNldF92ZXJzaW9uIl0gPT0gImQxMC1hYmMiCiAgICBhc3NlcnQgZFsic2NvcmVfdmVyc2lvbiJdID09IDEgYW5kIGRbInNwbGl0Il0gPT0gInRlc3QiCiAgICBhc3NlcnQgZFsibWV0YSJdID09IHsidHJhaW5hYmxlX3BhcmFtcyI6IDEwMDAsICJwZWFrX3ZyYW1fZ2IiOiA3LjUsICJub3RlIjogImhlbGxvIn0KCgpkZWYgdGVzdF9jbGlfZHVwbGljYXRlX2ltYWdlX2lkX2lzX2xvdWQodG1wX3BhdGgpOgogICAgc3AgPSB0bXBfcGF0aCAvICJzcGxpdHMiCiAgICBfd3JpdGUoc3AgLyAidGVzdC5qc29ubCIsIFtfZW50cnkoImEiLCBHVF9PTkUpXSkKICAgIChzcCAvICJtZXRhLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoeyJkYXRhc2V0X3ZlcnNpb24iOiAiZCJ9KSkKICAgIHByZWQgPSB0bXBfcGF0aCAvICJwLmpzb25sIgogICAgX3dyaXRlKHByZWQsIFt7ImltYWdlX2lkIjogImEiLCAicmF3IjogIltdIn0sIHsiaW1hZ2VfaWQiOiAiYSIsICJyYXciOiAiW10ifV0pCiAgICB3aXRoIHB5dGVzdC5yYWlzZXMoU3lzdGVtRXhpdCkgYXMgZToKICAgICAgICBzY29yZS5tYWluKFsiLS1wcmVkIiwgc3RyKHByZWQpLCAiLS1zcGxpdHMtZGlyIiwgc3RyKHNwKSwgIi0tbmFtZSIsICJFMSIsCiAgICAgICAgICAgICAgICAgICAgIi0tb3V0Iiwgc3RyKHRtcF9wYXRoIC8gIm8uanNvbiIpXSkKICAgIGFzc2VydCAiZHVwbGljYXRlIiBpbiBzdHIoZS52YWx1ZSkK", "tests/test_splitting.py": "aW1wb3J0IHB5dGVzdAoKZnJvbSBzY3JpcHRzLmxhYmNvbW1vbiBpbXBvcnQgQ0xBU1NFUywgcmVhZF9qc29ubCwgd3JpdGVfanNvbmwKZnJvbSBzY3JpcHRzLnNwbGl0dGluZyBpbXBvcnQgU1BMSVRTLCBhc3NpZ25fc3BsaXRzLCBkYXRhc2V0X3ZlcnNpb24sIHNlc3Npb25fa2V5CgoKZGVmIHRlc3RfanNvbmxfcm91bmR0cmlwKHRtcF9wYXRoKToKICAgIHJvd3MgPSBbeyJpZCI6ICJhIiwgInQiOiAixJHGsOG7nW5nIn0sIHsiaWQiOiAiYiIsICJuIjogWzEsIDJdfV0KICAgIHAgPSB0bXBfcGF0aCAvICJ4IiAvICJ5Lmpzb25sIgogICAgd3JpdGVfanNvbmwocCwgcm93cykKICAgIGFzc2VydCByZWFkX2pzb25sKHApID09IHJvd3MKICAgIGFzc2VydCBDTEFTU0VTID09ICgicG90aG9sZSIsICJjcmFjayIsICJtYW5ob2xlIikKCgpAcHl0ZXN0Lm1hcmsucGFyYW1ldHJpemUoIm5hbWUsZXhwZWN0ZWQiLCBbCiAgICAoInZsY3NuYXAtMjAyNS0wMi0xOC0yM2gwOG0xNHM1MjguanBnIiwgInZsYy0yMDI1LTAyLTE4LTIzIiksCiAgICAoInZsY3NuYXBfMjAyNS0wMy0xNi0xNWgxNm01OXM4MjQuanBnIiwgInZsYy0yMDI1LTAzLTE2LTE1IiksCiAgICAoIjIwMjUwMjE5XzE2NDY0OS5qcGciLCAicGhvbmUtMjAyNTAyMTkiKSwKICAgICgidmxjc25hcC0wMDA1MC5qcGciLCAibnVtLTEiKSwKICAgICgidmxjc25hcC0wMDA0OS5qcGciLCAibnVtLTAiKSwKICAgICgidmxjc25hcC0wMDA5Ny1NU0kuanBnIiwgIm51bS0xIiksCiAgICAoInZsY3NuYXAtMDAyMTQuanBnIiwgIm51bS00IiksCl0pCmRlZiB0ZXN0X3Nlc3Npb25fa2V5X2tub3duX3BhdHRlcm5zKG5hbWUsIGV4cGVjdGVkKToKICAgIGFzc2VydCBzZXNzaW9uX2tleShuYW1lKSA9PSBleHBlY3RlZAoKCmRlZiB0ZXN0X3Nlc3Npb25fa2V5X3Vua25vd25fbmFtZV9pc19sb3VkKCk6CiAgICB3aXRoIHB5dGVzdC5yYWlzZXMoVmFsdWVFcnJvciwgbWF0Y2g9IklNR18wMDEuanBnIik6CiAgICAgICAgc2Vzc2lvbl9rZXkoIklNR18wMDEuanBnIikKCgpkZWYgX3JlYWxfbGlrZV9ncm91cHMoKToKICAgIHNpemVzID0gewogICAgICAgICJ2bGMtMjAyNS0wMi0xOC0xNyI6IDEyNCwgInZsYy0yMDI1LTAyLTE4LTE4IjogMjQ0LCAidmxjLTIwMjUtMDItMTgtMjMiOiAyMTMsCiAgICAgICAgInZsYy0yMDI1LTAyLTE5LTEzIjogNTUsICJ2bGMtMjAyNS0wMi0xOS0xNCI6IDQ3NSwgInZsYy0yMDI1LTAyLTE5LTE1IjogMTE2LAogICAgICAgICJ2bGMtMjAyNS0wMi0xOS0xNiI6IDMyLCAidmxjLTIwMjUtMDItMTktMTciOiA4MywgInZsYy0yMDI1LTAyLTI2LTIwIjogMjE3LAogICAgICAgICJ2bGMtMjAyNS0wMy0xNi0xNSI6IDEzNiwgInBob25lLTIwMjUwMjE2IjogMywgInBob25lLTIwMjUwMjE5IjogNjAsCiAgICAgICAgInBob25lLTIwMjUwMjIzIjogMzksICJudW0tMCI6IDQ5LCAibnVtLTEiOiA1MSwgIm51bS0yIjogNTAsICJudW0tMyI6IDUwLCAibnVtLTQiOiAxMiwKICAgIH0KICAgIG1peCA9IHsiY3JhY2siOiAwLjUzLCAicG90aG9sZSI6IDAuMjcsICJtYW5ob2xlIjogMC4yMH0KICAgIHJldHVybiB7azogeyJpbWFnZXMiOiBuLCAiYm94ZXMiOiB7Yzogcm91bmQobiAqIDIuMzYgKiBmKSBmb3IgYywgZiBpbiBtaXguaXRlbXMoKX19CiAgICAgICAgICAgIGZvciBrLCBuIGluIHNpemVzLml0ZW1zKCl9CgoKZGVmIHRlc3RfYXNzaWduX2lzX2RldGVybWluaXN0aWNfYW5kX2NvdmVyc19hbGxfZ3JvdXBzKCk6CiAgICBnID0gX3JlYWxfbGlrZV9ncm91cHMoKQogICAgYXNzZXJ0IGFzc2lnbl9zcGxpdHMoZykgPT0gYXNzaWduX3NwbGl0cyhnKQogICAgYSwgXyA9IGFzc2lnbl9zcGxpdHMoZykKICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGcpIGFuZCBzZXQoYS52YWx1ZXMoKSkgPT0gc2V0KFNQTElUUykKCgpkZWYgdGVzdF9hc3NpZ25fdmFsaWRfZm9yX21hbnlfc2VlZHMoKToKICAgIGcgPSBfcmVhbF9saWtlX2dyb3VwcygpCiAgICB0b3RhbCA9IHN1bSh2WyJpbWFnZXMiXSBmb3IgdiBpbiBnLnZhbHVlcygpKQogICAgZm9yIGkgaW4gcmFuZ2UoMTAwKToKICAgICAgICBhLCBfID0gYXNzaWduX3NwbGl0cyhnLCBzZWVkPWYic3tpfSIpCiAgICAgICAgZm9yIHNwbGl0IGluIFNQTElUUzoKICAgICAgICAgICAgYXNzZXJ0IHN1bShnW2tdWyJpbWFnZXMiXSBmb3IgaywgcyBpbiBhLml0ZW1zKCkgaWYgcyA9PSBzcGxpdCkgPj0gMC4xMCAqIHRvdGFsCiAgICAgICAgZm9yIHNwbGl0IGluICgidmFsIiwgInRlc3QiKToKICAgICAgICAgICAgZm9yIGMgaW4gQ0xBU1NFUzoKICAgICAgICAgICAgICAgIGFzc2VydCBzdW0oZ1trXVsiYm94ZXMiXVtjXSBmb3IgaywgcyBpbiBhLml0ZW1zKCkgaWYgcyA9PSBzcGxpdCkgPj0gMjAKCgpkZWYgdGVzdF9hc3NpZ25fcmFpc2VzX3doZW5faW1wb3NzaWJsZSgpOgogICAgZyA9IHsib25seSI6IHsiaW1hZ2VzIjogMTAsICJib3hlcyI6IHsicG90aG9sZSI6IDEsICJjcmFjayI6IDEsICJtYW5ob2xlIjogMX19fQogICAgd2l0aCBweXRlc3QucmFpc2VzKFJ1bnRpbWVFcnJvcik6CiAgICAgICAgYXNzaWduX3NwbGl0cyhnLCBtYXhfc2FsdD01KQoKCmRlZiB0ZXN0X2RhdGFzZXRfdmVyc2lvbl9zdGFibGVfYW5kX3NlbnNpdGl2ZSgpOgogICAgcm93cyA9IFsoImEiLCAidHJhaW4iKSwgKCJiIiwgInRlc3QiKV0KICAgIHYgPSBkYXRhc2V0X3ZlcnNpb24ocm93cykKICAgIGFzc2VydCB2LnN0YXJ0c3dpdGgoImQxMC0iKSBhbmQgbGVuKHYpID09IDE0CiAgICBhc3NlcnQgdiA9PSBkYXRhc2V0X3ZlcnNpb24obGlzdChyZXZlcnNlZChyb3dzKSkpCiAgICBhc3NlcnQgdiAhPSBkYXRhc2V0X3ZlcnNpb24oWygiYSIsICJ0ZXN0IiksICgiYiIsICJ0cmFpbiIpXSkK", "configs/e0.yaml": "bmFtZTogRTAKZGVzY3JpcHRpb246IFplcm8tc2hvdCwga2jDtG5nIGh14bqlbiBsdXnhu4duLiBDaOG7iSBzdXkgbHXhuq1uIGLhurFuZyBtb2RlbCBn4buRYyBy4buTaSBjaOG6pW0uCm1vZGVsX2lkOiBRd2VuL1F3ZW4zLVZMLTRCLUluc3RydWN0CnF1YW50aXphdGlvbjogbmY0LTRiaXQgICAgICAjIFZpVCB2w6AgbG1faGVhZCBLSMOUTkcgbMaw4bujbmcgdOG7rSBob8OhCm1heF9uZXdfdG9rZW5zOiAyNTYK", "configs/e1.yaml": "bmFtZTogRTEKZGVzY3JpcHRpb246IExvUkEgY2jhu4kgdHLDqm4gTExNOyDEkcOzbmcgYsSDbmcgVmlUIHbDoCBwcm9qZWN0b3IuCm1vZGVsX2lkOiBRd2VuL1F3ZW4zLVZMLTRCLUluc3RydWN0CnF1YW50aXphdGlvbjogbmY0LTRiaXQgICAgICAjIFZpVCB2w6AgbG1faGVhZCBLSMOUTkcgbMaw4bujbmcgdOG7rSBob8OhCmxvcmE6IHtyOiAxNiwgYWxwaGE6IDMyLCBkcm9wb3V0OiAwLjA1LCB0YXJnZXRzOiBsbG19CnRyYWluX3Byb2plY3RvcjogZmFsc2UKbHI6IDIuMGUtNApiYXRjaF9zaXplOiAyCmdyYWRfYWNjdW06IDQKbWF4X3N0ZXBzOiAzMDAKc2VlZDogNDIK", "configs/e2.yaml": "bmFtZTogRTIKZGVzY3JpcHRpb246IEUxICsgbeG7nyDEkeG6p3kgxJHhu6cgcHJvamVjdG9yICh2aXN1YWwubWVyZ2VyIHbDoCB2aXN1YWwuZGVlcHN0YWNrX21lcmdlcl9saXN0KS4KbW9kZWxfaWQ6IFF3ZW4vUXdlbjMtVkwtNEItSW5zdHJ1Y3QKcXVhbnRpemF0aW9uOiBuZjQtNGJpdCAgICAgICMgVmlUIHbDoCBsbV9oZWFkIEtIw5RORyBsxrDhu6NuZyB04butIGhvw6EKbG9yYToge3I6IDE2LCBhbHBoYTogMzIsIGRyb3BvdXQ6IDAuMDUsIHRhcmdldHM6IGxsbX0KdHJhaW5fcHJvamVjdG9yOiB0cnVlCmxyOiAyLjBlLTQKYmF0Y2hfc2l6ZTogMgpncmFkX2FjY3VtOiA0Cm1heF9zdGVwczogMzAwCnNlZWQ6IDQyCg==", "configs/e3.yaml": "bmFtZTogRTMKZGVzY3JpcHRpb246IEUyICsgTG9SQSB0csOqbiBjw6FjIGto4buRaSBhdHRlbnRpb24gdsOgIE1MUCBj4bunYSBWaVQuCm1vZGVsX2lkOiBRd2VuL1F3ZW4zLVZMLTRCLUluc3RydWN0CnF1YW50aXphdGlvbjogbmY0LTRiaXQgICAgICAjIFZpVCB2w6AgbG1faGVhZCBLSMOUTkcgbMaw4bujbmcgdOG7rSBob8OhCmxvcmE6IHtyOiAxNiwgYWxwaGE6IDMyLCBkcm9wb3V0OiAwLjA1LCB0YXJnZXRzOiBsbG0rdml0fQp0cmFpbl9wcm9qZWN0b3I6IHRydWUKbHI6IDIuMGUtNApiYXRjaF9zaXplOiAyCmdyYWRfYWNjdW06IDQKbWF4X3N0ZXBzOiAzMDAKc2VlZDogNDIK", "configs/e4.yaml": "bmFtZTogRTQKZGVzY3JpcHRpb246IEFibGF0aW9uIHThu7EgY2jhu41uLiBDaOG7jW4gxJDDmk5HIE3hu5hUIGJp4bq/biDEkeG7gyDEkeG7lWksIGdp4buvIG3hu41pIHRo4bupIGtow6FjIG5oxrAgc2V0dXAgdOG7kXQgbmjhuqV0IHRyb25nIEUx4oCTRTMuCm9wdGlvbnM6CiAgcmFuazoge3ZhbHVlczogWzQsIDY0XSwgbm90ZTogIsSR4buVaSBsb3JhLnIgKMSR4bq3dCBhbHBoYSA9IDIqcikifQogIGRhdGFfc2l6ZToge3ZhbHVlczogWzAuMSwgMS4wXSwgbm90ZTogInRyYWluX2ZyYWN0aW9uOyAwLjEgPSAxMCUgc+G7kSDhuqNuaCB0cmFpbiJ9CiAgY29vcmRfc2NhbGU6IHt2YWx1ZXM6IFtub3JtMTAwMCwgcGl4ZWxdLCBub3RlOiAixJHhu5VpIHRoYW5nIHRv4bqhIMSR4buZIHRyb25nIGPhuqMgbmjDo24gbOG6q24gcHJvbXB0OyBjaOG6pW0gYuG6sW5nIC0tY29vcmQgdMawxqFuZyDhu6luZyJ9Cg==", "configs/e5.yaml": "bmFtZTogRTUKZGVzY3JpcHRpb246IChUdeG7syBjaOG7jW4sICvEkWnhu4NtKSBHUlBPIHbhu5tpIHJld2FyZCBJb1UsIGto4bufaSB04bqhbyB04burIGFkYXB0ZXIgdOG7kXQgbmjhuqV0IGPhu6dhIEUx4oCTRTMuCmdycG86CiAgbnVtX2dlbmVyYXRpb25zOiA0CiAgbWF4X2NvbXBsZXRpb25fbGVuZ3RoOiAxMjgKICBtYXhfc3RlcHM6IDYwCiAgbGVhcm5pbmdfcmF0ZTogNS4wZS02CiAgYmV0YTogMC4wCnJld2FyZDogIjAuMiAqIHZhbGlkX2pzb24gKyAwLjggKiAodOG7lW5nIElvVSBjw6FjIGPhurdwIGto4bubcCkgLyBtYXgobl9ndCwgbl9wcmVkLCAxKSIK", "student/__init__.py": "IiIiU3R1ZGVudCBpbXBsZW1lbnRhdGlvbjsgaW5zdHJ1Y3RvciBzY3JpcHRzIGFuZCB0ZXN0cyByZW1haW4gdW5jaGFuZ2VkLiIiIgo=", "student/analyze.py": "...", "student/data.py": "aW1wb3J0IGpzb24KaW1wb3J0IHJhbmRvbQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCB0b3JjaApmcm9tIFBJTCBpbXBvcnQgSW1hZ2UKClBST01QVCA9ICgnRGV0ZWN0IGFsbCB2aXNpYmxlIHJvYWQgZGFtYWdlIGluIHRoaXMgaW1hZ2UuIFVzZSBvbmx5IHRoZSBsYWJlbHMgInBvdGhvbGUiLCAnCiAgICAgICAgICAnImNyYWNrIiwgYW5kICJtYW5ob2xlIi4gUmV0dXJuIG9ubHkgYSBKU09OIGxpc3QsIHdpdGggbm8gZXhwbGFuYXRpb24gb3IgbWFya2Rvd24uICcKICAgICAgICAgICdFYWNoIG9iamVjdCBtdXN0IGhhdmUgImxhYmVsIiBhbmQgImJib3hfMmQiOiBbeDEseTEseDIseTJdLiAnCiAgICAgICAgICAnQ29vcmRpbmF0ZXMgbXVzdCBiZSBpbnRlZ2VycyBub3JtYWxpemVkIHRvIDAtMTAwMCByZWxhdGl2ZSB0byBpbWFnZSB3aWR0aCBhbmQgaGVpZ2h0LiAnCiAgICAgICAgICAnT3JkZXIgb2JqZWN0cyBmcm9tIHRvcCB0byBib3R0b20sIHRoZW4gbGVmdCB0byByaWdodC4gUmV0dXJuIFtdIGlmIG5vbmUgYXJlIHZpc2libGUuJykKCgpkZWYgb2JqZWN0c190b19hbnN3ZXIob2JqZWN0cyk6CiAgICByZXN1bHQgPSBbXQogICAgZm9yIG9iaiBpbiBvYmplY3RzOgogICAgICAgIHhjLCB5YywgdywgaCA9IG9ialsieW9sbyJdCiAgICAgICAgaWYgdyA8PSAwIG9yIGggPD0gMDoKICAgICAgICAgICAgIyBTb3VyY2UgYW5ub3RhdGlvbiBub2lzZToga2VlcCBvcmlnaW5hbCBHVCBmb3IgZXZhbHVhdGlvbiwgb21pdCBpbXBvc3NpYmxlIFNGVCB0YXJnZXRzLgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGJveCA9IFttYXgoMCwgbWluKDEwMDAsIHJvdW5kKHYgKiAxMDAwKSkpIGZvciB2IGluCiAgICAgICAgICAgICAgICh4YyAtIHcgLyAyLCB5YyAtIGggLyAyLCB4YyArIHcgLyAyLCB5YyArIGggLyAyKV0KICAgICAgICBpZiBib3hbMl0gPD0gYm94WzBdOgogICAgICAgICAgICBib3hbMF0gPSBtaW4oYm94WzBdLCA5OTkpCiAgICAgICAgICAgIGJveFsyXSA9IGJveFswXSArIDEKICAgICAgICBpZiBib3hbM10gPD0gYm94WzFdOgogICAgICAgICAgICBib3hbMV0gPSBtaW4oYm94WzFdLCA5OTkpCiAgICAgICAgICAgIGJveFszXSA9IGJveFsxXSArIDEKICAgICAgICByZXN1bHQuYXBwZW5kKHsibGFiZWwiOiBvYmpbImxhYmVsIl0sICJiYm94XzJkIjogYm94fSkKICAgIHJlc3VsdC5zb3J0KGtleT1sYW1iZGEgeDogKHhbImJib3hfMmQiXVsxXSwgeFsiYmJveF8yZCJdWzBdKSkKICAgIHJldHVybiBqc29uLmR1bXBzKHJlc3VsdCwgc2VwYXJhdG9ycz0oIiwiLCAiOiIpKQoKCmRlZiByZWFkX3Jvd3MocGF0aCk6CiAgICByZXR1cm4gW2pzb24ubG9hZHMobGluZSkgZm9yIGxpbmUgaW4gUGF0aChwYXRoKS5yZWFkX3RleHQoKS5zcGxpdGxpbmVzKCkgaWYgbGluZS5zdHJpcCgpXQoKCmRlZiBpbWFnZV9mb3Iocm93LCBkYXRhX2Rpcik6CiAgICB3aXRoIEltYWdlLm9wZW4oUGF0aChkYXRhX2RpcikgLyByb3dbImltYWdlX3BhdGgiXSkgYXMgaW1hZ2U6CiAgICAgICAgcmV0dXJuIGltYWdlLmNvbnZlcnQoIlJHQiIpCgoKZGVmIHVzZXJfbWVzc2FnZShpbWFnZSk6CiAgICByZXR1cm4geyJyb2xlIjogInVzZXIiLCAiY29udGVudCI6IFt7InR5cGUiOiAiaW1hZ2UiLCAiaW1hZ2UiOiBpbWFnZX0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHsidHlwZSI6ICJ0ZXh0IiwgInRleHQiOiBQUk9NUFR9XX0KCgpkZWYgZW5jb2RlX2V4YW1wbGUocm93LCBkYXRhX2RpciwgcHJvY2Vzc29yKToKICAgIG1lc3NhZ2UgPSB1c2VyX21lc3NhZ2UoaW1hZ2VfZm9yKHJvdywgZGF0YV9kaXIpKQogICAgYW5zd2VyID0gb2JqZWN0c190b19hbnN3ZXIocm93WyJvYmplY3RzIl0pCiAgICBwcm9tcHQgPSBwcm9jZXNzb3IuYXBwbHlfY2hhdF90ZW1wbGF0ZShbbWVzc2FnZV0sIHRva2VuaXplPVRydWUsIGFkZF9nZW5lcmF0aW9uX3Byb21wdD1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcmV0dXJuX2RpY3Q9VHJ1ZSwgcmV0dXJuX3RlbnNvcnM9InB0IikKICAgIGZ1bGwgPSBwcm9jZXNzb3IuYXBwbHlfY2hhdF90ZW1wbGF0ZShbbWVzc2FnZSwgeyJyb2xlIjogImFzc2lzdGFudCIsICJjb250ZW50IjogYW5zd2VyfV0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdG9rZW5pemU9VHJ1ZSwgYWRkX2dlbmVyYXRpb25fcHJvbXB0PUZhbHNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJldHVybl9kaWN0PVRydWUsIHJldHVybl90ZW5zb3JzPSJwdCIpCiAgICBsZW5ndGggPSBwcm9tcHRbImlucHV0X2lkcyJdLnNoYXBlWzFdCiAgICBhc3NlcnQgdG9yY2guZXF1YWwoZnVsbFsiaW5wdXRfaWRzIl1bOiwgOmxlbmd0aF0sIHByb21wdFsiaW5wdXRfaWRzIl0pLCAiUHJvbXB0IG5vdCBwcmVmaXgiCiAgICBsYWJlbHMgPSBmdWxsWyJpbnB1dF9pZHMiXS5jbG9uZSgpCiAgICBsYWJlbHNbOiwgOmxlbmd0aF0gPSAtMTAwCiAgICBmdWxsWyJsYWJlbHMiXSA9IGxhYmVscwogICAgYXNzZXJ0IChsYWJlbHMgIT0gLTEwMCkuYW55KCkKICAgIGFzc2VydCAibW1fdG9rZW5fdHlwZV9pZHMiIGluIGZ1bGwsICJNLVJvUEUgbmVlZHMgbXVsdGltb2RhbCB0b2tlbiB0eXBlcyIKICAgIHJldHVybiBkaWN0KGZ1bGwpCgoKZGVmIGNvbGxhdGUoZXhhbXBsZXMsIHBhZF9pZCk6CiAgICByZXN1bHQgPSB7fQogICAgbGVuZ3RoID0gbWF4KHhbImlucHV0X2lkcyJdLnNoYXBlWzFdIGZvciB4IGluIGV4YW1wbGVzKQogICAgc2VxdWVuY2Vfa2V5cyA9IHsiaW5wdXRfaWRzIiwgImF0dGVudGlvbl9tYXNrIiwgIm1tX3Rva2VuX3R5cGVfaWRzIiwgImxhYmVscyJ9CiAgICBmb3Iga2V5IGluIGV4YW1wbGVzWzBdOgogICAgICAgIGlmIGtleSBpbiBzZXF1ZW5jZV9rZXlzOgogICAgICAgICAgICBmaWxsID0gLTEwMCBpZiBrZXkgPT0gImxhYmVscyIgZWxzZSBwYWRfaWQgaWYga2V5ID09ICJpbnB1dF9pZHMiIGVsc2UgMAogICAgICAgICAgICByZXN1bHRba2V5XSA9IHRvcmNoLmNhdChbdG9yY2gubm4uZnVuY3Rpb25hbC5wYWQoeFtrZXldLCAoMCwgbGVuZ3RoIC0geFtrZXldLnNoYXBlWzFdKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdmFsdWU9ZmlsbCkgZm9yIHggaW4gZXhhbXBsZXNdKQogICAgICAgIGVsaWYga2V5IGluIHsicGl4ZWxfdmFsdWVzIiwgImltYWdlX2dyaWRfdGh3In06CiAgICAgICAgICAgIHJlc3VsdFtrZXldID0gdG9yY2guY2F0KFt4W2tleV0gZm9yIHggaW4gZXhhbXBsZXNdLCBkaW09MCkKICAgICAgICBlbHNlOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5oYW5kbGVkIHByb2Nlc3NvciBrZXkge2tleX0iKQogICAgcmV0dXJuIHJlc3VsdAoKCmNsYXNzIEVuY29kZWREYXRhOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHJvd3MsIGRhdGFfZGlyLCBwcm9jZXNzb3IpOgogICAgICAgIHNlbGYucm93cywgc2VsZi5kYXRhX2Rpciwgc2VsZi5wcm9jZXNzb3IgPSByb3dzLCBkYXRhX2RpciwgcHJvY2Vzc29yCiAgICAgICAgc2VsZi5jYWNoZSA9IHt9CgogICAgZGVmIGdldChzZWxmLCBpbmRleCk6CiAgICAgICAgaWYgaW5kZXggbm90IGluIHNlbGYuY2FjaGU6CiAgICAgICAgICAgIHNlbGYuY2FjaGVbaW5kZXhdID0gZW5jb2RlX2V4YW1wbGUoc2VsZi5yb3dzW2luZGV4XSwgc2VsZi5kYXRhX2Rpciwgc2VsZi5wcm9jZXNzb3IpCiAgICAgICAgcmV0dXJuIHNlbGYuY2FjaGVbaW5kZXhdCgoKZGVmIHN1YnNldChyb3dzLCBmcmFjdGlvbiwgc2VlZD00Mik6CiAgICBpbmRpY2VzID0gbGlzdChyYW5nZShsZW4ocm93cykpKQogICAgcmFuZG9tLlJhbmRvbShzZWVkKS5zaHVmZmxlKGluZGljZXMpCiAgICByZXR1cm4gW3Jvd3NbaV0gZm9yIGkgaW4gaW5kaWNlc1s6bWF4KDEsIHJvdW5kKGxlbihyb3dzKSAqIGZyYWN0aW9uKSldXQo=", "student/infer.py": "aW1wb3J0IGpzb24KaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgdG9yY2gKCmZyb20gc3R1ZGVudC5kYXRhIGltcG9ydCBpbWFnZV9mb3IsIHVzZXJfbWVzc2FnZQoKCkB0b3JjaC5pbmZlcmVuY2VfbW9kZSgpCmRlZiBwcmVkaWN0KG1vZGVsLCBwcm9jZXNzb3IsIHJvd3MsIGRhdGFfZGlyLCBvdXQsIGJhdGNoX3NpemU9NCwgbWF4X25ld190b2tlbnM9NTEyKToKICAgIG91dCA9IFBhdGgob3V0KQogICAgb3V0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBwcm9jZXNzb3IudG9rZW5pemVyLnBhZGRpbmdfc2lkZSA9ICJsZWZ0IgogICAgbW9kZWwuZXZhbCgpCiAgICBtb2RlbC5jb25maWcudXNlX2NhY2hlID0gVHJ1ZQogICAgc3RhcnQgPSB0aW1lLm1vbm90b25pYygpCiAgICBvdXRwdXRzID0gW10KICAgIHdpdGggb3V0Lm9wZW4oIngiLCBidWZmZXJpbmc9MSkgYXMgc3RyZWFtOgogICAgICAgIGZvciBvZmZzZXQgaW4gcmFuZ2UoMCwgbGVuKHJvd3MpLCBiYXRjaF9zaXplKToKICAgICAgICAgICAgYmF0Y2ggPSByb3dzW29mZnNldDpvZmZzZXQgKyBiYXRjaF9zaXplXQogICAgICAgICAgICBtZXNzYWdlcyA9IFtbdXNlcl9tZXNzYWdlKGltYWdlX2Zvcihyb3csIGRhdGFfZGlyKSldIGZvciByb3cgaW4gYmF0Y2hdCiAgICAgICAgICAgIGlucHV0cyA9IHByb2Nlc3Nvci5hcHBseV9jaGF0X3RlbXBsYXRlKG1lc3NhZ2VzLCB0b2tlbml6ZT1UcnVlLCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgIHJldHVybl9kaWN0PVRydWUsIHJldHVybl90ZW5zb3JzPSJwdCIsIHByb2Nlc3Nvcl9rd2FyZ3M9eyJwYWRkaW5nIjogVHJ1ZX0pLnRvKCJjdWRhIikKICAgICAgICAgICAgd2l0aCB0b3JjaC5hdXRvY2FzdCgiY3VkYSIsIGR0eXBlPXRvcmNoLmZsb2F0MTYpOgogICAgICAgICAgICAgICAgZ2VuZXJhdGVkID0gbW9kZWwuZ2VuZXJhdGUoKippbnB1dHMsIG1heF9uZXdfdG9rZW5zPW1heF9uZXdfdG9rZW5zLCBkb19zYW1wbGU9RmFsc2UsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB1c2VfY2FjaGU9VHJ1ZSkKICAgICAgICAgICAgcmF3ID0gcHJvY2Vzc29yLmJhdGNoX2RlY29kZShnZW5lcmF0ZWRbOiwgaW5wdXRzLmlucHV0X2lkcy5zaGFwZVsxXTpdLCBza2lwX3NwZWNpYWxfdG9rZW5zPVRydWUpCiAgICAgICAgICAgIGZvciByb3csIHRleHQgaW4gemlwKGJhdGNoLCByYXcpOgogICAgICAgICAgICAgICAgaXRlbSA9IHsiaW1hZ2VfaWQiOiByb3dbImltYWdlX2lkIl0sICJyYXciOiB0ZXh0fQogICAgICAgICAgICAgICAgc3RyZWFtLndyaXRlKGpzb24uZHVtcHMoaXRlbSwgZW5zdXJlX2FzY2lpPUZhbHNlKSArICJcbiIpCiAgICAgICAgICAgICAgICBvdXRwdXRzLmFwcGVuZChpdGVtKQogICAgICAgICAgICBpZiBvZmZzZXQgJSA0MCA9PSAwIG9yIG9mZnNldCArIGxlbihiYXRjaCkgPT0gbGVuKHJvd3MpOgogICAgICAgICAgICAgICAgcHJpbnQoIlBSRURJQ1QiLCBvdXQubmFtZSwgb2Zmc2V0ICsgbGVuKGJhdGNoKSwgIi8iLCBsZW4ocm93cyksCiAgICAgICAgICAgICAgICAgICAgICByb3VuZCh0aW1lLm1vbm90b25pYygpIC0gc3RhcnQsIDEpLCAic2Vjb25kcyIsIGZsdXNoPVRydWUpCiAgICAgICAgICAgIGRlbCBpbnB1dHMsIGdlbmVyYXRlZAogICAgcmV0dXJuIG91dHB1dHMK", "student/model.py": "aW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IHJlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IGJpdHNhbmRieXRlcyBhcyBibmIKaW1wb3J0IHRvcmNoCmZyb20gcGVmdCBpbXBvcnQgTG9yYUNvbmZpZywgUGVmdE1vZGVsLCBnZXRfcGVmdF9tb2RlbApmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b1Byb2Nlc3NvciwgQml0c0FuZEJ5dGVzQ29uZmlnLCBRd2VuM1ZMRm9yQ29uZGl0aW9uYWxHZW5lcmF0aW9uCgpNT0RFTF9JRCA9ICJRd2VuL1F3ZW4zLVZMLTRCLUluc3RydWN0IgpNT0RFTF9SRVZJU0lPTiA9ICJlYmIyODFlYzcwYjA1MDkwYWE2MTY1YjAxNmVhYzhlYzA4ZTcxYjE3IgpMTE1fVEFSR0VUID0gciJtb2RlbFwubGFuZ3VhZ2VfbW9kZWxcLmxheWVyc1wuXGQrXC4oPzpzZWxmX2F0dG5cLig/OnF8a3x2fG8pX3Byb2p8bWxwXC4oPzpnYXRlfHVwfGRvd24pX3Byb2opIgpWSVRfVEFSR0VUID0gciJtb2RlbFwudmlzdWFsXC5ibG9ja3NcLlxkK1wuKD86YXR0blwuKD86cWt2fHByb2opfG1scFwuKD86bGluZWFyX2ZjMXxsaW5lYXJfZmMyKSkiCgoKZGVmIHByb2plY3Rvcl9wYXJhbWV0ZXIobmFtZSk6CiAgICByZXR1cm4gInZpc3VhbC5tZXJnZXIuIiBpbiBuYW1lIG9yICJ2aXN1YWwuZGVlcHN0YWNrX21lcmdlcl9saXN0LiIgaW4gbmFtZQoKCmRlZiBsb2FkX2Jhc2UoKToKICAgIG1vZGVsID0gUXdlbjNWTEZvckNvbmRpdGlvbmFsR2VuZXJhdGlvbi5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgTU9ERUxfSUQsIHJldmlzaW9uPU1PREVMX1JFVklTSU9OLCBkdHlwZT10b3JjaC5mbG9hdDE2LCBkZXZpY2VfbWFwPXsiIjogMH0sIGF0dG5faW1wbGVtZW50YXRpb249InNkcGEiLAogICAgICAgIHF1YW50aXphdGlvbl9jb25maWc9Qml0c0FuZEJ5dGVzQ29uZmlnKGxvYWRfaW5fNGJpdD1UcnVlLCBibmJfNGJpdF9xdWFudF90eXBlPSJuZjQiLAogICAgICAgICAgICBibmJfNGJpdF9jb21wdXRlX2R0eXBlPXRvcmNoLmZsb2F0MTYsIGJuYl80Yml0X3VzZV9kb3VibGVfcXVhbnQ9VHJ1ZSwKICAgICAgICAgICAgIyBUcmFuc2Zvcm1lcnMgNS4xOSBtYXRjaGVzIGEgZnVsbC1uYW1lIHByZWZpeCwgbm90IGFuIGludGVyaW9yIHBhdGggY29tcG9uZW50LgogICAgICAgICAgICBsbG1faW50OF9za2lwX21vZHVsZXM9WyJtb2RlbC52aXN1YWwiLCAibG1faGVhZCJdKSwKICAgICkKICAgIGFzc2VydCBub3QgYW55KGlzaW5zdGFuY2UobSwgYm5iLm5uLkxpbmVhcjRiaXQpIGZvciBtIGluIG1vZGVsLm1vZGVsLnZpc3VhbC5tb2R1bGVzKCkpCiAgICBhc3NlcnQgbm90IGlzaW5zdGFuY2UobW9kZWwubG1faGVhZCwgYm5iLm5uLkxpbmVhcjRiaXQpCiAgICBhc3NlcnQgc3VtKGlzaW5zdGFuY2UobSwgYm5iLm5uLkxpbmVhcjRiaXQpIGZvciBtIGluIG1vZGVsLm1vZHVsZXMoKSkgPT0gMjUyCiAgICByZXR1cm4gbW9kZWwKCgpkZWYgbG9hZF9wcm9jZXNzb3IoKToKICAgIHByb2Nlc3NvciA9IEF1dG9Qcm9jZXNzb3IuZnJvbV9wcmV0cmFpbmVkKE1PREVMX0lELCByZXZpc2lvbj1NT0RFTF9SRVZJU0lPTikKICAgIHByb2Nlc3Nvci50b2tlbml6ZXIucGFkZGluZ19zaWRlID0gImxlZnQiCiAgICByZXR1cm4gcHJvY2Vzc29yCgoKZGVmIGFyY2hpdGVjdHVyZShtb2RlbCk6CiAgICBjb3VudHMgPSB7Imxhbmd1YWdlX21vZGVsIjogMCwgInZpc2lvbl9iYWNrYm9uZSI6IDAsICJwcm9qZWN0b3IiOiAwLCAib3RoZXIiOiAwfQogICAgZm9yIG5hbWUsIHAgaW4gbW9kZWwubmFtZWRfcGFyYW1ldGVycygpOgogICAgICAgIHNoYXBlID0gcC5xdWFudF9zdGF0ZS5zaGFwZSBpZiBpc2luc3RhbmNlKHAsIGJuYi5ubi5QYXJhbXM0Yml0KSBhbmQgcC5xdWFudF9zdGF0ZSBlbHNlIHAuc2hhcGUKICAgICAgICBzaXplID0gbWF0aC5wcm9kKHNoYXBlKQogICAgICAgIGJsb2NrID0gInByb2plY3RvciIgaWYgcHJvamVjdG9yX3BhcmFtZXRlcihuYW1lKSBlbHNlICgKICAgICAgICAgICAgInZpc2lvbl9iYWNrYm9uZSIgaWYgInZpc3VhbC4iIGluIG5hbWUgZWxzZQogICAgICAgICAgICAibGFuZ3VhZ2VfbW9kZWwiIGlmICJsYW5ndWFnZV9tb2RlbC4iIGluIG5hbWUgZWxzZSAib3RoZXIiKQogICAgICAgIGNvdW50c1tibG9ja10gKz0gc2l6ZQogICAgY291bnRzWyJ0b3RhbCJdID0gc3VtKGNvdW50cy52YWx1ZXMoKSkKICAgIHJldHVybiBjb3VudHMKCgpkZWYgYXBwbHlfc2V0dXAobW9kZWwsIHNldHVwLCByYW5rPTE2KToKICAgIGZvciBwIGluIG1vZGVsLnBhcmFtZXRlcnMoKToKICAgICAgICBwLnJlcXVpcmVzX2dyYWRfKEZhbHNlKQogICAgbW9kZWwuZW5hYmxlX2lucHV0X3JlcXVpcmVfZ3JhZHMoKQogICAgcGF0dGVybiA9IExMTV9UQVJHRVQgaWYgc2V0dXAgIT0gIkUzIiBlbHNlIGYiKD86e0xMTV9UQVJHRVR9fHtWSVRfVEFSR0VUfSkiCiAgICBtYXRjaGVzID0gW25hbWUgZm9yIG5hbWUsIG1vZHVsZSBpbiBtb2RlbC5uYW1lZF9tb2R1bGVzKCkgaWYgcmUuZnVsbG1hdGNoKHBhdHRlcm4sIG5hbWUpXQogICAgZXhwZWN0ZWQgPSAzNDggaWYgc2V0dXAgPT0gIkUzIiBlbHNlIDI1MgogICAgYXNzZXJ0IGxlbihtYXRjaGVzKSA9PSBleHBlY3RlZCwgKHNldHVwLCBsZW4obWF0Y2hlcyksIG1hdGNoZXNbOjVdKQogICAgbW9kZWwgPSBnZXRfcGVmdF9tb2RlbChtb2RlbCwgTG9yYUNvbmZpZyhyPXJhbmssIGxvcmFfYWxwaGE9MiAqIHJhbmssIGxvcmFfZHJvcG91dD0wLjA1LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRhcmdldF9tb2R1bGVzPXBhdHRlcm4sIHRhc2tfdHlwZT0iQ0FVU0FMX0xNIikpCiAgICBmb3IgbmFtZSwgcCBpbiBtb2RlbC5uYW1lZF9wYXJhbWV0ZXJzKCk6CiAgICAgICAgaWYgc2V0dXAgaW4geyJFMiIsICJFMyJ9IGFuZCBwcm9qZWN0b3JfcGFyYW1ldGVyKG5hbWUpOgogICAgICAgICAgICBwLmRhdGEgPSBwLmRhdGEuZmxvYXQoKQogICAgICAgICAgICBwLnJlcXVpcmVzX2dyYWRfKFRydWUpCiAgICAgICAgaWYgcC5yZXF1aXJlc19ncmFkOgogICAgICAgICAgICBhc3NlcnQgImxvcmFfIiBpbiBuYW1lIG9yIChzZXR1cCBpbiB7IkUyIiwgIkUzIn0gYW5kIHByb2plY3Rvcl9wYXJhbWV0ZXIobmFtZSkpLCBuYW1lCiAgICAgICAgICAgICMgR3JhZFNjYWxlciB1bnNjYWxlIHJlcXVpcmVzIGZwMzIgdHJhaW5hYmxlIGdyYWRpZW50cy4KICAgICAgICAgICAgcC5kYXRhID0gcC5kYXRhLmZsb2F0KCkKICAgIHRyYWluYWJsZXMgPSB7bmFtZTogcC5udW1lbCgpIGZvciBuYW1lLCBwIGluIG1vZGVsLm5hbWVkX3BhcmFtZXRlcnMoKSBpZiBwLnJlcXVpcmVzX2dyYWR9CiAgICBpbmZvID0geyJzZXR1cCI6IHNldHVwLCAicmFuayI6IHJhbmssICJtYXRjaGVkX21vZHVsZXMiOiBtYXRjaGVzLAogICAgICAgICAgICAidHJhaW5hYmxlX3BhcmFtcyI6IHN1bSh0cmFpbmFibGVzLnZhbHVlcygpKSwKICAgICAgICAgICAgInByb2plY3Rvcl90cmFpbmFibGVfcGFyYW1zIjogc3VtKG4gZm9yIG5hbWUsIG4gaW4gdHJhaW5hYmxlcy5pdGVtcygpIGlmIHByb2plY3Rvcl9wYXJhbWV0ZXIobmFtZSkpLAogICAgICAgICAgICAidHJhaW5hYmxlX3BhcmFtZXRlcl9uYW1lcyI6IGxpc3QodHJhaW5hYmxlcyl9CiAgICBwcmludCgiU0VUVVAiLCBzZXR1cCwgaW5mb1sidHJhaW5hYmxlX3BhcmFtcyJdLCAibW9kdWxlcyIsIGxlbihtYXRjaGVzKSwgZmx1c2g9VHJ1ZSkKICAgIG1vZGVsLmdyYWRpZW50X2NoZWNrcG9pbnRpbmdfZW5hYmxlKGdyYWRpZW50X2NoZWNrcG9pbnRpbmdfa3dhcmdzPXsidXNlX3JlZW50cmFudCI6IEZhbHNlfSkKICAgIG1vZGVsLmNvbmZpZy51c2VfY2FjaGUgPSBGYWxzZQogICAgbW9kZWwuY29uZmlnLnRleHRfY29uZmlnLnVzZV9jYWNoZSA9IEZhbHNlCiAgICByZXR1cm4gbW9kZWwsIGluZm8KCgpkZWYgdGVuc29yX2hhc2goc3RhdGUpOgogICAgZGlnZXN0ID0gaGFzaGxpYi5zaGEyNTYoKQogICAgZm9yIG5hbWUsIHRlbnNvciBpbiBzb3J0ZWQoc3RhdGUuaXRlbXMoKSk6CiAgICAgICAgZGlnZXN0LnVwZGF0ZShuYW1lLmVuY29kZSgpKQogICAgICAgIGRpZ2VzdC51cGRhdGUodGVuc29yLmRldGFjaCgpLmNwdSgpLmNvbnRpZ3VvdXMoKS5udW1weSgpLnRvYnl0ZXMoKSkKICAgIHJldHVybiBkaWdlc3QuaGV4ZGlnZXN0KCkKCgpkZWYgc2F2ZV9jaGVja3BvaW50KG1vZGVsLCBwcm9jZXNzb3IsIGZvbGRlciwgaW5mbyk6CiAgICBmb2xkZXIgPSBQYXRoKGZvbGRlcikKICAgIGZvbGRlci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPUZhbHNlKQogICAgbW9kZWwuc2F2ZV9wcmV0cmFpbmVkKGZvbGRlciwgc2FmZV9zZXJpYWxpemF0aW9uPVRydWUpCiAgICBwcm9jZXNzb3Iuc2F2ZV9wcmV0cmFpbmVkKGZvbGRlcikKICAgIHN0YXRlID0ge25hbWU6IHAuZGV0YWNoKCkuY3B1KCkuY2xvbmUoKSBmb3IgbmFtZSwgcCBpbiBtb2RlbC5uYW1lZF9wYXJhbWV0ZXJzKCkKICAgICAgICAgICAgIGlmIHAucmVxdWlyZXNfZ3JhZCBhbmQgcHJvamVjdG9yX3BhcmFtZXRlcihuYW1lKX0KICAgIGlmIHN0YXRlOgogICAgICAgIHRvcmNoLnNhdmUoc3RhdGUsIGZvbGRlciAvICJwcm9qZWN0b3IucHQiKQogICAgc2F2ZWQgPSBkaWN0KGluZm8sIHByb2plY3Rvcl9zaGEyNTY9dGVuc29yX2hhc2goc3RhdGUpIGlmIHN0YXRlIGVsc2UgTm9uZSkKICAgIChmb2xkZXIgLyAiY2hlY2twb2ludC1pbmZvLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoc2F2ZWQsIGluZGVudD0yKSkKICAgIHJldHVybiBzYXZlZAoKCmRlZiBsb2FkX2NoZWNrcG9pbnQoYmFzZSwgZm9sZGVyKToKICAgIGZvbGRlciA9IFBhdGgoZm9sZGVyKQogICAgbW9kZWwgPSBQZWZ0TW9kZWwuZnJvbV9wcmV0cmFpbmVkKGJhc2UsIGZvbGRlciwgaXNfdHJhaW5hYmxlPUZhbHNlKQogICAgaW5mbyA9IGpzb24ubG9hZHMoKGZvbGRlciAvICJjaGVja3BvaW50LWluZm8uanNvbiIpLnJlYWRfdGV4dCgpKQogICAgaWYgKGZvbGRlciAvICJwcm9qZWN0b3IucHQiKS5leGlzdHMoKToKICAgICAgICBzdGF0ZSA9IHRvcmNoLmxvYWQoZm9sZGVyIC8gInByb2plY3Rvci5wdCIsIG1hcF9sb2NhdGlvbj0iY3B1Iiwgd2VpZ2h0c19vbmx5PVRydWUpCiAgICAgICAgYXNzZXJ0IHRlbnNvcl9oYXNoKHN0YXRlKSA9PSBpbmZvWyJwcm9qZWN0b3Jfc2hhMjU2Il0KICAgICAgICBwYXJhbXMgPSBkaWN0KG1vZGVsLm5hbWVkX3BhcmFtZXRlcnMoKSkKICAgICAgICBmb3IgbmFtZSwgdmFsdWUgaW4gc3RhdGUuaXRlbXMoKToKICAgICAgICAgICAgYXNzZXJ0IG5hbWUgaW4gcGFyYW1zIGFuZCBwYXJhbXNbbmFtZV0uc2hhcGUgPT0gdmFsdWUuc2hhcGUsIG5hbWUKICAgICAgICAgICAgcGFyYW1zW25hbWVdLmRhdGEgPSB2YWx1ZS50byhkZXZpY2U9cGFyYW1zW25hbWVdLmRldmljZSwgZHR5cGU9dG9yY2guZmxvYXQzMikKICAgICAgICBhc3NlcnQgdGVuc29yX2hhc2goe25hbWU6IHBhcmFtc1tuYW1lXSBmb3IgbmFtZSBpbiBzdGF0ZX0pID09IGluZm9bInByb2plY3Rvcl9zaGEyNTYiXQogICAgbW9kZWwuY29uZmlnLnVzZV9jYWNoZSA9IFRydWUKICAgIG1vZGVsLmNvbmZpZy50ZXh0X2NvbmZpZy51c2VfY2FjaGUgPSBUcnVlCiAgICBtb2RlbC5ldmFsKCkKICAgIHJldHVybiBtb2RlbAo=", "student/pipeline.py": "IiIiUnVuIG9ubHkgb24gS2FnZ2xlOiBwcmVmbGlnaHQgb3IgdGhlIGZ1bGwgbWFuZGF0b3J5IGV4cGVyaW1lbnQgc2VxdWVuY2UuIiIiCmltcG9ydCBnYwppbXBvcnQgaW1wb3J0bGliLm1ldGFkYXRhCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgcmFuZG9tCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKaW1wb3J0IHRyYWNlYmFjawppbXBvcnQgemlwZmlsZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCB0b3JjaApmcm9tIFBJTCBpbXBvcnQgSW1hZ2VEcmF3Cgpmcm9tIHNjcmlwdHMuc2NvcmUgaW1wb3J0IGV2YWx1YXRlCmZyb20gc3R1ZGVudC5kYXRhIGltcG9ydCBQUk9NUFQsIGNvbGxhdGUsIGVuY29kZV9leGFtcGxlLCBpbWFnZV9mb3IsIG9iamVjdHNfdG9fYW5zd2VyLCByZWFkX3Jvd3MsIHN1YnNldApmcm9tIHN0dWRlbnQuaW5mZXIgaW1wb3J0IHByZWRpY3QKZnJvbSBzdHVkZW50Lm1vZGVsIGltcG9ydCBNT0RFTF9JRCwgTU9ERUxfUkVWSVNJT04sIGFwcGx5X3NldHVwLCBhcmNoaXRlY3R1cmUsIGxvYWRfYmFzZSwgbG9hZF9jaGVja3BvaW50LCBsb2FkX3Byb2Nlc3Nvcgpmcm9tIHN0dWRlbnQudHJhaW4gaW1wb3J0IHRyYWluCgpST09UID0gUGF0aC5jd2QoKQpEQVRBID0gUk9PVCAvICJkYXRhIgpXT1JLID0gUk9PVCAvICJ3b3JrIgpFViA9IFJPT1QgLyAicmVwb3J0L2V2aWRlbmNlIgpWQUwgPSBST09UIC8gInJlcG9ydC92YWxpZGF0aW9uIgoKCmRlZiB3cml0ZV9qc29uKHBhdGgsIGRhdGEpOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHBhdGgud3JpdGVfdGV4dChqc29uLmR1bXBzKGRhdGEsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpKQoKCmRlZiBzZWVkX2FsbChzZWVkPTQyKToKICAgIHJhbmRvbS5zZWVkKHNlZWQpCiAgICB0b3JjaC5tYW51YWxfc2VlZChzZWVkKQogICAgdG9yY2guY3VkYS5tYW51YWxfc2VlZF9hbGwoc2VlZCkKCgpkZWYgcHJlcGFyZSgpOgogICAgcm9vdHMgPSBbcC5wYXJlbnQgZm9yIHAgaW4gUGF0aCgiL2thZ2dsZS9pbnB1dCIpLnJnbG9iKCJpbWFnZXMiKQogICAgICAgICAgICAgaWYgcC5pc19kaXIoKSBhbmQgKHAucGFyZW50IC8gImxhYmVscy1ZT0xPIikuaXNfZGlyKCldCiAgICBhc3NlcnQgbGVuKHJvb3RzKSA9PSAxLCByb290cwogICAgZW52ID0gZGljdChvcy5lbnZpcm9uLCBEMTBfUkFXPXN0cihyb290c1swXSkpCiAgICBzdWJwcm9jZXNzLnJ1bihbc3lzLmV4ZWN1dGFibGUsICItbSIsICJweXRlc3QiLCAiLXEiLCAidGVzdHMiLCAic3R1ZGVudC90ZXN0X3N0dWRlbnQucHkiXSwKICAgICAgICAgICAgICAgICAgIGVudj1lbnYsIGNoZWNrPVRydWUpCiAgICBzdWJwcm9jZXNzLnJ1bihbc3lzLmV4ZWN1dGFibGUsICJzY3JpcHRzL3ByZXBhcmVfZGF0YS5weSIsICItLXNvdXJjZSIsIHN0cihyb290c1swXSldLCBjaGVjaz1UcnVlKQogICAgc3BsaXRzID0ge25hbWU6IHJlYWRfcm93cyhEQVRBIC8gZiJzcGxpdHMve25hbWV9Lmpzb25sIikgZm9yIG5hbWUgaW4gKCJ0cmFpbiIsICJ2YWwiLCAidGVzdCIpfQogICAgc2Vzc2lvbnMgPSB7bmFtZToge3JbInNlc3Npb24iXSBmb3IgciBpbiByb3dzfSBmb3IgbmFtZSwgcm93cyBpbiBzcGxpdHMuaXRlbXMoKX0KICAgIGFzc2VydCBub3Qgc2Vzc2lvbnNbInRyYWluIl0gJiBzZXNzaW9uc1sidmFsIl0KICAgIGFzc2VydCBub3Qgc2Vzc2lvbnNbInRyYWluIl0gJiBzZXNzaW9uc1sidGVzdCJdCiAgICBhc3NlcnQgbm90IHNlc3Npb25zWyJ2YWwiXSAmIHNlc3Npb25zWyJ0ZXN0Il0KICAgIG1ldGEgPSBqc29uLmxvYWRzKChEQVRBIC8gInNwbGl0cy9tZXRhLmpzb24iKS5yZWFkX3RleHQoKSkKICAgIGFzc2VydCBtZXRhWyJkYXRhc2V0X3ZlcnNpb24iXSA9PSAiZDEwLTIxZTQwNzAzMDEiLCBtZXRhCiAgICBhc3NlcnQgW2xlbihzcGxpdHNbbl0pIGZvciBuIGluICgidHJhaW4iLCAidmFsIiwgInRlc3QiKV0gPT0gWzE0MjIsIDI3NiwgMzExXQogICAgZm9yIHJvd3MgaW4gc3BsaXRzLnZhbHVlcygpOgogICAgICAgIGZvciByb3cgaW4gcm93czoKICAgICAgICAgICAgb2JqZWN0c190b19hbnN3ZXIocm93WyJvYmplY3RzIl0pCiAgICBpbnZhbGlkID0ge25hbWU6IFt7ImltYWdlX2lkIjogcm93WyJpbWFnZV9pZCJdLCAib2JqZWN0Ijogb2JqfSBmb3Igcm93IGluIHJvd3MKICAgICAgICAgICAgICAgICAgICAgZm9yIG9iaiBpbiByb3dbIm9iamVjdHMiXSBpZiBvYmpbInlvbG8iXVsyXSA8PSAwIG9yIG9ialsieW9sbyJdWzNdIDw9IDBdCiAgICAgICAgICAgICAgIGZvciBuYW1lLCByb3dzIGluIHNwbGl0cy5pdGVtcygpfQogICAgd3JpdGVfanNvbihST09UIC8gInJlcG9ydC9kZWdlbmVyYXRlLWxhYmVscy5qc29uIiwgaW52YWxpZCkKICAgIGZvciBpLCByb3cgaW4gZW51bWVyYXRlKHNwbGl0c1sidHJhaW4iXVs6Nl0pOgogICAgICAgIGltYWdlID0gaW1hZ2VfZm9yKHJvdywgREFUQSkKICAgICAgICBkcmF3ID0gSW1hZ2VEcmF3LkRyYXcoaW1hZ2UpCiAgICAgICAgZm9yIG9iaiBpbiBqc29uLmxvYWRzKG9iamVjdHNfdG9fYW5zd2VyKHJvd1sib2JqZWN0cyJdKSk6CiAgICAgICAgICAgIGJveCA9IG9ialsiYmJveF8yZCJdCiAgICAgICAgICAgIHh5ID0gW2JveFswXSAqIGltYWdlLndpZHRoIC8gMTAwMCwgYm94WzFdICogaW1hZ2UuaGVpZ2h0IC8gMTAwMCwKICAgICAgICAgICAgICAgICAgYm94WzJdICogaW1hZ2Uud2lkdGggLyAxMDAwLCBib3hbM10gKiBpbWFnZS5oZWlnaHQgLyAxMDAwXQogICAgICAgICAgICBkcmF3LnJlY3RhbmdsZSh4eSwgb3V0bGluZT0ibGltZSIsIHdpZHRoPTIpCiAgICAgICAgICAgIGRyYXcudGV4dCgoeHlbMF0sIHh5WzFdKSwgb2JqWyJsYWJlbCJdLCBmaWxsPSJsaW1lIikKICAgICAgICAoUk9PVCAvICJyZXBvcnQvZmlndXJlcyIpLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBpbWFnZS5zYXZlKFJPT1QgLyBmInJlcG9ydC9maWd1cmVzL2xhYmVscy17aX0ucG5nIikKICAgIHdyaXRlX2pzb24oUk9PVCAvICJyZXBvcnQvZGF0YS1hdWRpdC5qc29uIiwgbWV0YSkKICAgIHZlcnNpb25zID0ge25hbWU6IGltcG9ydGxpYi5tZXRhZGF0YS52ZXJzaW9uKG5hbWUpIGZvciBuYW1lIGluCiAgICAgICAgICAgICAgICAoInRvcmNoIiwgInRvcmNodmlzaW9uIiwgInRyYW5zZm9ybWVycyIsICJwZWZ0IiwgImFjY2VsZXJhdGUiLCAiYml0c2FuZGJ5dGVzIiwgInRybCIsICJ0b3JjaGFvIil9CiAgICB3cml0ZV9qc29uKFJPT1QgLyAicmVwb3J0L2Vudmlyb25tZW50Lmpzb24iLCBkaWN0KHZlcnNpb25zPXZlcnNpb25zLCBncHU9dG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoKSwKICAgICAgICAgICAgICAgIGN1ZGE9dG9yY2gudmVyc2lvbi5jdWRhLCBweXRob249c3lzLnZlcnNpb24sIHByb21wdD1QUk9NUFQsIG1vZGVsX2lkPU1PREVMX0lELAogICAgICAgICAgICAgICAgbW9kZWxfcmV2aXNpb249TU9ERUxfUkVWSVNJT04pKQogICAgcmV0dXJuIHNwbGl0cwoKCmRlZiBtZW1vcnlfcmVsZWFzZSgpOgogICAgZ2MuY29sbGVjdCgpCiAgICB0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKCkKCgpkZWYgdmFsaWRhdGVfZW5jb2RpbmcocHJvY2Vzc29yLCBzcGxpdHMpOgogICAgZXhhbXBsZXMgPSBbZW5jb2RlX2V4YW1wbGUocm93LCBEQVRBLCBwcm9jZXNzb3IpIGZvciByb3cgaW4gc3BsaXRzWyJ0cmFpbiJdWzoyXV0KICAgIGltYWdlX3Rva2VucyA9IGludCgoZXhhbXBsZXNbMF1bImlucHV0X2lkcyJdID09IHByb2Nlc3Nvci5pbWFnZV90b2tlbl9pZCkuc3VtKCkpCiAgICBkZWNvZGVkID0gcHJvY2Vzc29yLnRva2VuaXplci5kZWNvZGUoZXhhbXBsZXNbMF1bImxhYmVscyJdW2V4YW1wbGVzWzBdWyJsYWJlbHMiXSAhPSAtMTAwXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBza2lwX3NwZWNpYWxfdG9rZW5zPUZhbHNlKQogICAgYXNzZXJ0IGltYWdlX3Rva2VucyA9PSAyMjAsIGltYWdlX3Rva2VucwogICAgYXNzZXJ0IGRlY29kZWQuc3RhcnRzd2l0aChvYmplY3RzX3RvX2Fuc3dlcihzcGxpdHNbInRyYWluIl1bMF1bIm9iamVjdHMiXSkpCiAgICBiYXRjaCA9IGNvbGxhdGUoZXhhbXBsZXMsIHByb2Nlc3Nvci50b2tlbml6ZXIucGFkX3Rva2VuX2lkKQogICAgYXNzZXJ0IGJhdGNoWyJpbnB1dF9pZHMiXS5zaGFwZSA9PSBiYXRjaFsibGFiZWxzIl0uc2hhcGUKICAgIGFzc2VydCB0b3JjaC5hbGwoYmF0Y2hbImxhYmVscyJdW2JhdGNoWyJhdHRlbnRpb25fbWFzayJdID09IDBdID09IC0xMDApCiAgICB3cml0ZV9qc29uKFJPT1QgLyAicmVwb3J0L2VuY29kaW5nLWF1ZGl0Lmpzb24iLCBkaWN0KGltYWdlX3Rva2Vucz1pbWFnZV90b2tlbnMsCiAgICAgICAgICAgICAgICAgc3VwZXJ2aXNlZF90b2tlbnM9aW50KChleGFtcGxlc1swXVsibGFiZWxzIl0gIT0gLTEwMCkuc3VtKCkpLAogICAgICAgICAgICAgICAgIGRlY29kZWRfc3VwZXJ2aXNlZD1kZWNvZGVkLCBpbWFnZV9ncmlkX3Rodz1leGFtcGxlc1swXVsiaW1hZ2VfZ3JpZF90aHciXS50b2xpc3QoKSkpCgoKZGVmIHNjb3JlX2ZpbGUobmFtZSwgcHJlZGljdGlvbnMsIHNwbGl0PSJ0ZXN0IiwgbWV0YT1Ob25lLCBvdXRwdXQ9Tm9uZSk6CiAgICBvdXRwdXQgPSBvdXRwdXQgb3IgKEVWIC8gZiJ7bmFtZS5sb3dlcigpfS5qc29uIiBpZiBzcGxpdCA9PSAidGVzdCIgZWxzZSBWQUwgLyBmIntuYW1lLmxvd2VyKCl9Lmpzb24iKQogICAgY29tbWFuZCA9IFtzeXMuZXhlY3V0YWJsZSwgInNjcmlwdHMvc2NvcmUucHkiLCAiLS1wcmVkIiwgc3RyKHByZWRpY3Rpb25zKSwgIi0tbmFtZSIsIG5hbWUsCiAgICAgICAgICAgICAgICItLXNwbGl0Iiwgc3BsaXQsICItLW91dCIsIHN0cihvdXRwdXQpXQogICAgZm9yIGtleSBpbiAoInRyYWluYWJsZV9wYXJhbXMiLCAicGVha192cmFtX2diIiwgInRyYWluX21pbnV0ZXMiKToKICAgICAgICBpZiBtZXRhIGlzIG5vdCBOb25lOgogICAgICAgICAgICBjb21tYW5kICs9IFsiLS1tZXRhIiwgZiJ7a2V5fT17bWV0YVtrZXldfSJdCiAgICBpZiBtZXRhOgogICAgICAgIGZvciBrZXkgaW4gKCJvcHRpbWl6ZXJfc3RlcHMiLCAidHJhaW5faW1hZ2VzIiwgInNldHVwIiwgImJhdGNoX3NpemUiLCAiZ3JhZF9hY2N1bSIpOgogICAgICAgICAgICBjb21tYW5kICs9IFsiLS1tZXRhIiwgZiJ7a2V5fT17bWV0YVtrZXldfSJdCiAgICBzdWJwcm9jZXNzLnJ1bihjb21tYW5kLCBjaGVjaz1UcnVlKQogICAgcmV0dXJuIGpzb24ubG9hZHMoUGF0aChvdXRwdXQpLnJlYWRfdGV4dCgpKQoKCmRlZiBwcmVmbGlnaHQoc3BsaXRzLCBjb25maWcpOgogICAgcHJvY2Vzc29yID0gbG9hZF9wcm9jZXNzb3IoKQogICAgdmFsaWRhdGVfZW5jb2RpbmcocHJvY2Vzc29yLCBzcGxpdHMpCiAgICByZXN1bHRzID0ge30KICAgIGZvciBzZXR1cCBpbiAoIkUxIiwgIkUyIiwgIkUzIik6CiAgICAgICAgc2VlZF9hbGwoKQogICAgICAgIGJhc2UgPSBsb2FkX2Jhc2UoKQogICAgICAgIGFyY2ggPSBhcmNoaXRlY3R1cmUoYmFzZSkKICAgICAgICB3cml0ZV9qc29uKFJPT1QgLyAicmVwb3J0L2FyY2hpdGVjdHVyZS5qc29uIiwgYXJjaCkKICAgICAgICBtb2RlbCwgaW5mbyA9IGFwcGx5X3NldHVwKGJhc2UsIHNldHVwKQogICAgICAgIGRlbCBiYXNlCiAgICAgICAgIyBUd28gb3B0aW1pemVyIHN0ZXBzLCBvbiB0cmFpbiBvbmx5OyB0aGVzZSBhcmUgbm90IGZpbmFsIGV4cGVyaW1lbnQgcmVzdWx0cy4KICAgICAgICBydW4gPSB0cmFpbihtb2RlbCwgcHJvY2Vzc29yLCBzcGxpdHNbInRyYWluIl1bOjE2XSwgREFUQSwgV09SSyAvIGYic21va2Ute3NldHVwfSIsIGluZm8sCiAgICAgICAgICAgICAgICAgICAgc3RlcHM9MiwgYmF0Y2hfc2l6ZT0xLCBhY2N1bT04KQogICAgICAgIGRlbCBtb2RlbAogICAgICAgIG1lbW9yeV9yZWxlYXNlKCkKICAgICAgICBtb2RlbCA9IGxvYWRfY2hlY2twb2ludChsb2FkX2Jhc2UoKSwgV09SSyAvIGYic21va2Ute3NldHVwfS9maW5hbCIpCiAgICAgICAgb3V0cHV0ID0gV09SSyAvIGYic21va2Ute3NldHVwfS12YWwuanNvbmwiCiAgICAgICAgcmF3ID0gcHJlZGljdChtb2RlbCwgcHJvY2Vzc29yLCBzcGxpdHNbInZhbCJdWzoyXSwgREFUQSwgb3V0cHV0LCBiYXRjaF9zaXplPTIsCiAgICAgICAgICAgICAgICAgICAgICBtYXhfbmV3X3Rva2Vucz1jb25maWdbIm1heF9uZXdfdG9rZW5zIl0pCiAgICAgICAgcmVzdWx0ID0gZXZhbHVhdGUoc3BsaXRzWyJ2YWwiXVs6Ml0sIHtyWyJpbWFnZV9pZCJdOiByWyJyYXciXSBmb3IgciBpbiByYXd9LCBuX2Jvb3Q9MTAwKQogICAgICAgIHJlc3VsdHNbc2V0dXBdID0geyJ0cmFpbiI6IHJ1biwgInZhbGlkYXRpb24iOiByZXN1bHQsICJyYXciOiByYXd9CiAgICAgICAgZGVsIG1vZGVsCiAgICAgICAgbWVtb3J5X3JlbGVhc2UoKQogICAgYXNzZXJ0IHJlc3VsdHNbIkUyIl1bInRyYWluIl1bInRyYWluYWJsZV9wYXJhbXMiXSAtIHJlc3VsdHNbIkUxIl1bInRyYWluIl1bInRyYWluYWJsZV9wYXJhbXMiXSA9PSByZXN1bHRzWyJFMiJdWyJ0cmFpbiJdWyJwcm9qZWN0b3JfdHJhaW5hYmxlX3BhcmFtcyJdCiAgICB3cml0ZV9qc29uKFJPT1QgLyAicmVwb3J0L3ByZWZsaWdodC5qc29uIiwgcmVzdWx0cykKICAgIHByaW50KCJQUkVGTElHSFRfQ09NUExFVEUiLCBmbHVzaD1UcnVlKQoKCmRlZiB0cmFpbmVkX2V4cGVyaW1lbnQoc2V0dXAsIG5hbWUsIHNwbGl0cywgcHJvY2Vzc29yLCBjb25maWcsIGZyYWN0aW9uPTEuMCk6CiAgICBzZWVkX2FsbCgpCiAgICBtb2RlbCwgaW5mbyA9IGFwcGx5X3NldHVwKGxvYWRfYmFzZSgpLCBzZXR1cCkKICAgIHJvd3MgPSBzcGxpdHNbInRyYWluIl0gaWYgZnJhY3Rpb24gPT0gMSBlbHNlIHN1YnNldChzcGxpdHNbInRyYWluIl0sIGZyYWN0aW9uKQogICAgaW5mby51cGRhdGUobmFtZT1uYW1lLCB0cmFpbl9mcmFjdGlvbj1mcmFjdGlvbiwgbWF4X25ld190b2tlbnM9Y29uZmlnWyJtYXhfbmV3X3Rva2VucyJdKQogICAgZm9sZGVyID0gV09SSyAvIG5hbWUubG93ZXIoKQogICAgcnVuID0gdHJhaW4obW9kZWwsIHByb2Nlc3Nvciwgcm93cywgREFUQSwgZm9sZGVyLCBpbmZvLCBzdGVwcz0zMDAsIGJhdGNoX3NpemU9MSwgYWNjdW09OCkKICAgIGRlbCBtb2RlbAogICAgbWVtb3J5X3JlbGVhc2UoKQogICAgbW9kZWwgPSBsb2FkX2NoZWNrcG9pbnQobG9hZF9iYXNlKCksIGZvbGRlciAvICJmaW5hbCIpCiAgICB2YWxfcGF0aCA9IFdPUksgLyBmIntuYW1lLmxvd2VyKCl9LXZhbC5qc29ubCIKICAgIHByZWRpY3QobW9kZWwsIHByb2Nlc3Nvciwgc3BsaXRzWyJ2YWwiXSwgREFUQSwgdmFsX3BhdGgsIG1heF9uZXdfdG9rZW5zPWNvbmZpZ1sibWF4X25ld190b2tlbnMiXSkKICAgIHZhbGlkYXRpb24gPSBzY29yZV9maWxlKG5hbWUsIHZhbF9wYXRoLCBzcGxpdD0idmFsIiwgbWV0YT1ydW4pCiAgICB0ZXN0X3BhdGggPSBXT1JLIC8gZiJ7bmFtZS5sb3dlcigpfS5qc29ubCIKICAgIHByZWRpY3QobW9kZWwsIHByb2Nlc3Nvciwgc3BsaXRzWyJ0ZXN0Il0sIERBVEEsIHRlc3RfcGF0aCwgbWF4X25ld190b2tlbnM9Y29uZmlnWyJtYXhfbmV3X3Rva2VucyJdKQogICAgc2NvcmVfZmlsZShuYW1lLCB0ZXN0X3BhdGgsIG1ldGE9cnVuKQogICAgZGVsIG1vZGVsCiAgICBtZW1vcnlfcmVsZWFzZSgpCiAgICByZXR1cm4gdmFsaWRhdGlvbgoKCmRlZiBmdWxsKHNwbGl0cywgY29uZmlnKToKICAgIHByb2Nlc3NvciA9IGxvYWRfcHJvY2Vzc29yKCkKICAgIHZhbGlkYXRlX2VuY29kaW5nKHByb2Nlc3Nvciwgc3BsaXRzKQogICAgc2VlZF9hbGwoKQogICAgbW9kZWwgPSBsb2FkX2Jhc2UoKQogICAgd3JpdGVfanNvbihST09UIC8gInJlcG9ydC9hcmNoaXRlY3R1cmUuanNvbiIsIGFyY2hpdGVjdHVyZShtb2RlbCkpCiAgICBwcmVkaWN0KG1vZGVsLCBwcm9jZXNzb3IsIHNwbGl0c1sidmFsIl1bOjQwXSwgREFUQSwgV09SSyAvICJlMC12YWwtZGVidWcuanNvbmwiLAogICAgICAgICAgICBtYXhfbmV3X3Rva2Vucz1jb25maWdbIm1heF9uZXdfdG9rZW5zIl0pCiAgICBwcmVkaWN0KG1vZGVsLCBwcm9jZXNzb3IsIHNwbGl0c1sidGVzdCJdLCBEQVRBLCBXT1JLIC8gImUwLmpzb25sIiwKICAgICAgICAgICAgbWF4X25ld190b2tlbnM9Y29uZmlnWyJtYXhfbmV3X3Rva2VucyJdKQogICAgc2NvcmVfZmlsZSgiRTAiLCBXT1JLIC8gImUwLmpzb25sIikKICAgIGRlbCBtb2RlbAogICAgbWVtb3J5X3JlbGVhc2UoKQogICAgdmFsaWRhdGlvbnMgPSB7fQogICAgZm9yIHNldHVwIGluICgiRTEiLCAiRTIiLCAiRTMiKToKICAgICAgICB2YWxpZGF0aW9uc1tzZXR1cF0gPSB0cmFpbmVkX2V4cGVyaW1lbnQoc2V0dXAsIHNldHVwLCBzcGxpdHMsIHByb2Nlc3NvciwgY29uZmlnKQogICAgIyBTZWxlY3QgYmVmb3JlIGFueSBFNCB0ZXN0IGdlbmVyYXRpb24sIHVzaW5nIHZhbGlkYXRpb24gZXhjbHVzaXZlbHkuCiAgICBiZXN0ID0gbWF4KHZhbGlkYXRpb25zLCBrZXk9bGFtYmRhIG5hbWU6IHZhbGlkYXRpb25zW25hbWVdWyJtaWNybyJdWyJmMSJdKQogICAgd3JpdGVfanNvbihST09UIC8gInJlcG9ydC9lNC1zZWxlY3Rpb24uanNvbiIsIGRpY3Qoc2VsZWN0ZWQ9YmVzdCwgYmFzaXM9InZhbGlkYXRpb24gbWljcm8gRjEiLAogICAgICAgICAgICAgICB2YWxpZGF0aW9uX2YxPXtuYW1lOiB2YWx1ZVsibWljcm8iXVsiZjEiXSBmb3IgbmFtZSwgdmFsdWUgaW4gdmFsaWRhdGlvbnMuaXRlbXMoKX0pKQogICAgdHJhaW5lZF9leHBlcmltZW50KGJlc3QsICJFNF9kYXRhX3NpemVfMC4xIiwgc3BsaXRzLCBwcm9jZXNzb3IsIGNvbmZpZywgZnJhY3Rpb249LjEpCiAgICBiYXNlbGluZV9ydW4gPSBqc29uLmxvYWRzKChXT1JLIC8gZiJ7YmVzdC5sb3dlcigpfS9ydW4uanNvbiIpLnJlYWRfdGV4dCgpKQogICAgc2NvcmVfZmlsZSgiRTRfZGF0YV9zaXplXzEuMCIsIFdPUksgLyBmIntiZXN0Lmxvd2VyKCl9Lmpzb25sIiwgbWV0YT1iYXNlbGluZV9ydW4pCiAgICBmcm9tIHN0dWRlbnQuYW5hbHl6ZSBpbXBvcnQgYW5hbHl6ZQogICAgYW5hbHl6ZShST09UKQogICAgc3VicHJvY2Vzcy5ydW4oW3N5cy5leGVjdXRhYmxlLCAic2NyaXB0cy9jaGVja19zdWJtaXNzaW9uLnB5Il0sIGNoZWNrPVRydWUpCiAgICBwcmludCgiTUFOREFUT1JZX0VYUEVSSU1FTlRTX0NPTVBMRVRFIiwgZmx1c2g9VHJ1ZSkKCgpkZWYgY29tcGFjdF9hcnRpZmFjdHMoKToKICAgICMgQWx3YXlzIGV4cG9ydCBzbWFsbCBkaWFnbm9zdGljcywgZXZlbiBhZnRlciBmYWlsdXJlOyB3ZWlnaHRzIHN0YXkgb24gS2FnZ2xlLgogICAgdGFyZ2V0ID0gUGF0aCgiL2thZ2dsZS93b3JraW5nL2RheTEwLXJldmlldy56aXAiKQogICAgd2l0aCB6aXBmaWxlLlppcEZpbGUodGFyZ2V0LCAieCIsIGNvbXByZXNzaW9uPXppcGZpbGUuWklQX0RFRkxBVEVEKSBhcyBhcmNoaXZlOgogICAgICAgIGZvciBmb2xkZXIgaW4gKFJPT1QgLyAicmVwb3J0IiwgV09SSywgREFUQSAvICJzcGxpdHMiKToKICAgICAgICAgICAgaWYgZm9sZGVyLmV4aXN0cygpOgogICAgICAgICAgICAgICAgZm9yIHBhdGggaW4gZm9sZGVyLnJnbG9iKCIqIik6CiAgICAgICAgICAgICAgICAgICAgaWYgcGF0aC5pc19maWxlKCkgYW5kIHBhdGguc3VmZml4IGluIHsiLmpzb24iLCAiLmpzb25sIiwgIi5wbmciLCAiLm1kIiwgIi5jc3YifToKICAgICAgICAgICAgICAgICAgICAgICAgYXJjaGl2ZS53cml0ZShwYXRoLCBwYXRoLnJlbGF0aXZlX3RvKFJPT1QpKQogICAgICAgIGlmIChST09UIC8gIlJFUE9SVC5tZCIpLmV4aXN0cygpOgogICAgICAgICAgICBhcmNoaXZlLndyaXRlKFJPT1QgLyAiUkVQT1JULm1kIiwgIlJFUE9SVC5tZCIpCiAgICAgICAgaWYgKFJPT1QgLyAic291cmNlLW1hbmlmZXN0Lmpzb24iKS5leGlzdHMoKToKICAgICAgICAgICAgYXJjaGl2ZS53cml0ZShST09UIC8gInNvdXJjZS1tYW5pZmVzdC5qc29uIiwgInNvdXJjZS1tYW5pZmVzdC5qc29uIikKCgpkZWYgbWFpbigpOgogICAgY29uZmlnID0ganNvbi5sb2FkcygoUk9PVCAvICJydW4tY29uZmlnLmpzb24iKS5yZWFkX3RleHQoKSkKICAgIHdyaXRlX2pzb24oUk9PVCAvICJyZXBvcnQvcnVuLWNvbmZpZy5qc29uIiwgY29uZmlnKQogICAgdHJ5OgogICAgICAgIHNwbGl0cyA9IHByZXBhcmUoKQogICAgICAgIGlmIGNvbmZpZ1sibW9kZSJdID09ICJwcmVmbGlnaHQiOgogICAgICAgICAgICBwcmVmbGlnaHQoc3BsaXRzLCBjb25maWcpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZnVsbChzcGxpdHMsIGNvbmZpZykKICAgIGV4Y2VwdCBCYXNlRXhjZXB0aW9uOgogICAgICAgIHdyaXRlX2pzb24oUk9PVCAvICJyZXBvcnQvZmFpbHVyZS5qc29uIiwgeyJ0cmFjZWJhY2siOiB0cmFjZWJhY2suZm9ybWF0X2V4YygpfSkKICAgICAgICByYWlzZQogICAgZmluYWxseToKICAgICAgICBjb21wYWN0X2FydGlmYWN0cygpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=", "student/test_student.py": "aW1wb3J0IGpzb24KCmltcG9ydCBweXRlc3QKaW1wb3J0IHRvcmNoCgpmcm9tIHN0dWRlbnQuZGF0YSBpbXBvcnQgY29sbGF0ZSwgb2JqZWN0c190b19hbnN3ZXIsIHN1YnNldApmcm9tIHN0dWRlbnQgaW1wb3J0IGFuYWx5emUgICMgaW1wb3J0IHZhbGlkYXRlcyByZXBvcnQtZ2VuZXJhdGlvbiBzb3VyY2Ugb24gdGhlIEthZ2dsZSBydW50aW1lCgoKZGVmIHRlc3RfdGFyZ2V0X2NvbnZlcnNpb25fb3JkZXJfYW5kX2NsaXBwaW5nKCk6CiAgICBhbnN3ZXIgPSBvYmplY3RzX3RvX2Fuc3dlcihbeyJsYWJlbCI6ICJjcmFjayIsICJ5b2xvIjogWy4yLCAuOCwgLjEsIC4xXX0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgeyJsYWJlbCI6ICJwb3Rob2xlIiwgInlvbG8iOiBbLjUsIC41LCAuMiwgLjJdfV0pCiAgICBvYmplY3RzID0ganNvbi5sb2FkcyhhbnN3ZXIpCiAgICBhc3NlcnQgb2JqZWN0c1swXSA9PSB7ImxhYmVsIjogInBvdGhvbGUiLCAiYmJveF8yZCI6IFs0MDAsIDQwMCwgNjAwLCA2MDBdfQogICAgYXNzZXJ0IG9iamVjdHNfdG9fYW5zd2VyKFtdKSA9PSAiW10iCiAgICBhc3NlcnQganNvbi5sb2FkcyhvYmplY3RzX3RvX2Fuc3dlcihbeyJsYWJlbCI6ICJwb3Rob2xlIiwgInlvbG8iOiBbMCwgMCwgLjIsIC4yXX1dKSlbMF1bImJib3hfMmQiXSA9PSBbMCwgMCwgMTAwLCAxMDBdCgoKZGVmIHRlc3RfY29sbGF0b3JfcHJlc2VydmVzX3Zpc3VhbF9yb3dzX2FuZF9tYXNrc19wYWRkaW5nKCk6CiAgICBhID0geyJpbnB1dF9pZHMiOiB0b3JjaC50ZW5zb3IoW1sxLCAyXV0pLCAiYXR0ZW50aW9uX21hc2siOiB0b3JjaC50ZW5zb3IoW1sxLCAxXV0pLAogICAgICAgICAibW1fdG9rZW5fdHlwZV9pZHMiOiB0b3JjaC50ZW5zb3IoW1swLCAxXV0pLCAibGFiZWxzIjogdG9yY2gudGVuc29yKFtbLTEwMCwgMl1dKSwKICAgICAgICAgInBpeGVsX3ZhbHVlcyI6IHRvcmNoLm9uZXMoMywgNSksICJpbWFnZV9ncmlkX3RodyI6IHRvcmNoLnRlbnNvcihbWzEsIDIsIDZdXSl9CiAgICBiID0geyoqYSwgImlucHV0X2lkcyI6IHRvcmNoLnRlbnNvcihbWzEsIDIsIDNdXSksICJhdHRlbnRpb25fbWFzayI6IHRvcmNoLnRlbnNvcihbWzEsIDEsIDFdXSksCiAgICAgICAgICJtbV90b2tlbl90eXBlX2lkcyI6IHRvcmNoLnRlbnNvcihbWzAsIDEsIDBdXSksICJsYWJlbHMiOiB0b3JjaC50ZW5zb3IoW1stMTAwLCAyLCAzXV0pfQogICAgcmVzdWx0ID0gY29sbGF0ZShbYSwgYl0sIDApCiAgICBhc3NlcnQgcmVzdWx0WyJsYWJlbHMiXS50b2xpc3QoKSA9PSBbWy0xMDAsIDIsIC0xMDBdLCBbLTEwMCwgMiwgM11dCiAgICBhc3NlcnQgcmVzdWx0WyJwaXhlbF92YWx1ZXMiXS5zaGFwZSA9PSAoNiwgNSkKICAgIGFzc2VydCByZXN1bHRbImltYWdlX2dyaWRfdGh3Il0uc2hhcGUgPT0gKDIsIDMpCiAgICBhc3NlcnQgcmVzdWx0WyJhdHRlbnRpb25fbWFzayJdLnRvbGlzdCgpID09IFtbMSwgMSwgMF0sIFsxLCAxLCAxXV0KCgpkZWYgdGVzdF9zdWJzZXRfaXNfcmVwZWF0YWJsZV9hbmRfZG9lc19ub3RfbXV0YXRlKCk6CiAgICByb3dzID0gbGlzdChyYW5nZSgxNDIyKSkKICAgIHNtYWxsID0gc3Vic2V0KHJvd3MsIC4xKQogICAgYXNzZXJ0IGxlbihzbWFsbCkgPT0gMTQyIGFuZCBsZW4oc2V0KHNtYWxsKSkgPT0gMTQyCiAgICBhc3NlcnQgc21hbGwgPT0gc3Vic2V0KHJvd3MsIC4xKQogICAgYXNzZXJ0IHJvd3MgPT0gbGlzdChyYW5nZSgxNDIyKSkKCgpkZWYgdGVzdF9pbXBvc3NpYmxlX3NvdXJjZV9ib3hfaXNfbm90X2FfdHJhaW5pbmdfdGFyZ2V0KCk6CiAgICBhc3NlcnQgb2JqZWN0c190b19hbnN3ZXIoW3sibGFiZWwiOiAiY3JhY2siLCAieW9sbyI6IFsuNSwgLjUsIC4xLCAwXX1dKSA9PSAiW10iCiAgICBhc3NlcnQganNvbi5sb2FkcyhvYmplY3RzX3RvX2Fuc3dlcihbeyJsYWJlbCI6ICJjcmFjayIsICJ5b2xvIjogWy41LCAuNSwgLjAwMDEsIC4wMDAxXX1dKSlbMF1bImJib3hfMmQiXSA9PSBbNTAwLCA1MDAsIDUwMSwgNTAxXQo=", "student/train.py": "aW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IHJhbmRvbQppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCB0b3JjaAoKZnJvbSBzdHVkZW50LmRhdGEgaW1wb3J0IEVuY29kZWREYXRhLCBjb2xsYXRlCmZyb20gc3R1ZGVudC5tb2RlbCBpbXBvcnQgc2F2ZV9jaGVja3BvaW50CgoKZGVmIHRyYWluKG1vZGVsLCBwcm9jZXNzb3IsIHJvd3MsIGRhdGFfZGlyLCBmb2xkZXIsIGluZm8sIHN0ZXBzPTMwMCwgYmF0Y2hfc2l6ZT0xLCBhY2N1bT04LAogICAgICAgICAgbHI9MmUtNCwgc2VlZD00Mik6CiAgICBmb2xkZXIgPSBQYXRoKGZvbGRlcikKICAgIGZvbGRlci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPUZhbHNlKQogICAgZGF0YSA9IEVuY29kZWREYXRhKHJvd3MsIGRhdGFfZGlyLCBwcm9jZXNzb3IpCiAgICAoZm9sZGVyIC8gInRyYWluLWlkcy5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKFtyWyJpbWFnZV9pZCJdIGZvciByIGluIHJvd3NdKSkKICAgIHBhcmFtcyA9IFtwIGZvciBwIGluIG1vZGVsLnBhcmFtZXRlcnMoKSBpZiBwLnJlcXVpcmVzX2dyYWRdCiAgICBvcHRpbWl6ZXIgPSB0b3JjaC5vcHRpbS5BZGFtVyhwYXJhbXMsIGxyPWxyLCB3ZWlnaHRfZGVjYXk9MCwgZm9yZWFjaD1GYWxzZSkKICAgIHdhcm11cCA9IG1heCgxLCBtYXRoLmNlaWwoc3RlcHMgKiAuMDUpKQogICAgZGVmIHNjaGVkdWxlKHN0ZXApOgogICAgICAgIGlmIHN0ZXAgPCB3YXJtdXA6CiAgICAgICAgICAgIHJldHVybiAoc3RlcCArIDEpIC8gd2FybXVwCiAgICAgICAgcmV0dXJuIC41ICogKDEgKyBtYXRoLmNvcyhtYXRoLnBpICogKHN0ZXAgLSB3YXJtdXApIC8gbWF4KDEsIHN0ZXBzIC0gd2FybXVwKSkpCiAgICBzY2hlZHVsZXIgPSB0b3JjaC5vcHRpbS5scl9zY2hlZHVsZXIuTGFtYmRhTFIob3B0aW1pemVyLCBzY2hlZHVsZSkKICAgIHNjYWxlciA9IHRvcmNoLmFtcC5HcmFkU2NhbGVyKCJjdWRhIiwgaW5pdF9zY2FsZT0yNTYuMCkKICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oc2VlZCkKICAgIG9yZGVyLCBwb3NpdGlvbiA9IFtdLCAwCiAgICB0b3JjaC5jdWRhLnJlc2V0X3BlYWtfbWVtb3J5X3N0YXRzKCkKICAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKQogICAgc3RhcnRlZCA9IHRpbWUubW9ub3RvbmljKCkKICAgIG1vZGVsLnRyYWluKCkKICAgIG9wdGltaXplci56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICAgIHN0ZXAsIGF0dGVtcHRzID0gMCwgMAogICAgd2l0aCAoZm9sZGVyIC8gImxvc3MuanNvbmwiKS5vcGVuKCJ4IiwgYnVmZmVyaW5nPTEpIGFzIGxvZzoKICAgICAgICB3aGlsZSBzdGVwIDwgc3RlcHM6CiAgICAgICAgICAgIGF0dGVtcHRzICs9IDEKICAgICAgICAgICAgaWYgYXR0ZW1wdHMgPiBzdGVwcyArIDMwOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJUb28gbWFueSBza2lwcGVkIG9wdGltaXplciBzdGVwcyBmcm9tIG5vbi1maW5pdGUgZ3JhZGllbnRzIikKICAgICAgICAgICAgbG9zc19zdW0gPSAwLjAKICAgICAgICAgICAgZm9yIG1pY3JvIGluIHJhbmdlKGFjY3VtKToKICAgICAgICAgICAgICAgIGluZGljZXMgPSBbXQogICAgICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2UoYmF0Y2hfc2l6ZSk6CiAgICAgICAgICAgICAgICAgICAgaWYgcG9zaXRpb24gPj0gbGVuKG9yZGVyKToKICAgICAgICAgICAgICAgICAgICAgICAgb3JkZXIgPSBsaXN0KHJhbmdlKGxlbihyb3dzKSkpCiAgICAgICAgICAgICAgICAgICAgICAgIHJuZy5zaHVmZmxlKG9yZGVyKQogICAgICAgICAgICAgICAgICAgICAgICBwb3NpdGlvbiA9IDAKICAgICAgICAgICAgICAgICAgICBpbmRpY2VzLmFwcGVuZChvcmRlcltwb3NpdGlvbl0pCiAgICAgICAgICAgICAgICAgICAgcG9zaXRpb24gKz0gMQogICAgICAgICAgICAgICAgYmF0Y2ggPSB7azogdi50bygiY3VkYSIpIGZvciBrLCB2IGluIGNvbGxhdGUoW2RhdGEuZ2V0KGkpIGZvciBpIGluIGluZGljZXNdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHByb2Nlc3Nvci50b2tlbml6ZXIucGFkX3Rva2VuX2lkKS5pdGVtcygpfQogICAgICAgICAgICAgICAgd2l0aCB0b3JjaC5hdXRvY2FzdCgiY3VkYSIsIGR0eXBlPXRvcmNoLmZsb2F0MTYpOgogICAgICAgICAgICAgICAgICAgIGxvc3MgPSBtb2RlbCgqKmJhdGNoKS5sb3NzCiAgICAgICAgICAgICAgICBpZiBub3QgdG9yY2guaXNmaW5pdGUobG9zcyk6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgRmxvYXRpbmdQb2ludEVycm9yKGYiTm9uLWZpbml0ZSBsb3NzIGF0IHN0ZXAge3N0ZXB9LCBtaWNyb2JhdGNoIHttaWNyb30iKQogICAgICAgICAgICAgICAgbG9zc19zdW0gKz0gbG9zcy5kZXRhY2goKS5pdGVtKCkgLyBhY2N1bQogICAgICAgICAgICAgICAgc2NhbGVyLnNjYWxlKGxvc3MgLyBhY2N1bSkuYmFja3dhcmQoKQogICAgICAgICAgICAgICAgZGVsIGJhdGNoLCBsb3NzCiAgICAgICAgICAgIHNjYWxlci51bnNjYWxlXyhvcHRpbWl6ZXIpCiAgICAgICAgICAgIG5vcm0gPSB0b3JjaC5ubi51dGlscy5jbGlwX2dyYWRfbm9ybV8ocGFyYW1zLCAxLjApCiAgICAgICAgICAgIHByZXZpb3VzX3NjYWxlID0gc2NhbGVyLmdldF9zY2FsZSgpCiAgICAgICAgICAgIHNjYWxlci5zdGVwKG9wdGltaXplcikKICAgICAgICAgICAgc2NhbGVyLnVwZGF0ZSgpCiAgICAgICAgICAgIHNraXBwZWQgPSBzY2FsZXIuZ2V0X3NjYWxlKCkgPCBwcmV2aW91c19zY2FsZQogICAgICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKHNldF90b19ub25lPVRydWUpCiAgICAgICAgICAgIGlmIHNraXBwZWQ6CiAgICAgICAgICAgICAgICBwcmludCgiU0tJUFBFRCBzY2FsZWQgc3RlcCIsIHN0ZXAsICJzY2FsZSIsIHByZXZpb3VzX3NjYWxlLCBmbHVzaD1UcnVlKQogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgaWYgbm90IHRvcmNoLmlzZmluaXRlKG5vcm0pOgogICAgICAgICAgICAgICAgcmFpc2UgRmxvYXRpbmdQb2ludEVycm9yKCJOb24tZmluaXRlIHVuc2NhbGVkIGdyYWRpZW50IikKICAgICAgICAgICAgc3RlcCArPSAxCiAgICAgICAgICAgIHJvdyA9IHsic3RlcCI6IHN0ZXAsICJsb3NzIjogbG9zc19zdW0sICJncmFkX25vcm0iOiBub3JtLml0ZW0oKSwKICAgICAgICAgICAgICAgICAgICJsciI6IG9wdGltaXplci5wYXJhbV9ncm91cHNbMF1bImxyIl0sICJzZWNvbmRzIjogdGltZS5tb25vdG9uaWMoKSAtIHN0YXJ0ZWR9CiAgICAgICAgICAgIGxvZy53cml0ZShqc29uLmR1bXBzKHJvdykgKyAiXG4iKQogICAgICAgICAgICBzY2hlZHVsZXIuc3RlcCgpCiAgICAgICAgICAgIGlmIHN0ZXAgPD0gMyBvciBzdGVwICUgMTAgPT0gMDoKICAgICAgICAgICAgICAgIHByaW50KCJUUkFJTiIsIGluZm9bInNldHVwIl0sIHJvdywgZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgaWYgc3RlcCAlIDEwMCA9PSAwOgogICAgICAgICAgICAgICAgY2hlY2twb2ludCA9IGZvbGRlciAvIGYic3RlcC17c3RlcH0iCiAgICAgICAgICAgICAgICBzYXZlX2NoZWNrcG9pbnQobW9kZWwsIHByb2Nlc3NvciwgY2hlY2twb2ludCwgZGljdChpbmZvLCBvcHRpbWl6ZXJfc3RlcHM9c3RlcCkpCiAgICAgICAgICAgICAgICAjIEtlZXAgdGhlIGZ1bGwgcmVzdW1lIHN0YXRlIG91dHNpZGUgZXhwb3J0ZWQgb3V0cHV0IHRvIGF2b2lkIG11bHRpLUdCIGFydGlmYWN0cy4KICAgICAgICAgICAgICAgIHJlc3VtZSA9IFBhdGgoIi9rYWdnbGUvdGVtcC9kYXkxMC1yZXN1bWUiKSAvIGZvbGRlci5uYW1lIC8gZiJzdGVwLXtzdGVwfSIKICAgICAgICAgICAgICAgIHJlc3VtZS5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPUZhbHNlKQogICAgICAgICAgICAgICAgdG9yY2guc2F2ZSh7Im9wdGltaXplciI6IG9wdGltaXplci5zdGF0ZV9kaWN0KCksICJzY2hlZHVsZXIiOiBzY2hlZHVsZXIuc3RhdGVfZGljdCgpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgInNjYWxlciI6IHNjYWxlci5zdGF0ZV9kaWN0KCksICJybmciOiBybmcuZ2V0c3RhdGUoKSwgIm9yZGVyIjogb3JkZXIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAicG9zaXRpb24iOiBwb3NpdGlvbiwgInRvcmNoX3JuZyI6IHRvcmNoLmdldF9ybmdfc3RhdGUoKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICJjdWRhX3JuZyI6IHRvcmNoLmN1ZGEuZ2V0X3JuZ19zdGF0ZV9hbGwoKX0sIHJlc3VtZSAvICJ0cmFpbmluZy1zdGF0ZS5wdCIpCiAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCkKICAgIHJ1biA9IGRpY3QoaW5mbywgb3B0aW1pemVyX3N0ZXBzPXN0ZXAsIGF0dGVtcHRzPWF0dGVtcHRzLCBzZWVkPXNlZWQsIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSwKICAgICAgICAgICAgICAgZ3JhZF9hY2N1bT1hY2N1bSwgbGVhcm5pbmdfcmF0ZT1sciwgdHJhaW5faW1hZ2VzPWxlbihyb3dzKSwKICAgICAgICAgICAgICAgcGVha192cmFtX2diPXRvcmNoLmN1ZGEubWF4X21lbW9yeV9hbGxvY2F0ZWQoKSAvIDEwMjQgKiogMywKICAgICAgICAgICAgICAgdHJhaW5fbWludXRlcz0odGltZS5tb25vdG9uaWMoKSAtIHN0YXJ0ZWQpIC8gNjApCiAgICBzYXZlX2NoZWNrcG9pbnQobW9kZWwsIHByb2Nlc3NvciwgZm9sZGVyIC8gImZpbmFsIiwgcnVuKQogICAgKGZvbGRlciAvICJydW4uanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhydW4sIGluZGVudD0yKSkKICAgIHByaW50KCJUUkFJTl9ET05FIiwge2tleTogcnVuW2tleV0gZm9yIGtleSBpbiAoInNldHVwIiwgIm9wdGltaXplcl9zdGVwcyIsICJ0cmFpbmFibGVfcGFyYW1zIiwKICAgICAgICAgICAgICAgICAgICAgICAgICJwZWFrX3ZyYW1fZ2IiLCAidHJhaW5fbWludXRlcyIsICJ0cmFpbl9pbWFnZXMiKX0sIGZsdXNoPVRydWUpCiAgICByZXR1cm4gcnVuCg==", "lab_day10.ipynb": "..."}')
manifest = json.loads('{"README.md": "efe77fe86f254972461aa4d6032642459f52874573b94930cd07cff39176b63d", "GUIDE.md": "6926327a54f8be41a131c5d2086fbb296743c205956a18d898d2c94e7018c375", "RUBRIC.md": "467214bd9bcc01c102b1cff8ec4f478817ea8df4a721ebfebe56ec29186272aa", "REPORT.md": "b39c3a36dbbcec7b26dc60004a6d125bf987471b17ab7e986d76a5396a26932d", "requirements-colab.txt": "c6c5ba5beba1cbbd13afdbd0079ce1f641f13e35923956a41b2d5fbcab90c47b", "pytest.ini": "b160d82f7811555051f8966d8e02750a985227c1d022208c784e606946f2eada", "scripts/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "scripts/check_submission.py": "c4a872d46ebbbdafed98683d2c7fabf4afac3a1f64bd0ae7f5125725ba9f0c27", "scripts/labcommon.py": "77cc2bacd559c1040bdf7df831d3d8b6fbc58e5f5a4ffe7d998c056b0bcc0582", "scripts/make_notebook.py": "bb63a6568c33b64df122bbec8b051b074a64afa596a3dd69e689480f88870d26", "scripts/prepare_data.py": "c1dce60d2ddab83b8751575af06c35542d66e70b26229fd86f1e689c8012db2e", "scripts/score.py": "f07cd495542f7572dc51abf078a0a51559155c105c2bf12dbe796a0131fc38cb", "scripts/splitting.py": "e935f60b0145950e68cc05ae9f8d81338a2096d5d14761cc4b9b290a3580d67f", "tests/test_check_submission.py": "47b0e8264fcc6e2a68eea677e2a99804e1616b7f8951b825ec55284420a30797", "tests/test_configs_and_docs.py": "997a3931136f6506a4c70982a1d9d0840442945f58ff5308ec396ad98d092c03", "tests/test_prepare_data.py": "071fbceb8b3c3c0ee2143df1fb5ea7be6bcecaeacba4d680d42f38db347a3de5", "tests/test_score.py": "23437ffac657c36612bb48af7c63ae9446e4490b739a152073cd04a9032ee9e8", "tests/test_splitting.py": "6d2ac0b9a8df0568d2d101086a18dea846e8a2944512ae64807e99550ea79b43", "configs/e0.yaml": "0eb312a79d1248921b45a4d6c0c24aad6d940f52e355d92fec9ff8b2dcb5b3cf", "configs/e1.yaml": "7288969af7e1a06e998841ccbae3a6b06709b64db9226feefa68196eee58755a", "configs/e2.yaml": "60214bd647ecb5d6a7321d64a318767cc6c9f449cfc81eb6565d4151e22f2f19", "configs/e3.yaml": "1569f19bbe506f6cf1984dc58ca6d834df1b82df40e3eff338c0aba99c46e373", "configs/e4.yaml": "5ed40ed8c53f0db9696ce57e4a18cae6bf401e8a9b18dd1d36a48704af55afbd", "configs/e5.yaml": "67926dbdd5cfe300761aa2cd2cd879fe756011155839176a1a76f857a674e413", "student/__init__.py": "6ade19c041441200e24bbb1c503061da56328b24478315972e5dc429a158c6be", "student/analyze.py": "1663243a7220e360ecd5027dcea8814763cf2650773c28d8306081a5afa613ea", "student/data.py": "7a6bf5d5dde6cb15249e22131a013b67b4026d42e04603db83bd72871b3651e4", "student/infer.py": "c028fe6a8aa2a643deb21383a46b5e30d10352c4da6d3b8dc2e94990825578ba", "student/model.py": "279cf7f38dc4801f728d6a31dc011c1de255bc9a6e3f7dad1e0204acc88e0ed0", "student/pipeline.py": "acb2d8f0ca958d00ad90469174955be2cc8af153f24e342babcde4d560ba229a", "student/test_student.py": "5670968ef5fca96c6f15290c097ac346189654a631871f047fea8a72de7a8fb1", "student/train.py": "d122064b47b342e8b0463ebcccf7881542f4fb1a6c1deee49599dbe18adc77fd", "lab_day10.ipynb": "f319f7957f5417e8377a5c5c0c1167c8ace55d0f36ecacd84e95e19afad53563"}')
for relative, value in payload.items():
    raw = base64.b64decode(value)
    assert hashlib.sha256(raw).hexdigest() == manifest[relative]
    target = root / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(raw)
(root / 'source-manifest.json').write_text(json.dumps(manifest, indent=2))
(root / 'run-config.json').write_text('{"mode": "full", "max_new_tokens": 512, "optimizer_steps": 300, "batch_size": 1, "grad_accum": 8, "seed": 42, "note": "batch 1/accum 8 retains effective batch 8; completion cap 512 shared by all setups"}')
os.chdir(root)
os.environ['HF_HOME'] = '/kaggle/temp/day10-hf-cache'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
deps = '/kaggle/temp/day10-deps'
# Install into a fresh directory; preserve Kaggle's existing packages.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', deps, '--no-deps',
    'transformers==5.19.0', 'peft==0.21.2', 'trl==1.9.2', 'accelerate==1.15.0',
    'bitsandbytes==0.50.2', 'huggingface-hub==1.33.0', 'torchao==0.18.0'], check=True)
os.environ['PYTHONPATH'] = deps + ':' + str(root)
subprocess.run(['nvidia-smi', '-L'], check=True)


## 1. Dữ liệu: split theo session, test công cụ đề
Triển khai trong student/pipeline.py và các module student/ được đóng gói ở cell 0.


## 2. Nhãn YOLO → JSON 0–1000
Triển khai trong student/pipeline.py và các module student/ được đóng gói ở cell 0.


## 3. E0: greedy zero-shot, lưu raw
Triển khai trong student/pipeline.py và các module student/ được đóng gói ở cell 0.


## 4. Chat template, mask loss và collator
Triển khai trong student/pipeline.py và các module student/ được đóng gói ở cell 0.


## 5. Setup LoRA E1/E2/E3 và projector float32
Triển khai trong student/pipeline.py và các module student/ được đóng gói ở cell 0.


## 6. Train: 300 optimizer steps, checkpoint
Triển khai trong student/pipeline.py và các module student/ được đóng gói ở cell 0.


## 7. Reload, val và test với scorer gốc
Triển khai trong student/pipeline.py và các module student/ được đóng gói ở cell 0.


## 8. E4: 10%/100%, báo cáo và audit
Triển khai trong student/pipeline.py và các module student/ được đóng gói ở cell 0.


In [ ]:
subprocess.run([sys.executable, "-u", "-m", "student.pipeline"], check=True)
